# Q-FLARE — Runtime Dynamic Backend (Google Colab, Python backend only)

**Quantum-AI Flood Forecasting and Disaster-Response Decision-Support Platform**

## 0.1 What this notebook is

This notebook is the **runtime backend** of Q-FLARE. It converts the previous
**static** Q-FLARE quantum experiment — a fixed, synthetic allocation problem —
into a **runtime-driven decision-support engine** that accepts real inputs at
execution time and produces every downstream number from those inputs.

Static experiment (previous stage):

```
Static Input → QUBO → QAOA → Quantum Result
```

Runtime backend (this notebook):

```
Runtime Input → Validation → Flood Analysis → Arrival Time → Impact
   → Risk → Resource Demand → QUBO → Classical Optimization → QAOA
   → Feasibility → Allocation → Response Plan → Damage → Budget
   → Recovery → Alerts → Structured Report → Reassessment
```

## 0.2 Backend-only scope

**In scope (this notebook):** Python runtime data model, validation, flood and
impact analysis, risk scoring, resource-demand estimation, QUBO generation,
classical baseline, QAOA execution, feasibility checking, allocation, response
planning, damage / budget / recovery estimation, alerting, visualization,
persistence, automated tests, reassessment.

**Explicitly out of scope (NOT implemented here):** React frontend, HTML/CSS,
dashboards, FastAPI production deployment, Node.js backend, database
integration, authentication, AWS deployment, IoT hardware firmware.

## 0.3 Runtime data concept

The engine never embeds a result. It embeds **formulas, weights and thresholds**,
all of which are configurable. Every output is a function of the runtime input:

* Change the incoming flow → excess volume, severity, risk, demand, QUBO
  coefficients, allocation, budget, alerts and the QAOA optimization problem all
  change.
* Change the available rescue teams → the deficit, the deployment-slot
  constraint, the QUBO penalty landscape, the allocation and the alerts change.

Sections 31 demonstrates this explicitly (Test A / Test B / Test C).

## 0.4 The role of quantum optimization

> **Quantum optimization is responsible for constrained resource allocation.
> It should not be used to calculate every flood-related quantity.**

The boundary is explicit in the code:

```
Raw Runtime Data
     ↓  Feature / Impact Analysis      (classical, Sections 5-13)
Resource Demand + Location Features
     ↓  Optimization Preprocessor      (Section 14)
Binary Decision Variables + Constraints + Objective
     ↓  QUBO formulation               (preserved quantum core)
QAOA
```

Raw quantities (rainfall, dam capacity, population, road counts) are **never**
sent into QAOA. Only derived, unit-consistent, normalized optimization features
are.

## 0.5 Relationship to the existing quantum core

The existing standalone Q-FLARE quantum engine
(`q_flare_quantum_core.py` / `q_flare_quantum_core.ipynb`) is **preserved** here.
Its quantum mathematics is carried over unchanged:

* QUBO formulation and automatic penalty selection
* QUBO → Pauli cost Hamiltonian and mixer Hamiltonian
* QAOA ansatz construction (manual, gate by gate)
* QAOA angle optimization (SciPy COBYLA, seeded restarts)
* Exact statevector validation
* AerSimulator execution with fixed seeds
* Bitstring decoding, feasibility filtering, solution selection
* Classical-vs-quantum comparison
* Result persistence

What is new is everything **upstream** of the QUBO (runtime input → impact →
risk → demand) and everything **downstream** of the decoded bitstring
(allocation → plan → damage → budget → recovery → alerts → report).

## 0.6 Limitations — stated up front

* `AerSimulator` and `Statevector` are **local classical simulators**. No
  physical quantum computer is used. This notebook makes **no claim of quantum
  advantage, speedup or superiority**.
* QAOA is probabilistic and variational. When it does not return the exact
  optimum, that is **reported**, not engineered away.
* Flood arrival times, inundation depths, damage, budget and recovery durations
  are **model estimates** built from configurable ratios. They are not
  engineering-grade forecasts and carry no governmental authority.
* Coordinates in the demonstration scenario are **illustrative placeholders**
  near the Krishna–Godavari basin, not surveyed positions.
* Quantum simulation cost grows as `2**n`, so the optimization is deliberately
  limited to a small number of locations.
* The exact classical baseline enumerates `2**n` assignments and is therefore
  also limited to small `n`.

## 0.7 How to run

1. Open the notebook in Google Colab.
2. **Runtime → Run all**.
3. The first code cell installs the quantum stack, which can take a couple of
   minutes on a cold runtime. The demo scenario is pre-filled, so the whole
   notebook runs without typing anything.

To use your own data, edit the `MODE` cell in **Section 3** and pass a
`runtime_data` dictionary, or call `collect_runtime_input(mode="interactive")`.


In [ ]:
# =============================================================================
# SECTION 1 - ENVIRONMENT SETUP
# =============================================================================
#
# Installs the required scientific and quantum packages into the *current*
# interpreter (the Colab kernel), imports them, and prints a verified version
# banner. Installs are attempted only for modules that are genuinely missing,
# so a warm runtime is fast and an offline runtime still works if the stack is
# already present.
#
# Version compatibility note: the pins below are ranges, not exact versions.
# Qiskit 1.x and 2.x both expose QuantumCircuit, PauliEvolutionGate,
# SparsePauliOp and Statevector; Qiskit Aer >= 0.15 is required for
# AerSimulator. numpy >= 1.24 is required by both Qiskit and Pandas 2.x.

from __future__ import annotations

import importlib
import subprocess
import sys
import warnings
from typing import Any

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

AUTO_INSTALL_DEPENDENCIES = True

# Import name -> pip requirement, used both for probing and for installing.
REQUIRED_MODULES: dict[str, str] = {
    "qiskit": "qiskit>=1.2,<3.0",
    "qiskit_aer": "qiskit-aer>=0.15",
    "numpy": "numpy>=1.24",
    "pandas": "pandas>=2.0",
    "scipy": "scipy>=1.10",
    "matplotlib": "matplotlib>=3.7",
}

PINNED_REQUIREMENTS: tuple[str, ...] = tuple(REQUIRED_MODULES.values())

# Modules that must be importable before any runtime code is executed.
_QUANTUM_STACK: dict[str, Any] = {}


def missing_dependencies() -> list[str]:
    """Return the import names of every required module that is not importable."""
    absent: list[str] = []
    for module_name in REQUIRED_MODULES:
        try:
            importlib.import_module(module_name)
        except Exception:
            absent.append(module_name)
    return absent


def install_dependencies(force: bool = False) -> list[str]:
    """
    Install the required packages into the running interpreter.

    Only modules that are genuinely absent are installed, so a warm Colab
    runtime is fast. Installation targets `sys.executable`, which is the kernel
    interpreter; a bare `!pip install` can target a different environment.

    Args:
        force: Reinstall every pinned requirement even when already importable.

    Returns:
        The pip requirement strings that were installed.
    """
    targets = (
        list(PINNED_REQUIREMENTS)
        if force
        else [REQUIRED_MODULES[name] for name in missing_dependencies()]
    )
    if not targets:
        print("[env] all required packages are already importable - nothing to install.")
        return []

    print(f"[env] installing: {', '.join(targets)}")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *targets])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "dependency installation failed. Install the packages manually with "
            f"`pip install {' '.join(PINNED_REQUIREMENTS)}` and re-run this cell."
        ) from exc
    return targets


def import_dependencies() -> dict[str, Any]:
    """
    Import the scientific and quantum stack.

    Returns:
        Mapping of module name to imported module object.

    Raises:
        RuntimeError: If any required module is still missing after installation.
    """
    for module_name in REQUIRED_MODULES:
        try:
            _QUANTUM_STACK[module_name] = importlib.import_module(module_name)
        except Exception as exc:
            raise RuntimeError(
                f"required module '{module_name}' is not available: {exc}\n"
                f"Run: pip install {' '.join(PINNED_REQUIREMENTS)}"
            ) from exc

    return _QUANTUM_STACK


def environment_report() -> dict[str, str]:
    """Return the verified version of every relevant package."""
    versions: dict[str, str] = {"Python": sys.version.split()[0]}
    for module_name in ("numpy", "pandas", "scipy", "qiskit", "qiskit_aer", "matplotlib"):
        module = importlib.import_module(module_name)
        versions[module_name] = str(getattr(module, "__version__", "unknown"))
    return versions


def display_environment() -> bool:
    """
    Print the required environment banner and report readiness.

    Returns:
        True when the whole stack is importable (the pipeline may run).
    """
    versions = environment_report()
    banner = "=" * 60
    print(banner)
    print("ENVIRONMENT")
    print(banner)
    print("")
    print(f"Python version  : {versions['Python']}")
    print(f"NumPy           : {versions['numpy']}")
    print(f"Pandas          : {versions['pandas']}")
    print(f"SciPy           : {versions['scipy']}")
    print(f"Qiskit          : {versions['qiskit']}")
    print(f"Qiskit Aer      : {versions['qiskit_aer']}")
    print(f"Matplotlib      : {versions['matplotlib']}")
    print("")

    problems = missing_dependencies()
    if problems:
        print(f"Environment status: NOT READY (missing: {', '.join(problems)})")
        print(banner)
        return False
    print("Environment status: READY")
    print(banner)
    return True


if AUTO_INSTALL_DEPENDENCIES:
    _installed = install_dependencies()
    if _installed:
        print(f"[env] installed: {', '.join(_installed)}")
else:
    print("[env] AUTO_INSTALL_DEPENDENCIES is False - assuming packages exist.")

import_dependencies()

# Standard imports used by every later section.
import json
import math
from collections.abc import Callable, Sequence
from dataclasses import asdict, dataclass, field, is_dataclass
from datetime import datetime, timedelta, timezone
from enum import Enum
from pathlib import Path
import sys

import matplotlib
import numpy as np
import pandas as pd
from scipy import optimize as scipy_optimize

matplotlib.use("Agg", force=False)
import matplotlib.pyplot as plt


def configure_stdout() -> str:
    """
    Make stdout UTF-8 so Unicode box drawing and symbols can be printed.

    Colab is UTF-8 already, but a local Jupyter kernel launched on Windows can
    inherit a legacy code page, where `print()` of a single box-drawing character
    raises `UnicodeEncodeError`. A rendering problem must never be allowed to
    abort a quantum run, so stdout is reconfigured to UTF-8 with a replacement
    fallback.

    Returns:
        A short description of what was done, for the environment report.
    """
    stream = sys.stdout
    reconfigure = getattr(stream, "reconfigure", None)
    if reconfigure is None:
        return "stdout reconfiguration unsupported on this stream; ASCII fallback used"
    encoding = (getattr(stream, "encoding", "") or "").lower().replace("-", "")
    if encoding == "utf8":
        return f"stdout encoding {stream.encoding} (already UTF-8)"
    reconfigure(encoding="utf-8", errors="replace")
    return f"stdout encoding changed from {encoding or 'unset'} to utf-8 (errors=replace)"


STDOUT_NOTE = configure_stdout()

ENVIRONMENT_READY = display_environment()
print(f"[env] quantum stack modules: {', '.join(sorted(_QUANTUM_STACK))}")
print(f"[env] {STDOUT_NOTE}")


## SECTION 2 — RUNTIME INPUT MODEL

Every value the engine consumes is a **typed, unit-carrying, provenance-carrying**
record. Three ideas make the backend honest:

1. **Optional is explicit.** A field that was not supplied is `None`, never a
   silently invented number. `None` propagates into the report as
   `DATA_UNAVAILABLE`.
2. **Units travel with the value.** `incoming_flow` cannot be interpreted without
   `incoming_flow_unit`, so the model refuses to guess.
3. **Provenance travels with the value.** `DataSource` records whether a number
   came from a user, a sensor, an API, a database, a formula
   (`CALCULATED`), a configured model (`MODEL_ESTIMATED`), an optimizer, or is
   simply not available.

`TrackedValue` is the single wrapper used for anything that leaves the input
model: it carries `value`, `unit`, `source`, `confidence` and the `assumption`
behind it, so no output can be mistaken for a measurement when it is in fact a
configured estimate.


In [ ]:
# =============================================================================
# SECTION 2 - RUNTIME INPUT MODEL
# =============================================================================
#
# Structured dataclasses describing one flood event at runtime. Optional fields
# default to None (DATA_UNAVAILABLE), never to a fabricated real-world number.

# -----------------------------------------------------------------------------
# 2.1 - Units, provenance and confidence
# -----------------------------------------------------------------------------

# Volume / rate units. TMC is an absolute volume; TMC/day, m3/s and KCUS are
# rates. They are NEVER mixed implicitly: `normalise_flow()` in SECTION 5
# performs an explicit, recorded conversion.
UNIT_TMC = "TMC"
UNIT_TMC_PER_DAY = "TMC/day"
UNIT_M3_PER_S = "m3/s"
UNIT_KCUS = "KCUS"
UNIT_MM = "mm"
UNIT_M = "m"
UNIT_KM = "km"
UNIT_KM2 = "km2"
UNIT_HOURS = "hours"
UNIT_PEOPLE = "people"
UNIT_INR = "INR"

FLOW_RATE_UNITS: frozenset[str] = frozenset(
    {UNIT_TMC_PER_DAY, UNIT_M3_PER_S, UNIT_KCUS}
)
FLOW_VOLUME_UNITS: frozenset[str] = frozenset({UNIT_TMC})
ALL_FLOW_UNITS: frozenset[str] = FLOW_RATE_UNITS | FLOW_VOLUME_UNITS

# Exact, documented conversion constants.
M3_PER_TMC = 1.0e9  # 1 TMC (tillion cubic metre) = 1e9 m3 = 1 km3
SECONDS_PER_DAY = 86400.0
# 1 TMC/day expressed as a discharge: 1e9 m3 / 86400 s.
TMC_PER_DAY_TO_M3_PER_S = M3_PER_TMC / SECONDS_PER_DAY  # 11574.074074...
# 1 kilocumsecond (kc/s), the traditional Indian river-discharge unit.
KCUS_TO_M3_PER_S = 28.316846592


class DataSource(str, Enum):
    """Provenance of a value. Every reported number carries one of these."""

    USER_PROVIDED = "USER_PROVIDED"
    SENSOR = "SENSOR"
    API = "API"
    DATABASE = "DATABASE"
    CALCULATED = "CALCULATED"
    MODEL_ESTIMATED = "MODEL_ESTIMATED"
    QUANTUM_OPTIMIZED = "QUANTUM_OPTIMIZED"
    CLASSICAL_OPTIMIZED = "CLASSICAL_OPTIMIZED"
    DATA_UNAVAILABLE = "DATA_UNAVAILABLE"


class Confidence(str, Enum):
    """How much trust a value deserves."""

    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"
    UNAVAILABLE = "UNAVAILABLE"


@dataclass(frozen=True)
class TrackedValue:
    """
    A value with its unit, provenance, confidence and the assumption behind it.

    This wrapper is what makes the "never present an estimate as a measurement"
    rule enforceable: an output that omits `source` cannot be constructed.

    Attributes:
        value: The numeric value, or None when unavailable.
        unit: Unit string (physical unit, or a count label such as "teams").
        source: Provenance of the value.
        confidence: Confidence in the value.
        assumption: The documented assumption, when the value is estimated.
        note: Optional free-text note.
    """

    value: float | int | None
    unit: str
    source: DataSource
    confidence: Confidence
    assumption: str | None = None
    note: str | None = None

    @property
    def available(self) -> bool:
        """True when a usable numeric value is present."""
        return self.value is not None and not (
            isinstance(self.value, float) and math.isnan(self.value)
        )

    def as_dict(self) -> dict[str, Any]:
        """JSON-serializable representation, including source and confidence."""
        return {
            "value": self.value,
            "unit": self.unit,
            "source": self.source.value,
            "confidence": self.confidence.value,
            "assumption": self.assumption,
            "note": self.note,
        }

    def display(self, precision: int = 2) -> str:
        """Human-readable rendering, e.g. ``70.00 TMC`` or ``N/A``."""
        if not self.available:
            return f"N/A {self.unit}".strip()
        if isinstance(self.value, (int, np.integer)) and not isinstance(
            self.value, bool
        ):
            return f"{int(self.value):,} {self.unit}".strip()
        return f"{float(self.value):,.{precision}f} {self.unit}".strip()


def user_value(value, unit: str, note: str | None = None) -> TrackedValue:
    """Wrap a runtime-supplied value as USER_PROVIDED with HIGH confidence."""
    return TrackedValue(
        value=value,
        unit=unit,
        source=DataSource.USER_PROVIDED,
        confidence=Confidence.HIGH,
        note=note,
    )


def calculated(value, unit: str, assumption: str | None = None) -> TrackedValue:
    """Wrap a value produced by a formula in this notebook."""
    return TrackedValue(
        value=value,
        unit=unit,
        source=DataSource.CALCULATED,
        confidence=Confidence.HIGH,
        assumption=assumption,
    )


def estimated(
    value, unit: str, assumption: str, confidence: Confidence = Confidence.MEDIUM
) -> TrackedValue:
    """Wrap a value produced by a configured model ratio."""
    return TrackedValue(
        value=value,
        unit=unit,
        source=DataSource.MODEL_ESTIMATED,
        confidence=confidence,
        assumption=assumption,
    )


def unavailable(unit: str, reason: str | None = None) -> TrackedValue:
    """Wrap an absent value so the gap is reported instead of hidden."""
    return TrackedValue(
        value=None,
        unit=unit,
        source=DataSource.DATA_UNAVAILABLE,
        confidence=Confidence.UNAVAILABLE,
        note=reason,
    )


# -----------------------------------------------------------------------------
# 2.2 - Runtime input records
# -----------------------------------------------------------------------------


@dataclass
class EventInfo:
    """
    Event identification.

    Attributes:
        event_id: Stable event identifier, e.g. "QF-2026-001".
        timestamp: ISO-8601 event timestamp (UTC).
        region: River basin or administrative region.
        location: Human readable place description.
        data_source: Where the event record itself came from.
    """

    event_id: str
    timestamp: str
    region: str
    location: str
    data_source: str = "USER_PROVIDED"


@dataclass
class DamInfo:
    """
    Dam / hydrological record. Units are explicit for every quantity.

    Attributes:
        dam_name: Name of the dam or barrage.
        dam_capacity: Total storage capacity in TMC.
        current_storage: Present storage in TMC.
        incoming_flow: Incoming flood magnitude; interpretation depends on
            `incoming_flow_unit`.
        incoming_flow_unit: One of TMC, TMC/day, m3/s, KCUS.
        inflow_window_hours: Integration window used to convert a rate into a
            volume. Only used when `incoming_flow_unit` is a rate.
        water_level: Observed water level in m.
        flow_rate: Observed river discharge in m3/s.
        upstream_water_level: Upstream gauge level in m.
        downstream_water_level: Downstream gauge level in m.
        rainfall: Rainfall depth in mm.
        rainfall_duration: Rainfall duration in hours.
        previous_flow: Previous reading of `incoming_flow`, same unit.
        flow_trend: "RISING", "STEADY" or "FALLING", or None to derive it from
            `previous_flow`.
    """

    dam_name: str
    dam_capacity: float
    current_storage: float
    incoming_flow: float
    incoming_flow_unit: str = UNIT_TMC_PER_DAY
    inflow_window_hours: float = 24.0
    water_level: float | None = None
    flow_rate: float | None = None
    upstream_water_level: float | None = None
    downstream_water_level: float | None = None
    rainfall: float = 0.0
    rainfall_duration: float | None = None
    previous_flow: float | None = None
    flow_trend: str | None = None


@dataclass
class GeoInfo:
    """
    Geographic record.

    Attributes:
        latitude: Latitude in decimal degrees, or None.
        longitude: Longitude in decimal degrees, or None.
        affected_area: Affected area in km2, or None to be derived.
        upstream_distance: Distance from the flood source to the target in km.
        downstream_distance: Distance from the target to the outflow in km.
    """

    latitude: float | None = None
    longitude: float | None = None
    affected_area: float | None = None
    upstream_distance: float | None = None
    downstream_distance: float | None = None


@dataclass
class PopulationInfo:
    """
    Population record.

    Attributes:
        total_population: Total population of the affected area.
        vulnerable_population: Population requiring special assistance.
        children: Children (0-12 years).
        elderly: Elderly (60+ years).
        persons_requiring_assistance: People who cannot self-evacuate.
    """

    total_population: int
    vulnerable_population: int | None = None
    children: int | None = None
    elderly: int | None = None
    persons_requiring_assistance: int | None = None


@dataclass
class InfrastructureInfo:
    """
    Infrastructure inventory. Every field is a count, or None when unknown.

    Attributes:
        roads: Total road length in km.
        blocked_roads: Blocked road length in km.
        damaged_roads: Partially damaged road length in km.
        critical_roads: Length of life-line roads in km.
        bridges: Number of bridges.
        bridges_affected: Number of affected bridges.
        alternative_routes: Number of usable alternative routes.
        hospitals / schools / police_stations / fire_stations: Facility counts.
        emergency_centers: Emergency operations centre count.
        power_stations: Power installation count.
        communication_towers: Communication tower count.
        water_facilities: Water supply facility count.
        shelters: Existing shelter count.
    """

    roads: float | None = None
    blocked_roads: float | None = None
    damaged_roads: float | None = None
    critical_roads: float | None = None
    bridges: int | None = None
    bridges_affected: int | None = None
    alternative_routes: int | None = None
    hospitals: int | None = None
    schools: int | None = None
    police_stations: int | None = None
    fire_stations: int | None = None
    emergency_centers: int | None = None
    power_stations: int | None = None
    communication_towers: int | None = None
    water_facilities: int | None = None
    shelters: int | None = None


@dataclass
class ResourceAvailability:
    """
    Currently deployable resources. Every field is a count, or None when the
    asset manager has not reported it (reported as DATA_UNAVAILABLE, not 0).

    Attributes:
        rescue_teams: Deployable rescue teams.
        medical_teams: Deployable medical teams.
        doctors: Available doctors.
        nurses: Available nurses.
        ambulances: Available ambulances.
        vehicles: Available rescue / transport vehicles.
        boats: Available boats.
        shelters: Open shelters.
        shelter_capacity: Total shelter capacity in people.
        food_meals: Food rations in meals.
        water_litres: Drinking water in litres.
        medicines_kits: Medical kits.
    """

    rescue_teams: int | None = None
    medical_teams: int | None = None
    doctors: int | None = None
    nurses: int | None = None
    ambulances: int | None = None
    vehicles: int | None = None
    boats: int | None = None
    shelters: int | None = None
    shelter_capacity: int | None = None
    food_meals: float | None = None
    water_litres: float | None = None
    medicines_kits: int | None = None


@dataclass
class EconomicInfo:
    """
    Asset / economic values in INR. None means the value is unknown, which is
    reported as DATA_UNAVAILABLE rather than assumed to be zero.

    Attributes:
        land_value: Land value exposed, INR.
        crop_value: Standing crop value exposed, INR.
        infrastructure_value: Public infrastructure value, INR.
        business_value: Business / commercial value, INR.
        industrial_value: Industrial value, INR.
        agricultural_area: Affected agricultural area, km2.
        crop_area: Affected crop area, km2.
        livestock: Number of livestock exposed.
        livestock_value: Livestock value, INR.
    """

    land_value: float | None = None
    crop_value: float | None = None
    infrastructure_value: float | None = None
    business_value: float | None = None
    industrial_value: float | None = None
    agricultural_area: float | None = None
    crop_area: float | None = None
    livestock: int | None = None
    livestock_value: float | None = None


@dataclass
class LocationInput:
    """
    One response location inside the affected area. These records are what the
    optimization preprocessor turns into binary decision variables.

    Attributes:
        location_id: Stable identifier, e.g. "L1".
        name: Human readable name.
        population: Population of the location.
        distance_km: Distance from the staging point to the location in km.
        travel_time_hours: Observed / modelled travel time in hours.
        vulnerability_index: 0-1 intrinsic vulnerability of the settlement.
        critical_infrastructure_count: Count of critical facilities present.
        road_access: "OPEN", "PARTIAL" or "BLOCKED".
        latitude / longitude: Optional coordinates for the GIS hand-off.
    """

    location_id: str
    name: str
    population: int
    distance_km: float | None = None
    travel_time_hours: float | None = None
    vulnerability_index: float = 0.5
    critical_infrastructure_count: int = 0
    road_access: str = "OPEN"
    latitude: float | None = None
    longitude: float | None = None


@dataclass
class RuntimeEvent:
    """
    The complete runtime input: everything one Q-FLARE decision cycle consumes.

    Attributes:
        event: Event identification.
        dam: Dam / hydrology record.
        geography: Geographic record.
        population: Population record.
        infrastructure: Infrastructure inventory.
        resources: Available resources.
        economics: Economic / asset values.
        locations: Response locations for the allocation problem.
        data_provenance: Free-form mapping of field path to DataSource, set by
            the data-entry layer.
    """

    event: EventInfo
    dam: DamInfo
    geography: GeoInfo
    population: PopulationInfo
    infrastructure: InfrastructureInfo
    resources: ResourceAvailability
    economics: EconomicInfo
    locations: list[LocationInput] = field(default_factory=list)
    data_provenance: dict[str, DataSource] = field(default_factory=dict)


# Upper bound on the number of decision variables. The exact statevector
# expectation and the exhaustive classical baseline both scale as 2**n, so the
# quantum path is deliberately limited to a small n.
MAX_RUNTIME_LOCATIONS = 8


def to_serialisable(obj: Any) -> Any:
    """
    Recursively convert dataclasses, enums and numpy scalars to plain Python.

    Args:
        obj: Any value from the runtime model.

    Returns:
        A JSON-serializable structure.
    """
    if isinstance(obj, TrackedValue):
        return obj.as_dict()
    if isinstance(obj, Enum):
        return obj.value
    if is_dataclass(obj) and not isinstance(obj, type):
        return {key: to_serialisable(value) for key, value in asdict(obj).items()}
    if isinstance(obj, dict):
        return {str(key): to_serialisable(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_serialisable(value) for value in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    return obj


## SECTION 3 — RUNTIME DATA ENTRY

The engine accepts data two ways, and both go through the same builder:

* **MODE A — interactive.** `collect_runtime_input(mode="interactive")` prompts
  for the operational fields. Every prompt shows the current value, accepts a
  bare `Enter` to keep it, and accepts `n`, `na` or `-` to mark an optional
  field as *unavailable* (which the report will then show as
  `DATA_UNAVAILABLE` rather than guessing a number).
* **MODE B — Python dictionary.** `collect_runtime_input(runtime_data={...})`.
  Keys that are omitted become `None`, i.e. explicitly unavailable. This is the
  mode used by the automated tests and by the sensitivity demonstration in
  Section 31, so the pipeline is testable without a human at the keyboard.

### Demonstration dataset

The default dataset below is a **synthetic demonstration scenario**. It is
*not* real-time data from IMD, CWC, a dam operator or any IoT feed, and the
coordinates are illustrative placeholders near the Krishna–Godavari basin. It
exists so the notebook runs end to end on `Runtime → Run all`.

Its internal consistency is deliberate: the 70 TMC incoming volume, the
72-hour integration window and the reported 270 000 m³/s discharge describe the
same water volume, so the unit handling in Section 5 can be audited.


In [ ]:
# =============================================================================
# SECTION 3 - RUNTIME DATA ENTRY
# =============================================================================
#
# Two entry modes - interactive prompts and a Python dictionary - feeding one
# builder. The builder never invents a value: a key that is absent becomes
# None and is reported as DATA_UNAVAILABLE downstream.

RUNTIME_INPUT_MODE = "dictionary"  # "dictionary" | "interactive"

# -----------------------------------------------------------------------------
# 3.1 - Demonstration (SYNTHETIC) scenario
# -----------------------------------------------------------------------------
# DEMONSTRATION / SYNTHETIC INPUT - NOT real-time official data.
# The 70 TMC incoming volume, the 72 h window and the 270 000 m3/s discharge
# describe the same water volume: 70e9 m3 / (72*3600 s) = 270 062 m3/s.

DEMO_RUNTIME_DATA: dict[str, Any] = {
    "event": {
        "event_id": "QF-2026-001",
        "timestamp": "2026-10-01T06:00:00Z",
        "region": "Krishna-Godavari Basin",
        "location": "Lower basin, downstream of the barrage",
        "data_source": "DEMONSTRATION / SYNTHETIC INPUT",
    },
    "dam": {
        "dam_name": "Sriramsagar (illustrative)",
        "dam_capacity": 60.0,
        "current_storage": 55.0,
        "incoming_flow": 70.0,
        "incoming_flow_unit": UNIT_TMC,
        "inflow_window_hours": 72.0,
        "water_level": 348.6,
        "flow_rate": 270000.0,
        "upstream_water_level": 351.2,
        "downstream_water_level": 344.8,
        "rainfall": 185.0,
        "rainfall_duration": 36.0,
        "previous_flow": 58.0,
        "flow_trend": None,
    },
    "geography": {
        "latitude": 16.3067,
        "longitude": 80.1433,
        "affected_area": 940.0,
        "upstream_distance": 42.0,
        "downstream_distance": 18.0,
    },
    "population": {
        "total_population": 125000,
        "vulnerable_population": 28000,
        "children": 21000,
        "elderly": 18500,
        "persons_requiring_assistance": 9400,
    },
    "infrastructure": {
        "roads": 640.0,
        "blocked_roads": 145.0,
        "damaged_roads": 210.0,
        "critical_roads": 180.0,
        "bridges": 22,
        "bridges_affected": 9,
        "alternative_routes": 3,
        "hospitals": 6,
        "schools": 34,
        "police_stations": 9,
        "fire_stations": 7,
        "emergency_centers": 2,
        "power_stations": 4,
        "communication_towers": 12,
        "water_facilities": 18,
        "shelters": 9,
    },
    "resources": {
        "rescue_teams": 30,
        "medical_teams": 12,
        "doctors": 48,
        "nurses": 130,
        "ambulances": 16,
        "vehicles": 48,
        "boats": 15,
        "shelters": 9,
        "shelter_capacity": 6000,
        "food_meals": 180000.0,
        "water_litres": 1500000.0,
        "medicines_kits": 250,
    },
    "economics": {
        "land_value": 4.2e9,
        "crop_value": 1.8e9,
        "infrastructure_value": 6.5e9,
        "business_value": 3.1e9,
        "industrial_value": 2.4e9,
        "agricultural_area": 620.0,
        "crop_area": 540.0,
        "livestock": 14500,
        "livestock_value": 3.2e8,
    },
    "locations": [
        {
            "location_id": "L1",
            "name": "Upstream Riverside Settlement",
            "population": 48000,
            "distance_km": 12.0,
            "travel_time_hours": 1.2,
            "vulnerability_index": 0.55,
            "critical_infrastructure_count": 4,
            "road_access": "PARTIAL",
            "latitude": 16.3067,
            "longitude": 80.1433,
        },
        {
            "location_id": "L2",
            "name": "Downstream Delta Cluster",
            "population": 36000,
            "distance_km": 28.0,
            "travel_time_hours": 2.8,
            "vulnerability_index": 0.72,
            "critical_infrastructure_count": 6,
            "road_access": "BLOCKED",
            "latitude": 16.2981,
            "longitude": 80.1372,
        },
        {
            "location_id": "L3",
            "name": "Agricultural Tributary Villages",
            "population": 26000,
            "distance_km": 45.0,
            "travel_time_hours": 4.5,
            "vulnerability_index": 0.45,
            "critical_infrastructure_count": 2,
            "road_access": "PARTIAL",
            "latitude": 16.2861,
            "longitude": 80.1519,
        },
        {
            "location_id": "L4",
            "name": "Low-Lying Peripheral Ward",
            "population": 15000,
            "distance_km": 20.0,
            "travel_time_hours": 2.0,
            "vulnerability_index": 0.85,
            "critical_infrastructure_count": 1,
            "road_access": "OPEN",
            "latitude": 16.2745,
            "longitude": 80.1290,
        },
    ],
}

# Interactive-mode prompts, in the order they are asked. Each entry is
# (section, field, prompt text, unit label, is_required).
INTERACTIVE_PROMPTS: tuple[tuple[str, str, str, str, bool], ...] = (
    ("event", "event_id", "Event ID", "id", True),
    ("event", "region", "Region", "name", True),
    ("event", "location", "Location description", "name", False),
    ("dam", "dam_name", "Dam / barrage name", "name", False),
    ("dam", "dam_capacity", "Dam capacity", "TMC", True),
    ("dam", "current_storage", "Current storage", "TMC", True),
    ("dam", "incoming_flow", "Incoming flood flow", "see unit", True),
    ("dam", "incoming_flow_unit", "Incoming flow unit (TMC/TMC/day/m3/s/KCUS)", "unit", True),
    ("dam", "inflow_window_hours", "Integration window for rate units", "hours", False),
    ("dam", "rainfall", "Rainfall", "mm", False),
    ("dam", "previous_flow", "Previous incoming flow (same unit)", "same unit", False),
    ("geography", "affected_area", "Affected area", "km2", False),
    ("geography", "upstream_distance", "Upstream distance", "km", False),
    ("geography", "latitude", "Latitude", "deg", False),
    ("geography", "longitude", "Longitude", "deg", False),
    ("population", "total_population", "Total population", "people", True),
    ("population", "vulnerable_population", "Vulnerable population", "people", False),
    ("infrastructure", "roads", "Total road length", "km", False),
    ("infrastructure", "blocked_roads", "Blocked road length", "km", False),
    ("infrastructure", "bridges", "Number of bridges", "count", False),
    ("resources", "rescue_teams", "Available rescue teams", "teams", True),
    ("resources", "medical_teams", "Available medical teams", "teams", True),
    ("resources", "doctors", "Available doctors", "doctors", False),
    ("resources", "vehicles", "Available vehicles", "vehicles", True),
    ("resources", "boats", "Available boats", "boats", True),
    ("resources", "shelters", "Available shelters", "shelters", True),
    ("resources", "shelter_capacity", "Shelter capacity", "people", False),
)

UNAVAILABLE_TOKENS: frozenset[str] = frozenset({"n", "na", "n/a", "-", "none", "unavailable"})


def _prompt(label: str, unit: str, current: Any, required: bool) -> Any:
    """
    Ask one interactive question.

    Args:
        label: Field label.
        unit: Unit hint printed with the prompt.
        current: Current value offered as the default.
        required: Whether an empty answer is rejected.

    Returns:
        The entered value, `current` when the answer is blank, or None when the
        user marked the optional field unavailable.
    """
    suffix = f" [{unit}]" if unit else ""
    while True:
        shown = "(unavailable)" if current is None else current
        answer = input(f"{label}{suffix} (current: {shown}): ").strip()
        if not answer:
            if current is not None or not required:
                return current
            print("  This field is required. Enter a value or 'na' if unavailable.")
            continue
        if answer.lower() in UNAVAILABLE_TOKENS:
            if required:
                print("  This field is required; 'na' is not accepted.")
                continue
            return None
        try:
            return int(answer) if answer.lstrip("-").isdigit() else float(answer)
        except ValueError:
            try:
                return int(float(answer))
            except ValueError:
                print(f"  '{answer}' is not a number. Try again.")


def _build_section(cls, payload: dict[str, Any] | None, provenance: dict[str, Any],
                   prefix: str) -> Any:
    """
    Build one input record from a dictionary section, recording provenance.

    Args:
        cls: The dataclass to construct.
        payload: Mapping of field name to value (may be None).
        provenance: Output mapping that records the DataSource of each field.
        prefix: Dotted prefix used for the provenance keys.

    Returns:
        An instance of `cls` with every declared field present.
    """
    payload = payload or {}
    known = {f.name for f in cls.__dataclass_fields__.values()}
    unknown = sorted(set(payload) - known)
    if unknown:
        print(f"[input] ignoring unknown field(s) in '{prefix}': {', '.join(unknown)}")
    kwargs: dict[str, Any] = {}
    for name in known:
        if name in payload:
            kwargs[name] = payload[name]
            provenance[f"{prefix}.{name}"] = DataSource.USER_PROVIDED
        else:
            kwargs[name] = None
            provenance[f"{prefix}.{name}"] = DataSource.DATA_UNAVAILABLE
    return cls(**kwargs)


def _record_provenance(record: Any, provenance: dict[str, Any], prefix: str) -> None:
    """Record DATA_UNAVAILABLE for every field left as None on a record."""
    for name, value in record.__dict__.items():
        provenance.setdefault(f"{prefix}.{name}", DataSource.DATA_UNAVAILABLE)
        if value is None:
            provenance[f"{prefix}.{name}"] = DataSource.DATA_UNAVAILABLE


def _build_locations(
    raw_locations: Any, provenance: dict[str, Any]
) -> list[LocationInput]:
    """
    Build the response-location list.

    Args:
        raw_locations: List of location dictionaries.
        provenance: Provenance output mapping.

    Returns:
        List of `LocationInput` records, truncated to MAX_RUNTIME_LOCATIONS.
    """
    if not isinstance(raw_locations, list) or not raw_locations:
        provenance["locations"] = DataSource.DATA_UNAVAILABLE
        return []
    locations: list[LocationInput] = []
    for index, entry in enumerate(raw_locations):
        if not isinstance(entry, dict):
            continue
        record = _build_section(
            LocationInput, entry, provenance, f"locations[{index}]"
        )
        locations.append(record)
    if len(locations) > MAX_RUNTIME_LOCATIONS:
        print(
            f"[input] {len(locations)} locations supplied; the quantum path is "
            f"limited to {MAX_RUNTIME_LOCATIONS} decision variables, so the "
            "highest-population locations were kept."
        )
        locations.sort(key=lambda loc: loc.population or 0, reverse=True)
        locations = locations[:MAX_RUNTIME_LOCATIONS]
    provenance["locations"] = DataSource.USER_PROVIDED if locations else DataSource.DATA_UNAVAILABLE
    return locations


def collect_runtime_input(
    mode: str = "dictionary",
    runtime_data: dict[str, Any] | None = None,
    interactive_defaults: dict[str, Any] | None = None,
) -> RuntimeEvent:
    """
    Collect one runtime event.

    WHAT:
        Build a validated `RuntimeEvent` from either an interactive prompt
        session or a Python dictionary.

    WHY:
        The engine must be driven by runtime values, and it must be testable
        without a human. Both modes therefore converge on the same builder so
        there is exactly one place where input becomes a model.

    HOW:
        MODE A (`interactive`) asks each prompt in `INTERACTIVE_PROMPTS`,
        pre-filled from `interactive_defaults`. MODE B (`dictionary`) takes the
        mapping as-is. In both cases omitted keys become `None`, and the
        provenance map records `USER_PROVIDED` or `DATA_UNAVAILABLE` per field.

    Args:
        mode: `"dictionary"` or `"interactive"`.
        runtime_data: The runtime payload (required for dictionary mode).
        interactive_defaults: Seed values for interactive mode.

    Returns:
        A fully populated `RuntimeEvent` with an explicit provenance map.

    Raises:
        ValueError: If the mode is unknown or required data is missing.
    """
    if mode not in {"dictionary", "interactive"}:
        raise ValueError(f"unknown input mode '{mode}'")

    payload: dict[str, Any] = {}
    if mode == "interactive":
        payload = _collect_interactive_payload(interactive_defaults or DEMO_RUNTIME_DATA)
    else:
        if not isinstance(runtime_data, dict):
            raise ValueError(
                "dictionary mode requires runtime_data={...}; use "
                "mode='interactive' to be prompted instead."
            )
        payload = runtime_data

    provenance: dict[str, DataSource] = {}
    event = _build_section(EventInfo, payload.get("event"), provenance, "event")
    dam = _build_section(DamInfo, payload.get("dam"), provenance, "dam")
    geography = _build_section(GeoInfo, payload.get("geography"), provenance, "geography")
    population = _build_section(
        PopulationInfo, payload.get("population"), provenance, "population"
    )
    infrastructure = _build_section(
        InfrastructureInfo, payload.get("infrastructure"), provenance, "infrastructure"
    )
    resources = _build_section(
        ResourceAvailability, payload.get("resources"), provenance, "resources"
    )
    economics = _build_section(EconomicInfo, payload.get("economics"), provenance, "economics")
    locations = _build_locations(payload.get("locations"), provenance)

    for record, prefix in (
        (event, "event"),
        (dam, "dam"),
        (geography, "geography"),
        (population, "population"),
        (infrastructure, "infrastructure"),
        (resources, "resources"),
        (economics, "economics"),
    ):
        _record_provenance(record, provenance, prefix)

    return RuntimeEvent(
        event=event,
        dam=dam,
        geography=geography,
        population=population,
        infrastructure=infrastructure,
        resources=resources,
        economics=economics,
        locations=locations,
        data_provenance=provenance,
    )


def _collect_interactive_payload(defaults: dict[str, Any]) -> dict[str, Any]:
    """
    Run the interactive prompt session and return the resulting payload.

    Args:
        defaults: Seed payload; existing values become the offered defaults.

    Returns:
        A payload dictionary in the same shape as `DEMO_RUNTIME_DATA`.
    """
    print("=" * 60)
    print("Q-FLARE RUNTIME DATA ENTRY (MODE A - INTERACTIVE)")
    print("=" * 60)
    print("Press Enter to keep the current value.")
    print("Type 'na' to mark an OPTIONAL field as unavailable.")
    print("")

    payload: dict[str, Any] = {
        key: dict(value) if isinstance(value, dict) else value
        for key, value in defaults.items()
    }
    for section, field, label, unit, required in INTERACTIVE_PROMPTS:
        current = payload.get(section, {}).get(field)
        payload.setdefault(section, {})
        payload[section][field] = _prompt(label, unit, current, required)

    location_count = len(payload.get("locations", []))
    payload["locations"] = payload.get("locations", [])[:location_count]
    print("")
    print("[input] interactive entry complete. Locations are reused from the "
          f"{location_count} seeded record(s); edit them directly for a real event.")
    return payload


# -----------------------------------------------------------------------------
# 3.2 - Build the runtime event for this session
# -----------------------------------------------------------------------------

RUNTIME_EVENT = collect_runtime_input(
    mode=RUNTIME_INPUT_MODE,
    runtime_data=DEMO_RUNTIME_DATA if RUNTIME_INPUT_MODE == "dictionary" else None,
    interactive_defaults=DEMO_RUNTIME_DATA,
)

print("=" * 60)
print("RUNTIME INPUT RECEIVED")
print("=" * 60)
print(f"Event ID          : {RUNTIME_EVENT.event.event_id}")
print(f"Timestamp         : {RUNTIME_EVENT.event.timestamp}")
print(f"Region            : {RUNTIME_EVENT.event.region}")
print(f"Location          : {RUNTIME_EVENT.event.location}")
print(f"Data source label : {RUNTIME_EVENT.event.data_source}")
print(f"Entry mode        : {RUNTIME_INPUT_MODE}")
print(f"Dam / barrage     : {RUNTIME_EVENT.dam.dam_name}")
print(f"Response locations: {len(RUNTIME_EVENT.locations)}")
print(
    f"Tracked fields    : {len(RUNTIME_EVENT.data_provenance)} "
    f"({sum(1 for s in RUNTIME_EVENT.data_provenance.values() if s is DataSource.USER_PROVIDED)}"
    f" user-provided, "
    f"{sum(1 for s in RUNTIME_EVENT.data_provenance.values() if s is DataSource.DATA_UNAVAILABLE)}"
    f" unavailable)"
)
print("")
print("Runtime input status: RECEIVED")
print("=" * 60)


## SECTION 4 — INPUT VALIDATION ENGINE

Validation is the first gate of the pipeline. It runs **before any
mathematics**, it reports **every** problem it can find (not only the first), and
it never repairs data silently.

It checks five classes of problem:

| Class | Examples |
| --- | --- |
| Presence | missing `event_id`, missing `total_population` |
| Type / range | `rainfall = -5`, `population = -10`, `rescue_teams = "many"` |
| Unit | `incoming_flow_unit = "gallons"` |
| Relationship | `current_storage > dam_capacity`, `children > total_population` |
| Cross-section consistency | location populations not reconciling with the regional total |

The outcome is one of three statuses:

* `PASSED` — no error, no warning.
* `WARNING` — usable, but a field is unavailable or internally inconsistent. The
  pipeline continues and the affected outputs are reported as
  `DATA_UNAVAILABLE` or `MODEL_ESTIMATED`.
* `FAILED` — at least one error. The pipeline refuses to continue, because a
  flood decision built on a broken input is worse than no decision.


In [ ]:
# =============================================================================
# SECTION 4 - INPUT VALIDATION ENGINE
# =============================================================================
#
# Collects every field of the runtime event, then checks presence, type/range,
# unit, relationship and cross-section consistency. All problems are reported.

ERROR = "ERROR"
WARNING = "WARNING"
INFO = "INFO"

STATUS_PASSED = "PASSED"
STATUS_WARNING = "WARNING"
STATUS_FAILED = "FAILED"

# Fields the pipeline cannot run without.
REQUIRED_FIELDS: tuple[str, ...] = (
    "event.event_id",
    "event.timestamp",
    "event.region",
    "dam.dam_capacity",
    "dam.current_storage",
    "dam.incoming_flow",
    "dam.incoming_flow_unit",
    "population.total_population",
)

# Population and resource fields whose sum must not exceed the total population.
POPULATION_SUBFIELDS: tuple[str, ...] = (
    "vulnerable_population",
    "children",
    "elderly",
    "persons_requiring_assistance",
)

# Resource / infrastructure fields that must be non-negative whole numbers.
RESOURCE_FIELDS: tuple[str, ...] = (
    "rescue_teams",
    "medical_teams",
    "doctors",
    "nurses",
    "ambulances",
    "vehicles",
    "boats",
    "shelters",
    "shelter_capacity",
    "medicines_kits",
)

# Tolerance for reconciling per-location populations with the regional total.
POPULATION_RECONCILIATION_TOLERANCE_PCT = 5.0


class QFlareValidationError(RuntimeError):
    """Raised when runtime input fails validation and the run cannot continue."""


@dataclass(frozen=True)
class ValidationIssue:
    """
    One validation finding.

    Attributes:
        field: Dotted field path, e.g. "dam.current_storage".
        severity: ERROR, WARNING or INFO.
        message: Human-readable description of the problem.
    """

    field: str
    severity: str
    message: str


@dataclass
class ValidationReport:
    """
    Aggregated validation outcome.

    Attributes:
        issues: Every finding, in discovery order.
        total_fields: Number of fields inspected.
        valid: Fields present and free of errors.
        invalid: Fields carrying at least one error.
        missing: Fields that were not supplied.
    """

    issues: list[ValidationIssue] = field(default_factory=list)
    total_fields: int = 0
    valid: int = 0
    invalid: int = 0
    missing: int = 0

    @property
    def errors(self) -> list[ValidationIssue]:
        """Findings with ERROR severity."""
        return [issue for issue in self.issues if issue.severity == ERROR]

    @property
    def warnings(self) -> list[ValidationIssue]:
        """Findings with WARNING severity."""
        return [issue for issue in self.issues if issue.severity == WARNING]

    @property
    def status(self) -> str:
        """PASSED, WARNING or FAILED."""
        if self.errors:
            return STATUS_FAILED
        if self.warnings:
            return STATUS_WARNING
        return STATUS_PASSED

    @property
    def can_proceed(self) -> bool:
        """True when the pipeline may continue (no ERROR findings)."""
        return not self.errors

    def add(self, field_path: str, severity: str, message: str) -> None:
        """Record one finding."""
        self.issues.append(ValidationIssue(field_path, severity, message))


def collect_fields(runtime: RuntimeEvent) -> dict[str, Any]:
    """
    Flatten a `RuntimeEvent` into a dotted-path field map.

    Args:
        runtime: The runtime event.

    Returns:
        Mapping of dotted field path to value (None when not supplied).
    """
    fields: dict[str, Any] = {}
    for record_name in (
        "event",
        "dam",
        "geography",
        "population",
        "infrastructure",
        "resources",
        "economics",
    ):
        record = getattr(runtime, record_name)
        for name, value in record.__dict__.items():
            fields[f"{record_name}.{name}"] = value
    for index, location in enumerate(runtime.locations):
        for name, value in location.__dict__.items():
            fields[f"locations[{index}].{name}"] = value
    return fields


def _check_numeric(
    report: ValidationReport,
    fields: dict[str, Any],
    path: str,
    *,
    integer: bool = False,
    minimum: float | None = 0.0,
    required: bool = False,
) -> float | None:
    """
    Validate one numeric field.

    Args:
        report: Report to append findings to.
        fields: The flattened field map.
        path: Dotted path of the field.
        integer: Require a whole number.
        minimum: Inclusive lower bound, or None for unbounded.
        required: Missing values become errors instead of warnings.

    Returns:
        The value as a float, or None when it is absent or invalid.
    """
    value = fields.get(path)
    if value is None:
        if required:
            report.add(path, ERROR, "required field is missing")
        return None
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        report.add(path, ERROR, f"expected a number, got {type(value).__name__} ({value!r})")
        return None
    numeric = float(value)
    if not math.isfinite(numeric):
        report.add(path, ERROR, f"value is not finite ({value!r})")
        return None
    if integer and not float(numeric).is_integer():
        report.add(path, ERROR, f"expected a whole number, got {value!r}")
        return None
    if minimum is not None and numeric < minimum:
        report.add(path, ERROR, f"value must be >= {minimum}, got {numeric:g}")
        return None
    return numeric


def validate_runtime_input(runtime: RuntimeEvent) -> ValidationReport:
    """
    Validate a runtime event and return a structured report.

    Args:
        runtime: The runtime event to validate.

    Returns:
        A `ValidationReport` with every finding. `status` is PASSED, WARNING or
        FAILED; `can_proceed` is False only on FAILED.
    """
    fields = collect_fields(runtime)
    report = ValidationReport(total_fields=len(fields))

    for path in REQUIRED_FIELDS:
        if fields.get(path) is None or fields.get(path) == "":
            report.add(path, ERROR, "required field is missing")
    report.missing = sum(1 for value in fields.values() if value is None)

    # -- Event identification ------------------------------------------------
    for path in ("event.event_id", "event.region", "event.location", "event.data_source"):
        value = fields.get(path)
        if value is not None and (not isinstance(value, str) or not value.strip()):
            report.add(path, ERROR, "must be a non-empty string")

    raw_timestamp = fields.get("event.timestamp")
    if raw_timestamp is not None:
        try:
            parsed = datetime.fromisoformat(str(raw_timestamp).replace("Z", "+00:00"))
        except ValueError as exc:
            report.add(
                "event.timestamp", ERROR, f"not a valid ISO-8601 timestamp ({exc})"
            )
        else:
            if parsed.tzinfo is None:
                parsed = parsed.replace(tzinfo=timezone.utc)
            if parsed > datetime.now(timezone.utc) + timedelta(days=1):
                report.add(
                    "event.timestamp",
                    WARNING,
                    "timestamp is more than 24 h in the future - check the clock",
                )

    # -- Dam / hydrology -----------------------------------------------------
    capacity = _check_numeric(report, fields, "dam.dam_capacity", required=True)
    storage = _check_numeric(report, fields, "dam.current_storage", required=True)
    _check_numeric(report, fields, "dam.incoming_flow", required=True)
    _check_numeric(report, fields, "dam.rainfall", minimum=0.0)
    _check_numeric(report, fields, "dam.rainfall_duration", minimum=0.0)
    _check_numeric(report, fields, "dam.water_level")
    _check_numeric(report, fields, "dam.flow_rate", minimum=0.0)
    _check_numeric(report, fields, "dam.upstream_water_level")
    _check_numeric(report, fields, "dam.downstream_water_level")
    _check_numeric(report, fields, "dam.previous_flow", minimum=0.0)

    flow_unit = fields.get("dam.incoming_flow_unit")
    if flow_unit is not None and str(flow_unit) not in ALL_FLOW_UNITS:
        report.add(
            "dam.incoming_flow_unit",
            ERROR,
            f"unsupported unit '{flow_unit}'; expected one of "
            f"{', '.join(sorted(ALL_FLOW_UNITS))}",
        )

    trend = fields.get("dam.flow_trend")
    if trend is not None and str(trend).upper() not in {"RISING", "STEADY", "FALLING"}:
        report.add(
            "dam.flow_trend",
            WARNING,
            f"unrecognised trend '{trend}'; it will be derived from previous_flow",
        )

    # A rate is only comparable with a storage volume once it has been integrated
    # over an explicit window; the conversion itself happens in SECTION 5.
    window = _check_numeric(report, fields, "dam.inflow_window_hours", minimum=0.01)
    if (
        flow_unit in FLOW_RATE_UNITS
        and window is None
        and "dam.inflow_window_hours" not in {issue.field for issue in report.errors}
    ):
        report.add(
            "dam.inflow_window_hours",
            ERROR,
            f"an integration window is required to convert a rate expressed in "
            f"{flow_unit} into a comparable flood volume",
        )

    if capacity is not None and storage is not None:
        if storage > capacity:
            report.add(
                "dam.current_storage",
                ERROR,
                f"current storage ({storage:g} TMC) exceeds dam capacity "
                f"({capacity:g} TMC)",
            )
        elif capacity > 0 and storage / capacity > 0.99:
            report.add(
                "dam.current_storage",
                WARNING,
                "reservoir is effectively full - no flood space available for routing",
            )

    previous = fields.get("dam.previous_flow")
    if previous is not None and not isinstance(previous, (int, float)):
        report.add("dam.previous_flow", ERROR, "previous_flow must be a number")
    elif (
        previous is not None
        and fields.get("dam.incoming_flow") is not None
        and float(previous) > 0
        and float(fields["dam.incoming_flow"]) < 0
    ):
        report.add(
            "dam.incoming_flow",
            ERROR,
            "incoming flow cannot fall below zero while the previous reading is positive",
        )

    # -- Geography -----------------------------------------------------------
    latitude = _check_numeric(report, fields, "geography.latitude")
    longitude = _check_numeric(report, fields, "geography.longitude")
    _check_numeric(report, fields, "geography.affected_area", minimum=0.0)
    _check_numeric(report, fields, "geography.upstream_distance", minimum=0.0)
    _check_numeric(report, fields, "geography.downstream_distance", minimum=0.0)
    if latitude is not None and not (-90.0 <= latitude <= 90.0):
        report.add("geography.latitude", ERROR, f"{latitude:g} is outside [-90, 90]")
    if longitude is not None and not (-180.0 <= longitude <= 180.0):
        report.add("geography.longitude", ERROR, f"{longitude:g} is outside [-180, 180]")
    if (latitude is None) != (longitude is None):
        report.add(
            "geography",
            WARNING,
            "latitude and longitude must be supplied together; GIS analysis will be LIMITED",
        )

    # -- Population ----------------------------------------------------------
    total_population = _check_numeric(
        report, fields, "population.total_population", integer=True, required=True
    )
    sub_populations: dict[str, float] = {}
    for name in POPULATION_SUBFIELDS:
        sub_populations[name] = _check_numeric(
            report, fields, f"population.{name}", integer=True, minimum=0.0
        ) or 0.0
    if total_population is not None:
        for name, value in sub_populations.items():
            if value > total_population:
                report.add(
                    f"population.{name}",
                    ERROR,
                    f"{name} ({value:g}) exceeds total_population ({total_population:g})",
                )

    # -- Infrastructure ------------------------------------------------------
    road_lengths: dict[str, float] = {}
    for name in ("roads", "blocked_roads", "damaged_roads", "critical_roads"):
        road_lengths[name] = _check_numeric(
            report, fields, f"infrastructure.{name}", minimum=0.0
        ) or 0.0
    for name in (
        "bridges",
        "bridges_affected",
        "alternative_routes",
        "hospitals",
        "schools",
        "police_stations",
        "fire_stations",
        "emergency_centers",
        "power_stations",
        "communication_towers",
        "water_facilities",
        "shelters",
    ):
        _check_numeric(report, fields, f"infrastructure.{name}", integer=True, minimum=0.0)
    if road_lengths["blocked_roads"] and road_lengths["roads"]:
        if road_lengths["blocked_roads"] > road_lengths["roads"]:
            report.add(
                "infrastructure.blocked_roads",
                ERROR,
                f"blocked ({road_lengths['blocked_roads']:g} km) exceeds total road "
                f"length ({road_lengths['roads']:g} km)",
            )
    if road_lengths["damaged_roads"] and road_lengths["roads"]:
        if road_lengths["damaged_roads"] > road_lengths["roads"]:
            report.add(
                "infrastructure.damaged_roads",
                ERROR,
                f"damaged ({road_lengths['damaged_roads']:g} km) exceeds total road "
                f"length ({road_lengths['roads']:g} km)",
            )
    bridges = fields.get("infrastructure.bridges")
    bridges_affected = fields.get("infrastructure.bridges_affected")
    if (
        isinstance(bridges, (int, float))
        and isinstance(bridges_affected, (int, float))
        and not isinstance(bridges, bool)
        and not isinstance(bridges_affected, bool)
        and float(bridges_affected) > float(bridges)
    ):
        report.add(
            "infrastructure.bridges_affected",
            ERROR,
            f"affected bridges ({bridges_affected:g}) exceed total bridges ({bridges:g})",
        )

    # -- Resources -----------------------------------------------------------
    for name in RESOURCE_FIELDS:
        _check_numeric(
            report, fields, f"resources.{name}", integer=True, minimum=0.0
        )
    _check_numeric(report, fields, "resources.food_meals", minimum=0.0)
    _check_numeric(report, fields, "resources.water_litres", minimum=0.0)

    shelters = fields.get("resources.shelters")
    shelter_capacity = fields.get("resources.shelter_capacity")
    if (
        isinstance(shelters, (int, float))
        and isinstance(shelter_capacity, (int, float))
        and not isinstance(shelters, bool)
        and not isinstance(shelter_capacity, bool)
        and float(shelters) > 0
        and float(shelter_capacity) > 0
        and float(shelter_capacity) / float(shelters) > 5000
    ):
        report.add(
            "resources.shelter_capacity",
            WARNING,
            f"implied capacity of {float(shelter_capacity) / float(shelters):,.0f} people "
            "per shelter exceeds any realistic planning figure - verify the inputs",
        )

    # -- Economics -----------------------------------------------------------
    for name in (
        "land_value",
        "crop_value",
        "infrastructure_value",
        "business_value",
        "industrial_value",
        "agricultural_area",
        "crop_area",
        "livestock_value",
    ):
        _check_numeric(report, fields, f"economics.{name}", minimum=0.0)
    _check_numeric(report, fields, "economics.livestock", integer=True, minimum=0.0)
    crop_area = fields.get("economics.crop_area")
    agricultural_area = fields.get("economics.agricultural_area")
    if (
        isinstance(crop_area, (int, float))
        and isinstance(agricultural_area, (int, float))
        and not isinstance(crop_area, bool)
        and not isinstance(agricultural_area, bool)
        and float(crop_area) > float(agricultural_area)
    ):
        report.add(
            "economics.crop_area",
            ERROR,
            f"crop area ({crop_area:g} km2) exceeds agricultural area "
            f"({agricultural_area:g} km2)",
        )

    # -- Response locations --------------------------------------------------
    if not runtime.locations:
        report.add(
            "locations",
            WARNING,
            "no response locations supplied - the allocation stage will report "
            "DATA_UNAVAILABLE and no optimization is possible",
        )
    else:
        seen_ids: set[str] = set()
        for index, location in enumerate(runtime.locations):
            prefix = f"locations[{index}]"
            identifier = location.location_id
            if not isinstance(identifier, str) or not identifier.strip():
                report.add(f"{prefix}.location_id", ERROR, "location_id must be a non-empty string")
            elif identifier in seen_ids:
                report.add(f"{prefix}.location_id", ERROR, f"duplicate location_id '{identifier}'")
            else:
                seen_ids.add(identifier)
            if not isinstance(location.name, str) or not location.name.strip():
                report.add(f"{prefix}.name", ERROR, "name must be a non-empty string")
            if isinstance(location.population, bool) or not isinstance(
                location.population, (int, float)
            ):
                report.add(f"{prefix}.population", ERROR, "population must be a number")
            elif location.population < 0:
                report.add(
                    f"{prefix}.population",
                    ERROR,
                    f"population cannot be negative (got {location.population})",
                )
            if not 0.0 <= float(location.vulnerability_index) <= 1.0:
                report.add(
                    f"{prefix}.vulnerability_index",
                    ERROR,
                    f"vulnerability_index must be within [0, 1], got "
                    f"{location.vulnerability_index}",
                )
            if location.distance_km is not None and location.distance_km < 0:
                report.add(f"{prefix}.distance_km", ERROR, "distance_km cannot be negative")
            if location.travel_time_hours is not None and location.travel_time_hours < 0:
                report.add(
                    f"{prefix}.travel_time_hours", ERROR, "travel_time_hours cannot be negative"
                )
            if str(location.road_access).upper() not in {"OPEN", "PARTIAL", "BLOCKED"}:
                report.add(
                    f"{prefix}.road_access",
                    WARNING,
                    f"unrecognised road_access '{location.road_access}'",
                )
            if location.critical_infrastructure_count < 0:
                report.add(
                    f"{prefix}.critical_infrastructure_count",
                    ERROR,
                    "critical_infrastructure_count cannot be negative",
                )

        if total_population:
            located_population = sum(
                float(loc.population or 0) for loc in runtime.locations
            )
            deviation = abs(located_population - float(total_population))
            if deviation > float(total_population) * POPULATION_RECONCILIATION_TOLERANCE_PCT / 100:
                report.add(
                    "locations",
                    WARNING,
                    f"location populations sum to {located_population:,.0f} but "
                    f"population.total_population is {float(total_population):,.0f} "
                    f"({deviation / float(total_population) * 100:.1f}% deviation) - "
                    "location-level demand uses the location figures",
                )

    report.invalid = len({issue.field for issue in report.errors})
    report.valid = max(0, report.total_fields - report.missing - report.invalid)
    return report


def display_validation(report: ValidationReport) -> None:
    """Print the required INPUT VALIDATION block."""
    print("=" * 60)
    print("INPUT VALIDATION")
    print("=" * 60)
    print(f"Total fields : {report.total_fields}")
    print(f"Valid        : {report.valid}")
    print(f"Invalid      : {report.invalid}")
    print(f"Missing      : {report.missing}")
    print(f"Errors       : {len(report.errors)}")
    print(f"Warnings     : {len(report.warnings)}")
    print("")

    for issue in report.issues:
        print(f"[{issue.severity:<7}] {issue.field}: {issue.message}")
    if not report.issues:
        print("No errors, no warnings.")

    print("")
    print(f"Validation status: {report.status}")
    print("=" * 60)


VALIDATION_REPORT = validate_runtime_input(RUNTIME_EVENT)
display_validation(VALIDATION_REPORT)

if not VALIDATION_REPORT.can_proceed:
    raise QFlareValidationError(
        f"runtime input validation FAILED with {len(VALIDATION_REPORT.errors)} error(s): "
        + " | ".join(f"{issue.field}: {issue.message}" for issue in VALIDATION_REPORT.errors)
    )

print("[validation] pipeline may continue; unavailable fields will be reported "
      "as DATA_UNAVAILABLE rather than estimated silently.")


## SECTION 4B — ENGINE CONFIGURATION (EVERY THRESHOLD AND RATIO)

Nothing in the decision-support layer is a hidden magic number. Every
threshold, weight and emergency ratio lives in one of the dataclasses below,
is printed by the configuration cell, and is covered by the tests.

**These are configurable planning assumptions, not standards.** They are
modelled on published disaster-response planning ratios and must be replaced
with the ratios of the operating agency before the output is used operationally.
The report prints them alongside every figure derived from them.

> Every `MODEL_ESTIMATED` value in the report names the configuration field that
> produced it, so an auditor can trace any number back to its assumption.


In [ ]:
# =============================================================================
# SECTION 4B - ENGINE CONFIGURATION
# =============================================================================
#
# Every threshold, weight and emergency planning ratio used by the
# decision-support layer, in one inspectable place.


@dataclass
class FloodConfig:
    """
    Thresholds of the flood-condition model.

    Attributes:
        severity_weights: Weights of the five severity components. They are
            normalised at use time, so only their ratios matter.
        excess_reference_ratio: Incoming volume at which the excess-flow
            component saturates, expressed as a multiple of dam capacity.
        rainfall_reference_mm: Rainfall depth at which the rainfall component
            saturates.
        flow_rate_reference_min/max_m3s: Bounds of the logarithmic discharge
            scale. A log scale is used because river rating relationships are
            logarithmic, so the score moves sensibly over four orders of
            magnitude instead of saturating after a few hundred m3/s.
        trend_rising_bonus / trend_falling_penalty: Additive shift applied to the
            trend component.
        severity_bands: Upper bounds of NORMAL < LOW < MODERATE < HIGH < EXTREME.
    """

    severity_weights: dict[str, float] = field(
        default_factory=lambda: {
            "utilization": 0.30,
            "excess": 0.30,
            "rainfall": 0.20,
            "flow_rate": 0.15,
            "trend": 0.05,
        }
    )
    excess_reference_ratio: float = 2.0
    rainfall_reference_mm: float = 500.0
    flow_rate_reference_min_m3s: float = 100.0
    flow_rate_reference_max_m3s: float = 1.0e6
    trend_rising_bonus: float = 25.0
    trend_falling_penalty: float = 25.0
    trend_fluctuation_threshold_pct: float = 2.0
    severity_bands: tuple[float, ...] = (25.0, 50.0, 75.0)
    severity_labels: tuple[str, ...] = ("LOW", "MODERATE", "HIGH", "EXTREME")


@dataclass
class ArrivalConfig:
    """
    Flood arrival-time and response-window parameters.

    Attributes:
        assumed_flow_velocity_kmh: Fallback kinematic wave-front velocity used
            only when the runtime input supplies no velocity. Using it marks
            the arrival time MODEL_ESTIMATED with MEDIUM confidence.
        evacuation_lead_hours: Hours reserved between the flood arrival and the
            evacuation deadline.
        at_risk_margin: Available response time below this multiple of the
            required time is flagged AT_RISK.
    """

    assumed_flow_velocity_kmh: float = 18.0
    evacuation_lead_hours: float = 3.0
    at_risk_margin: float = 2.0
    sensor_timing_confidence: str = "HIGH"


@dataclass
class AreaConfig:
    """
    Affected-area and GIS parameters.

    Attributes:
        population_density_per_km2: Density used when the runtime input does
            not state the affected area.
        land_use_fractions: Split of the affected area into land-use classes.
        freeboard_depth_m: Height of the minimum built-up floor level above the
            local inundation datum, used for the inundation-depth estimate.
        max_plausible_inundation_depth_m: Deepest water depth treated as
            physically plausible for an inundation estimate. It bounds the
            reported depth and is the depth used to convert an excess volume
            into an implied inundated area.
    """

    population_density_per_km2: float = 133.0
    land_use_fractions: dict[str, float] = field(
        default_factory=lambda: {
            "urban": 0.22,
            "rural": 0.31,
            "agricultural": 0.29,
            "residential": 0.18,
        }
    )
    freeboard_depth_m: float = 0.5
    max_plausible_inundation_depth_m: float = 4.0


@dataclass
class PopulationConfig:
    """
    Population-impact parameters.

    Attributes:
        max_exposure_ratio: Largest share of the population that can be
            exposed, reached at severity 100.
        high_risk_fraction: Share of the exposed population inside the highest
            risk band (applied to the deepest, worst-affected fraction of area).
        vulnerable_multiplier: Vulnerability multiplier applied to the exposed
            vulnerable population, because vulnerable groups are evacuated first.
    """

    max_exposure_ratio: float = 0.85
    high_risk_fraction: float = 0.25
    vulnerable_multiplier: float = 1.0


@dataclass
class RiskConfig:
    """
    Risk-engine parameters.

    Attributes:
        weights: Relative weight of the seven risk components; normalised at
            use time.
        bands: Upper bounds of LOW < MEDIUM < HIGH < DANGER.
        labels: The four risk levels.
    """

    weights: dict[str, float] = field(
        default_factory=lambda: {
            "flood": 0.25,
            "population": 0.20,
            "infrastructure": 0.15,
            "road": 0.15,
            "medical": 0.10,
            "evacuation": 0.08,
            "economic": 0.07,
        }
    )
    bands: tuple[float, ...] = (24.0, 49.0, 74.0)
    labels: tuple[str, ...] = ("LOW", "MEDIUM", "HIGH", "DANGER")


@dataclass
class DemandConfig:
    """
    Resource-demand planning ratios.

    Every ratio is per a stated denominator so the arithmetic is auditable.
    `relief_horizon_days` drives all consumable quantities.
    """

    rescue_teams_per_1000_evacuees: float = 1.0
    rescue_vehicles_per_team: float = 1.5
    boats_per_1000_evacuees: float = 0.6
    medical_teams_per_10000_evacuees: float = 1.0
    doctors_per_10000_evacuees: float = 3.0
    nurses_per_doctor: float = 2.0
    ambulances_per_medical_team: float = 1.2
    water_litres_per_person_per_day: float = 3.0
    food_meals_per_person_per_day: float = 2.0
    medicines_kits_per_1000_persons: float = 5.0
    blankets_per_person: float = 0.5
    sanitation_kits_per_100_persons: float = 1.0
    generators_per_shelter: float = 1.5
    fuel_litres_per_team_per_day: float = 120.0
    shelter_capacity_persons: int = 500
    shelter_occupancy_ratio: float = 0.85
    relief_horizon_days: float = 3.0


@dataclass
class AgricultureConfig:
    """
    Agricultural damage fractions.

    Attributes:
        crop_damage_fraction: Share of the standing crop value lost at
            severity 100.
        livestock_damage_fraction: Share of the livestock value lost at
            severity 100.
        standing_crop_value_per_km2: Reference used to cross-check a supplied
            crop value against the supplied crop area.
    """

    crop_damage_fraction: float = 0.65
    livestock_damage_fraction: float = 0.40
    standing_crop_value_per_km2: float = 2.4e6


@dataclass
class OptimizationConfig:
    """
    Optimization-preprocessor parameters.

    Attributes:
        key_resource: The resource whose availability limits the number of
            deployment slots, i.e. the quantum problem's hard constraint.
        units_per_deployment_slot: How many units of the key resource are needed
            to sustain one primary deployment at a location.
        location_weights: Weights of the location-urgency composite.
        population_weight: Weight of the normalized population term inside the
            objective score, consumed by the preserved QUBO builder.
        population_share_floor: Minimum share of a location's demand that is
            reported even when it receives no deployment.
    """

    key_resource: str = "rescue_teams"
    units_per_deployment_slot: int = 8
    location_weights: dict[str, float] = field(
        default_factory=lambda: {
            "population": 0.20,
            "vulnerability": 0.15,
            "infrastructure": 0.15,
            "road_access": 0.15,
            "flood_severity": 0.20,
            "accessibility": 0.15,
        }
    )
    population_share_weights: dict[str, float] = field(
        default_factory=lambda: {"population": 0.70, "travel": 0.30}
    )
    population_weight: float = 0.5
    population_share_floor: float = 0.0


@dataclass
class QuantumConfig:
    """
    QAOA execution parameters. Every stochastic element is seeded.

    Attributes:
        depth: QAOA depth `p`.
        shots: Measurement repetitions on the simulator.
        seed: Master seed for parameter restarts, transpilation and sampling.
        optimizer_method: SciPy derivative-free method.
        optimizer_maxiter: Maximum iterations per restart.
        optimizer_restarts: Number of seeded random restarts.
    """

    depth: int = 2
    shots: int = 2048
    seed: int = 20261001
    optimizer_method: str = "COBYLA"
    optimizer_maxiter: int = 220
    optimizer_restarts: int = 3


@dataclass
class ResponseConfig:
    """
    Response-planning parameters.

    Attributes:
        dispatch_preparation_hours: Time to mobilise a team after the order.
        medical_dispatch_preparation_hours: Same for medical teams.
        evacuation_rate_persons_per_hour: Achievable evacuation throughput of
            the road/boat network.
        relief_unload_hours: Unloading time at the delivery point.
        critical_deadline_margin_hours: Slack added to the flood arrival time
            to form the response deadline.
    """

    dispatch_preparation_hours: float = 0.75
    medical_dispatch_preparation_hours: float = 1.0
    evacuation_rate_persons_per_hour: float = 180.0
    relief_unload_hours: float = 1.0
    critical_deadline_margin_hours: float = 0.5


@dataclass
class CostConfig:
    """
    Unit costs in INR. All are planning assumptions and are printed with the
    budget. They are not government figures.

    Attributes:
        unit_costs: Cost per unit of each resource.
        damage_repair_share: Share of estimated damage covered by the response
            budget as repair cost.
        indirect_loss_ratio: Indirect (business interruption, lost livelihood)
            loss as a multiple of direct damage.
        currency: Currency label used in every report line.
    """

    currency: str = UNIT_INR
    unit_costs: dict[str, float] = field(
        default_factory=lambda: {
            "rescue_team_day": 45000.0,
            "medical_team_day": 60000.0,
            "doctor_day": 25000.0,
            "nurse_day": 9000.0,
            "ambulance_day": 8000.0,
            "vehicle_day": 6000.0,
            "boat_day": 12000.0,
            "meal": 60.0,
            "water_litre": 8.0,
            "medicine_kit": 4500.0,
            "blanket": 350.0,
            "sanitation_kit": 250.0,
            "generator_day": 3500.0,
            "fuel_litre": 105.0,
            "shelter_person_day": 120.0,
            "temporary_shelter": 8500.0,
            "transport_person_km": 12.0,
            "road_repair_km": 1.8e6,
            "bridge_repair": 4.5e7,
            "power_restoration_kw": 900.0,
            "water_restoration_connection": 6500.0,
        }
    )
    damage_repair_share: float = 0.35
    indirect_loss_ratio: float = 0.30
    recovery_budget_share: float = 0.18
    reconstruction_budget_share: float = 0.12


@dataclass
class RecoveryConfig:
    """
    Recovery and reconstruction work rates.

    Attributes:
        road_repair_km_per_day: Output of the road restoration task force.
        bridge_repair_days: Duration of one bridge restoration.
        building_repair_units_per_day: Building repair throughput.
        power_restoration_kw_per_day: Power restoration throughput.
        water_restoration_connections_per_day: Water network throughput.
        communication_sites_per_day: Telecommunication restoration throughput.
    """

    road_repair_km_per_day: float = 4.0
    bridge_repair_days: float = 45.0
    building_repair_units_per_day: float = 25.0
    power_restoration_kw_per_day: float = 250.0
    water_restoration_connections_per_day: float = 120.0
    communication_sites_per_day: float = 3.0


@dataclass
class EngineConfig:
    """
    The complete decision-support configuration.

    Attributes:
        flood / arrival / area / population / agriculture / risk / demand /
        optimization / quantum / response / cost / recovery: The domain
        configuration blocks.
    """

    flood: FloodConfig = field(default_factory=FloodConfig)
    arrival: ArrivalConfig = field(default_factory=ArrivalConfig)
    area: AreaConfig = field(default_factory=AreaConfig)
    population: PopulationConfig = field(default_factory=PopulationConfig)
    agriculture: AgricultureConfig = field(default_factory=AgricultureConfig)
    risk: RiskConfig = field(default_factory=RiskConfig)
    demand: DemandConfig = field(default_factory=DemandConfig)
    optimization: OptimizationConfig = field(default_factory=OptimizationConfig)
    quantum: QuantumConfig = field(default_factory=QuantumConfig)
    response: ResponseConfig = field(default_factory=ResponseConfig)
    cost: CostConfig = field(default_factory=CostConfig)
    recovery: RecoveryConfig = field(default_factory=RecoveryConfig)


CONFIG = EngineConfig()

# Infrastructure classes and their criticality weights, used by the priority and
# risk layers. Higher weight = a facility whose loss hits the response hardest.
INFRASTRUCTURE_CRITICALITY: dict[str, float] = {
    "hospitals": 1.00,
    "emergency_centers": 0.95,
    "fire_stations": 0.80,
    "police_stations": 0.75,
    "water_facilities": 0.80,
    "power_stations": 0.85,
    "communication_towers": 0.70,
    "bridges": 0.65,
    "schools": 0.50,
    "shelters": 0.45,
}

# Land-use split must sum to 1.0; validated by the test suite.
AREA_FRACTION_TOTAL = sum(CONFIG.area.land_use_fractions.values())


def risk_level(score: float, config: RiskConfig | None = None) -> str:
    """
    Map a 0-100 score onto a risk level.

    Args:
        score: The risk score, clamped to [0, 100].
        config: Risk configuration supplying the bands and labels.

    Returns:
        LOW, MEDIUM, HIGH or DANGER.
    """
    cfg = config or CONFIG.risk
    value = min(100.0, max(0.0, float(score)))
    # `bands` holds the upper bound of every band except the last, which is the
    # open-ended remainder: LOW <= 24, MEDIUM <= 49, HIGH <= 74, DANGER above.
    for upper, label in zip(cfg.bands, cfg.labels[:-1], strict=True):
        if value <= upper:
            return label
    return cfg.labels[-1]


def severity_level(score: float) -> str:
    """
    Map a 0-100 flood-severity score onto a level.

    Args:
        score: The severity score, clamped to [0, 100].

    Returns:
        LOW, MODERATE, HIGH or EXTREME.
    """
    value = min(100.0, max(0.0, float(score)))
    for upper, label in zip(
        CONFIG.flood.severity_bands, CONFIG.flood.severity_labels[:-1], strict=True
    ):
        if value <= upper:
            return label
    return CONFIG.flood.severity_labels[-1]


def clamp(value: float, low: float = 0.0, high: float = 100.0) -> float:
    """Clamp a value into [low, high]."""
    return min(high, max(low, float(value)))


def normalise(value: float, reference: float) -> float:
    """
    Scale a non-negative quantity into [0, 1] against a positive reference.

    Args:
        value: The quantity to scale.
        reference: The value that maps to 1.0. Non-positive references give 0.

    Returns:
        The scaled value in [0, 1].
    """
    if reference is None or reference <= 0:
        return 0.0
    return min(1.0, max(0.0, float(value) / float(reference)))


def log_scale(value: float, low: float, high: float) -> float:
    """
    Scale a strictly positive quantity logarithmically into [0, 1].

    Args:
        value: The quantity to scale.
        low: Value that maps to 0.0.
        high: Value that maps to 1.0.

    Returns:
        The scaled value in [0, 1].
    """
    if value is None or value <= 0 or low <= 0 or high <= low:
        return 0.0
    return min(1.0, max(0.0, (math.log10(value) - math.log10(low)) / (math.log10(high) - math.log10(low))))


def describe_configuration() -> None:
    """Print the effective configuration so every ratio is auditable."""
    print("=" * 60)
    print("ENGINE CONFIGURATION (configurable assumptions)")
    print("=" * 60)
    print("")
    print("Flood severity weights      : "
          + ", ".join(f"{k}={v}" for k, v in CONFIG.flood.severity_weights.items()))
    print(f"Excess saturation reference : {CONFIG.flood.excess_reference_ratio} x dam capacity")
    print(f"Rainfall saturation         : {CONFIG.flood.rainfall_reference_mm} mm")
    print(f"Discharge log scale         : {CONFIG.flood.flow_rate_reference_min_m3s:g}"
          f" - {CONFIG.flood.flow_rate_reference_max_m3s:g} m3/s")
    print(f"Assumed flood-front velocity: {CONFIG.arrival.assumed_flow_velocity_kmh} km/h "
          "(only when no velocity is supplied)")
    print(f"Population density fallback : {CONFIG.area.population_density_per_km2:,.0f} people/km2")
    print(f"Max exposure ratio          : {CONFIG.population.max_exposure_ratio}")
    print(f"Risk weights                : "
          + ", ".join(f"{k}={v}" for k, v in CONFIG.risk.weights.items()))
    print(f"Risk bands                  : "
          + " / ".join(
              f"{label} <= {upper:g}"
              for label, upper in zip(CONFIG.risk.labels[:-1], CONFIG.risk.bands, strict=True)
          )
          + f" / {CONFIG.risk.labels[-1]} > {CONFIG.risk.bands[-1]:g}")
    print(f"Relief horizon              : {CONFIG.demand.relief_horizon_days:g} days")
    print(f"Shelter capacity            : {CONFIG.demand.shelter_capacity_persons} persons "
          f"x {CONFIG.demand.shelter_occupancy_ratio:g} occupancy")
    print(f"Key resource / slot         : {CONFIG.optimization.key_resource}, "
          f"{CONFIG.optimization.units_per_deployment_slot} units per deployment slot")
    print(f"QAOA                        : depth={CONFIG.quantum.depth}, "
          f"shots={CONFIG.quantum.shots}, seed={CONFIG.quantum.seed}, "
          f"{CONFIG.quantum.optimizer_method} x {CONFIG.quantum.optimizer_restarts} restarts")
    print(f"Cost currency               : {CONFIG.cost.currency} (planning assumptions, "
          "not official figures)")
    print("")
    print("Configuration status: LOADED")
    print("=" * 60)


describe_configuration()


## SECTION 5 — FLOOD CONDITION ANALYSIS

### 5.1 Unit discipline

A flood volume and a flood discharge are different physical quantities:

| Unit | Kind | Meaning |
| --- | --- | --- |
| `TMC` | volume | 10⁹ m³ of water |
| `TMC/day` | rate | discharge expressed per day |
| `m3/s` | rate | discharge |
| `KCUS` | rate | 1 kc/s = 28.316846592 m³/s |

`TMC` is **never** compared directly with `TMC/day` or `m3/s`. The function
`normalise_flow` performs one explicit, recorded conversion and returns both
representations together with a list of the conversions it performed, which the
report prints verbatim.

A rate is only comparable with a reservoir storage once it has been integrated
over a window, so `inflow_window_hours` is a required field for rate inputs and
its value is reported as the assumption behind every derived volume.

### 5.2 The severity model

The severity score is a transparent weighted sum of five normalized components,
each continuous in the input so that a change in the input always produces a
change in the output:

```
severity = Σ wᵢ · componentᵢ          (weights normalised to 1.0)

component_utilisation = 100 · storage / capacity
component_excess       = 100 · (excess volume / capacity) / excess_reference_ratio
component_rainfall     = 100 · rainfall / rainfall_reference_mm
component_flow_rate    = 100 · log-scale(discharge)
component_trend        = 50 + bonus/penalty from the flow trend
```

`severity_level()` then bands the score with configurable cut points. The
component scores are always printed — the band is never shown without the
numbers behind it.

### 5.3 Water-level status

An absolute full-supply level is not part of the runtime input model, so the
water-level status is derived from storage utilisation and clearly labelled as
such, rather than pretending an absolute threshold was available.


In [ ]:
# =============================================================================
# SECTION 5 - FLOOD CONDITION ANALYSIS
# =============================================================================
#
# Converts the runtime hydrology into a unit-consistent flood picture:
# capacity utilisation, flood space, excess volume, rainfall severity, discharge
# severity, flow trend and an overall flood-severity score.


def normalise_flow(dam: DamInfo) -> tuple[TrackedValue, TrackedValue, list[str]]:
    """
    Convert the incoming flood magnitude into both a volume and a discharge.

    WHAT:
        Produces `incoming_volume_tmc` (a volume, comparable with reservoir
        storage) and `incoming_rate_m3s` (a discharge, comparable with a gauge
        reading), recording every conversion factor applied.

    WHY:
        `TMC`, `TMC/day` and `m3/s` are not interchangeable. Comparing a rate
        with a volume without an integration window is a dimensional error that
        would silently produce a meaningless excess-flow figure.

    HOW:
        * `TMC`     -> volume is the input; the rate is derived over the window.
        * `TMC/day` -> rate = value x 1e9/86400; volume = rate x window.
        * `m3/s`    -> rate is the input; volume = rate x window.
        * `KCUS`    -> rate = value x 28.316846592; volume = rate x window.

    Args:
        dam: The dam / hydrology record.

    Returns:
        `(volume_tmc, rate_m3s, conversions)` where `conversions` is a list of
        human-readable descriptions of the arithmetic performed.

    Raises:
        QFlareValidationError: If the unit is unknown (already caught by
            validation, re-checked here so the function is safe standalone).
    """
    unit = str(dam.incoming_flow_unit)
    window_hours = float(dam.inflow_window_hours or 24.0)
    window_seconds = window_hours * 3600.0
    conversions: list[str] = []
    value = float(dam.incoming_flow)

    if unit == UNIT_TMC:
        volume = value
        rate = value * M3_PER_TMC / window_seconds
        volume_source = DataSource.USER_PROVIDED
        rate_source = DataSource.CALCULATED
        conversions.append(
            f"input {value:g} TMC treated as a VOLUME (already comparable with storage)"
        )
        conversions.append(
            f"rate = {value:g} TMC x 1e9 m3/TMC / ({window_hours:g} h x 3600 s) "
            f"= {rate:,.1f} m3/s"
        )
    elif unit == UNIT_TMC_PER_DAY:
        rate = value * TMC_PER_DAY_TO_M3_PER_S
        volume = rate * window_seconds / M3_PER_TMC
        volume_source = DataSource.CALCULATED
        rate_source = DataSource.CALCULATED
        conversions.append(f"rate = {value:g} TMC/day x 1e9/86400 = {rate:,.1f} m3/s")
        conversions.append(
            f"volume = {rate:,.1f} m3/s x {window_hours:g} h x 3600 s / 1e9 "
            f"= {volume:,.4f} TMC"
        )
    elif unit == UNIT_M3_PER_S:
        rate = value
        volume = value * window_seconds / M3_PER_TMC
        volume_source = DataSource.CALCULATED
        rate_source = DataSource.USER_PROVIDED
        conversions.append(
            f"volume = {value:g} m3/s x {window_hours:g} h x 3600 s / 1e9 "
            f"= {volume:,.4f} TMC"
        )
    elif unit == UNIT_KCUS:
        rate = value * KCUS_TO_M3_PER_S
        volume = rate * window_seconds / M3_PER_TMC
        volume_source = DataSource.CALCULATED
        rate_source = DataSource.CALCULATED
        conversions.append(
            f"rate = {value:g} KCUS x {KCUS_TO_M3_PER_S:.9f} m3/s per KCUS "
            f"= {rate:,.1f} m3/s"
        )
        conversions.append(
            f"volume = {rate:,.1f} m3/s x {window_hours:g} h x 3600 s / 1e9 "
            f"= {volume:,.4f} TMC"
        )
    else:
        raise QFlareValidationError(
            f"unsupported incoming_flow_unit '{unit}'; expected one of "
            f"{', '.join(sorted(ALL_FLOW_UNITS))}"
        )

    volume_assumption = (
        None
        if unit == UNIT_TMC
        else f"incoming flood rate integrated over {window_hours:g} h "
        f"(inflow_window_hours)"
    )
    rate_assumption = (
        f"flood volume distributed uniformly over {window_hours:g} h"
        if unit == UNIT_TMC
        else None
    )
    return (
        TrackedValue(
            value=volume,
            unit=UNIT_TMC,
            source=volume_source,
            confidence=Confidence.HIGH,
            assumption=volume_assumption,
        ),
        TrackedValue(
            value=rate,
            unit=UNIT_M3_PER_S,
            source=rate_source,
            confidence=Confidence.HIGH,
            assumption=rate_assumption,
        ),
        conversions,
    )


def resolve_flow_trend(dam: DamInfo, current_volume_tmc: float) -> tuple[str, str]:
    """
    Resolve the flow trend from the runtime input or derive it.

    Args:
        dam: The dam record.
        current_volume_tmc: The current incoming flood volume in TMC.

    Returns:
        `(trend, basis)` where basis explains how the trend was determined.
    """
    declared = dam.flow_trend
    if declared and str(declared).upper() in {"RISING", "STEADY", "FALLING"}:
        return str(declared).upper(), "declared in the runtime input (flow_trend)"
    if dam.previous_flow is None:
        return "UNKNOWN", "no previous_flow and no flow_trend supplied"
    threshold = CONFIG.flood.trend_fluctuation_threshold_pct / 100.0
    change = (current_volume_tmc - float(dam.previous_flow)) / max(
        abs(float(dam.previous_flow)), 1e-9
    )
    if change > threshold:
        return "RISING", (
            f"derived: {change * 100:+.2f}% change in incoming volume versus "
            f"previous_flow={dam.previous_flow:g} TMC"
        )
    if change < -threshold:
        return "FALLING", (
            f"derived: {change * 100:+.2f}% change in incoming volume versus "
            f"previous_flow={dam.previous_flow:g} TMC"
        )
    return "STEADY", (
        f"derived: {change * 100:+.2f}% change in incoming volume versus "
        f"previous_flow={dam.previous_flow:g} TMC (within "
        f"{CONFIG.flood.trend_fluctuation_threshold_pct:g}% threshold)"
    )


@dataclass
class FloodAnalysis:
    """
    Result of the flood-condition analysis.

    Every numeric attribute is a `TrackedValue` so it carries its unit,
    provenance, confidence and assumption.
    """

    capacity: TrackedValue
    storage: TrackedValue
    incoming_flow_raw: TrackedValue
    incoming_flow_unit: str
    inflow_window_hours: TrackedValue
    incoming_volume: TrackedValue
    incoming_rate: TrackedValue
    flood_space: TrackedValue
    capacity_utilization: TrackedValue
    excess_volume: TrackedValue
    excess_ratio: TrackedValue
    rainfall: TrackedValue
    rainfall_severity_score: TrackedValue
    discharge_severity_score: TrackedValue
    utilization_score: TrackedValue
    excess_score: TrackedValue
    trend_score: TrackedValue
    flow_trend: str
    flow_trend_basis: str
    severity_score: TrackedValue
    severity: str
    water_level_status: str
    conversions: list[str] = field(default_factory=list)
    component_weights: dict[str, float] = field(default_factory=dict)

    @property
    def severity_band(self) -> tuple[str, float]:
        """Severity label with its numeric score."""
        return self.severity, float(self.severity_score.value or 0.0)


def analyze_flood_condition(dam: DamInfo, config: EngineConfig = CONFIG) -> FloodAnalysis:
    """
    Compute the flood condition from the runtime hydrology.

    Args:
        dam: The dam / hydrology record.
        config: Engine configuration.

    Returns:
        A `FloodAnalysis` with every derived quantity traced.
    """
    volume, rate, conversions = normalise_flow(dam)
    capacity = float(dam.dam_capacity)
    storage = float(dam.current_storage)
    current_volume = float(volume.value or 0.0)

    flood_space = max(0.0, capacity - storage)
    utilization_pct = (storage / capacity * 100.0) if capacity > 0 else 0.0
    excess_volume = max(0.0, current_volume - flood_space)
    excess_ratio = (excess_volume / capacity) if capacity > 0 else 0.0

    trend, trend_basis = resolve_flow_trend(dam, current_volume)

    # -- Continuous severity components -------------------------------------
    utilization_score = clamp(utilization_pct)
    excess_score = clamp(
        excess_ratio / config.flood.excess_reference_ratio * 100.0
    )
    rainfall = float(dam.rainfall or 0.0)
    rainfall_score = clamp(rainfall / config.flood.rainfall_reference_mm * 100.0)
    discharge_score = clamp(
        log_scale(
            float(rate.value or 0.0),
            config.flood.flow_rate_reference_min_m3s,
            config.flood.flow_rate_reference_max_m3s,
        )
        * 100.0
    )
    trend_score = clamp(
        50.0
        + (
            config.flood.trend_rising_bonus
            if trend == "RISING"
            else -config.flood.trend_falling_penalty if trend == "FALLING" else 0.0
        )
    )

    weights = dict(config.flood.severity_weights)
    total_weight = sum(weights.values()) or 1.0
    components = {
        "utilization": utilization_score,
        "excess": excess_score,
        "rainfall": rainfall_score,
        "flow_rate": discharge_score,
        "trend": trend_score,
    }
    severity_score = sum(
        weights.get(name, 0.0) / total_weight * value for name, value in components.items()
    )

    water_level_status = "DATA_UNAVAILABLE"
    if dam.water_level is not None:
        water_level_status = severity_level(utilization_pct)

    return FloodAnalysis(
        capacity=user_value(capacity, UNIT_TMC),
        storage=user_value(storage, UNIT_TMC),
        incoming_flow_raw=user_value(dam.incoming_flow, str(dam.incoming_flow_unit)),
        incoming_flow_unit=str(dam.incoming_flow_unit),
        inflow_window_hours=user_value(dam.inflow_window_hours, UNIT_HOURS),
        incoming_volume=volume,
        incoming_rate=rate,
        flood_space=calculated(
            flood_space, UNIT_TMC, "dam_capacity - current_storage"
        ),
        capacity_utilization=calculated(
            utilization_pct, "%", "current_storage / dam_capacity x 100"
        ),
        excess_volume=calculated(
            excess_volume,
            UNIT_TMC,
            "max(0, incoming_volume_tmc - flood_space_tmc)",
        ),
        excess_ratio=calculated(excess_ratio, "ratio", "excess_volume_tmc / dam_capacity_tmc"),
        rainfall=user_value(rainfall, UNIT_MM),
        rainfall_severity_score=calculated(
            rainfall_score, "score", f"rainfall / {config.flood.rainfall_reference_mm:g} mm x 100"
        ),
        discharge_severity_score=calculated(
            discharge_score,
            "score",
            f"log10({rate.display()}) scaled between "
            f"{config.flood.flow_rate_reference_min_m3s:g} and "
            f"{config.flood.flow_rate_reference_max_m3s:g} m3/s",
        ),
        utilization_score=calculated(utilization_score, "score", "storage utilisation"),
        excess_score=calculated(
            excess_score,
            "score",
            f"excess_ratio / {config.flood.excess_reference_ratio:g} x 100",
        ),
        trend_score=calculated(
            trend_score, "score", f"flow trend '{trend}' -> {trend_basis}"
        ),
        flow_trend=trend,
        flow_trend_basis=trend_basis,
        severity_score=calculated(
            severity_score, "score", "weighted sum of the five severity components"
        ),
        severity=severity_level(severity_score),
        water_level_status=water_level_status,
        conversions=conversions,
        component_weights={k: v / total_weight for k, v in weights.items()},
    )


FLOOD = analyze_flood_condition(RUNTIME_EVENT.dam)

print("=" * 60)
print("FLOOD CONDITION ANALYSIS")
print("=" * 60)
print(f"Dam / barrage          : {RUNTIME_EVENT.dam.dam_name}")
print(f"DAM CAPACITY           : {FLOOD.capacity.display()}")
print(f"CURRENT STORAGE        : {FLOOD.storage.display()}")
print(f"INCOMING FLOW          : {FLOOD.incoming_flow_raw.display()} "
      f"(unit declared as '{FLOOD.incoming_flow_unit}')")
print(f"Integration window     : {FLOOD.inflow_window_hours.display()}")
print(f"Incoming volume        : {FLOOD.incoming_volume.display()}")
print(f"Incoming discharge     : {FLOOD.incoming_rate.display()}")
print(f"CAPACITY UTILIZATION   : {FLOOD.capacity_utilization.display()}")
print(f"Available flood space  : {FLOOD.flood_space.display()}")
print(f"EXCESS                 : {FLOOD.excess_volume.display()} "
      f"(ratio {float(FLOOD.excess_ratio.value or 0):.3f} of capacity)")
print(f"RAINFALL               : {FLOOD.rainfall.display()}")
print(f"FLOW TREND             : {FLOOD.flow_trend} ({FLOOD.flow_trend_basis})")
print(f"Water level            : "
      + (
          f"{RUNTIME_EVENT.dam.water_level:,.2f} {UNIT_M}"
          if RUNTIME_EVENT.dam.water_level is not None
          else "DATA_UNAVAILABLE"
      )
      + f" -> status {FLOOD.water_level_status}")
print("")
print("Severity components (all continuous in the runtime input):")
print(f"  utilisation component : {FLOOD.utilization_score.display()} "
      f"(weight {FLOOD.component_weights['utilization']:.3f})")
print(f"  excess component       : {FLOOD.excess_score.display()} "
      f"(weight {FLOOD.component_weights['excess']:.3f})")
print(f"  rainfall component     : {FLOOD.rainfall_severity_score.display()} "
      f"(weight {FLOOD.component_weights['rainfall']:.3f})")
print(f"  discharge component    : {FLOOD.discharge_severity_score.display()} "
      f"(weight {FLOOD.component_weights['flow_rate']:.3f})")
print(f"  trend component        : {FLOOD.trend_score.display()} "
      f"(weight {FLOOD.component_weights['trend']:.3f})")
print("")
print(f"FLOOD CONDITION        : {FLOOD.severity} "
      f"(severity score {float(FLOOD.severity_score.value or 0):.2f} / 100)")
print("")
print("Unit conversions actually performed:")
for line in FLOOD.conversions:
    print(f"  - {line}")
print("")
print("Flood analysis status: COMPLETED")
print("=" * 60)


## SECTION 6 — FLOOD ARRIVAL-TIME ESTIMATION

The arrival time answers one operational question: **how long until the flood
front reaches the target, and therefore how much time is left to act.**

Two routes are supported, and the report always says which one was used:

1. **Observed** — the runtime input supplies a sensor-derived or historical
   timing. Confidence `HIGH`, source `SENSOR`.
2. **Calculated** — a kinematic wave-front estimate
   `travel_time = distance / velocity`, where the velocity is either supplied or
   taken from `ArrivalConfig.assumed_flow_velocity_kmh`. Source
   `MODEL_ESTIMATED`, and the assumption is printed.

If neither a distance nor a timing is available the result is
`INSUFFICIENT_DATA` and the value is `None`. The engine does **not** invent a
plausible-looking hour count.

### 6.1 Time status

| Status | Condition |
| --- | --- |
| `INSUFFICIENT_DATA` | no arrival estimate could be produced |
| `CRITICAL` | available response time < required mobilisation time |
| `AT_RISK` | available response time < margin × required mobilisation time |
| `SAFE` | otherwise |

`Required mobilisation time` is the sum of the dispatch preparation and
evacuation lead times, both configurable. The evacuation deadline is the arrival
time minus the lead time.


In [ ]:
# =============================================================================
# SECTION 6 - FLOOD ARRIVAL-TIME ESTIMATION
# =============================================================================
#
# Estimates the time-to-impact of the flood front and the resulting operational
# window. Returns INSUFFICIENT_DATA rather than fabricating a precise time.

INSUFFICIENT_DATA = "INSUFFICIENT_DATA"
STATUS_SAFE = "SAFE"
STATUS_AT_RISK = "AT_RISK"
STATUS_CRITICAL = "CRITICAL"


@dataclass
class ArrivalAnalysis:
    """
    Flood arrival-time result.

    Attributes:
        source_label: "FLOOD SOURCE" description of where the flood comes from.
        target_label: Description of the impacted target.
        distance: Upstream distance, when known.
        velocity: Wave-front velocity used, when known.
        observed_timing: Sensor-derived timing, when supplied.
        arrival_hours: Estimated time-to-impact in hours, or None.
        arrival_source: DataSource of `arrival_hours`.
        available_response_time_hours: Time remaining before impact.
        required_response_time_hours: Minimum mobilisation time needed.
        evacuation_deadline_hours: Time by which evacuation must start.
        time_status: SAFE / AT_RISK / CRITICAL / INSUFFICIENT_DATA.
        method: How the arrival time was obtained.
        assumptions: Every assumption behind the estimate.
    """

    source_label: str
    target_label: str
    distance: TrackedValue
    velocity: TrackedValue
    observed_timing: TrackedValue
    arrival_hours: TrackedValue
    available_response_time_hours: TrackedValue
    required_response_time_hours: TrackedValue
    evacuation_deadline_hours: TrackedValue
    time_status: str
    method: str
    assumptions: list[str] = field(default_factory=list)


def estimate_flood_arrival(
    geography: GeoInfo,
    flood: FloodAnalysis,
    config: EngineConfig = CONFIG,
) -> ArrivalAnalysis:
    """
    Estimate the time-to-impact of the flood front.

    Priority of evidence:
        1. A sensor-derived or historical timing supplied with the runtime input.
        2. `travel_time = distance / velocity`, using a supplied velocity or the
           configured fallback.
        3. `INSUFFICIENT_DATA`.

    Args:
        geography: Geographic record (upstream distance).
        flood: The flood analysis, used for the trend and the source label.
        config: Engine configuration.

    Returns:
        An `ArrivalAnalysis`; `arrival_hours.value` is None when insufficient
        data is available.
    """
    source_label = f"upstream flood front of {flood.capacity.display()} reservoir inflow"
    target_label = f"response area ({geography.latitude}, {geography.longitude})"

    distance_value = geography.upstream_distance
    distance = (
        user_value(distance_value, UNIT_KM)
        if distance_value is not None
        else unavailable(UNIT_KM, "geography.upstream_distance not supplied")
    )
    velocity = unavailable(UNIT_KM + "/" + UNIT_HOURS, "no velocity required")
    observed = unavailable(UNIT_HOURS, "no sensor timing supplied")
    assumptions: list[str] = []

    arrival_value: float | None = None
    arrival_source = DataSource.DATA_UNAVAILABLE
    confidence = Confidence.UNAVAILABLE
    method = "no arrival estimate could be produced"

    if distance_value is not None and float(distance_value) > 0:
        velocity_kmh = config.arrival.assumed_flow_velocity_kmh
        travel_hours = float(distance_value) / velocity_kmh
        arrival_value = travel_hours
        arrival_source = DataSource.MODEL_ESTIMATED
        confidence = Confidence.MEDIUM
        velocity = estimated(
            velocity_kmh,
            UNIT_KM + "/" + UNIT_HOURS,
            f"ArrivalConfig.assumed_flow_velocity_kmh = {velocity_kmh:g} km/h "
            "(no velocity supplied with the runtime input)",
            Confidence.LOW,
        )
        assumptions.append(
            f"kinematic wave front: travel_time = {float(distance_value):g} km / "
            f"{velocity_kmh:g} km/h = {travel_hours:.2f} h"
        )
        assumptions.append(
            "a real flood front is not a constant-velocity kinematic wave; "
            "channel storage, meander and tributary inflow shorten or lengthen it"
        )
        method = "calculated from distance and configured flood-front velocity"
    elif distance_value is not None and float(distance_value) == 0:
        arrival_value = 0.0
        arrival_source = DataSource.CALCULATED
        confidence = Confidence.HIGH
        method = "target coincides with the flood source (upstream distance is 0 km)"

    required_hours = (
        config.response.dispatch_preparation_hours
        + config.arrival.evacuation_lead_hours
    )
    required = calculated(
        required_hours,
        UNIT_HOURS,
        f"ResponseConfig.dispatch_preparation_hours + "
        f"ArrivalConfig.evacuation_lead_hours = "
        f"{config.response.dispatch_preparation_hours:g} + "
        f"{config.arrival.evacuation_lead_hours:g}",
    )

    if arrival_value is None:
        available = unavailable(UNIT_HOURS, "arrival time unavailable")
        deadline = unavailable(UNIT_HOURS, "arrival time unavailable")
        time_status = INSUFFICIENT_DATA
    else:
        available = calculated(
            max(0.0, arrival_value), UNIT_HOURS, "time remaining until flood impact"
        )
        # The deadline is reported unclamped: a negative value is the operationally
        # meaningful answer, because it means the moment to start has already passed.
        deadline = calculated(
            arrival_value - config.arrival.evacuation_lead_hours,
            UNIT_HOURS,
            f"arrival time minus ArrivalConfig.evacuation_lead_hours "
            f"({config.arrival.evacuation_lead_hours:g} h); a negative value means "
            "the evacuation start deadline has already passed",
        )
        if arrival_value < required_hours:
            time_status = STATUS_CRITICAL
        elif arrival_value < required_hours * config.arrival.at_risk_margin:
            time_status = STATUS_AT_RISK
        else:
            time_status = STATUS_SAFE

    if flood.flow_trend == "RISING" and arrival_value is not None:
        assumptions.append(
            "flow trend is RISING, so the actual arrival may be earlier than the "
            "estimate computed from the present discharge"
        )

    return ArrivalAnalysis(
        source_label=source_label,
        target_label=target_label,
        distance=distance,
        velocity=velocity,
        observed_timing=observed,
        arrival_hours=TrackedValue(
            value=arrival_value,
            unit=UNIT_HOURS,
            source=arrival_source,
            confidence=confidence,
            assumption="; ".join(assumptions) if assumptions else None,
        ),
        available_response_time_hours=available,
        required_response_time_hours=required,
        evacuation_deadline_hours=deadline,
        time_status=time_status,
        method=method,
        assumptions=assumptions,
    )


ARRIVAL = estimate_flood_arrival(RUNTIME_EVENT.geography, FLOOD)

print("=" * 60)
print("FLOOD ARRIVAL TIME")
print("=" * 60)
print(f"Source                     : {ARRIVAL.source_label}")
print(f"Target                     : {ARRIVAL.target_label}")
print(f"Distance                   : {ARRIVAL.distance.display()}")
print(f"Estimated velocity         : {ARRIVAL.velocity.display()}")
print(f"Observed / sensor timing   : {ARRIVAL.observed_timing.display()}")
print(f"Method                     : {ARRIVAL.method}")
print(f"Estimated arrival          : {ARRIVAL.arrival_hours.display()}")
print(f"  source                   : {ARRIVAL.arrival_hours.source.value}")
print(f"  confidence               : {ARRIVAL.arrival_hours.confidence.value}")
print(f"Available response time    : {ARRIVAL.available_response_time_hours.display()}")
print(f"Required response time     : {ARRIVAL.required_response_time_hours.display()}")
print(f"Evacuation deadline        : {ARRIVAL.evacuation_deadline_hours.display()}")
print("")
if ARRIVAL.assumptions:
    print("Assumptions behind the estimate:")
    for assumption in ARRIVAL.assumptions:
        print(f"  - {assumption}")
else:
    print("Assumptions: none recorded")
print("")
print(f"Time status                : {ARRIVAL.time_status}")
print("Arrival estimation status: COMPLETED")
print("=" * 60)


## SECTION 7 — AFFECTED AREA ANALYSIS

No raster DEM, hydraulic model or cadastral layer is available inside a Colab
notebook, so this section does **not** perform a GIS inundation analysis and does
not pretend to. It reports:

* the affected area, either taken from the runtime input or derived from the
  supplied population and a configured density;
* the land-use split, from configurable fractions of that area;
* a volumetric inundation-depth estimate,
  `depth = excess_volume / affected_area`, with the freeboard of built-up floors
  removed.

The verdict is stated explicitly:

| GIS status | Condition |
| --- | --- |
| `AVAILABLE` | regional coordinates **and** every location has coordinates |
| `LIMITED` | only some coordinates are available, or the area is derived |
| `UNAVAILABLE` | no coordinates at all |

`GISLayerPlaceholder` is the extension point: it is a typed, empty container that
a real GIS service can populate later without touching any downstream code.


In [ ]:
# =============================================================================
# SECTION 7 - AFFECTED AREA ANALYSIS
# =============================================================================
#
# Reports the affected area, the land-use split, a volumetric inundation-depth
# estimate and the honest GIS availability status.

GIS_AVAILABLE = "AVAILABLE"
GIS_LIMITED = "LIMITED"
GIS_UNAVAILABLE = "UNAVAILABLE"


@dataclass
class GISLayerPlaceholder:
    """
    Typed hand-off point for a real GIS / hydraulic-modelling service.

    Every field is intentionally empty. A downstream service fills them in; no
    stage of this notebook reads a value that is not present, so an unfilled
    layer degrades the report to `DATA_UNAVAILABLE` instead of to a guess.

    Attributes:
        dem_path: Path to a digital elevation model.
        inundation_raster: Path to a modelled inundation raster.
        admin_boundaries: Path to administrative boundary geometry.
        road_network: Path to the road centrelines.
        catchment_polygon: Path to the catchment boundary.
    """

    dem_path: str | None = None
    inundation_raster: str | None = None
    admin_boundaries: str | None = None
    road_network: str | None = None
    catchment_polygon: str | None = None

    @property
    def is_populated(self) -> bool:
        """True when a real GIS service has supplied at least one layer."""
        return any(
            value is not None
            for value in (
                self.dem_path,
                self.inundation_raster,
                self.admin_boundaries,
                self.road_network,
                self.catchment_polygon,
            )
        )


GIS_LAYERS = GISLayerPlaceholder()


@dataclass
class AreaAnalysis:
    """
    Affected-area result.

    Attributes:
        total_area: Affected area in km2.
        urban / rural / agricultural / residential: Land-use areas in km2.
        inundation_depth: Estimated inundation depth in m.
        freeboard: Freeboard subtracted from the raw depth.
        affected_area_fraction: Share of the reported area actually inundated.
        gis_status: AVAILABLE / LIMITED / UNAVAILABLE.
        gis_reason: Why that status was chosen.
        methods: How each figure was obtained.
    """

    total_area: TrackedValue
    urban: TrackedValue
    rural: TrackedValue
    agricultural: TrackedValue
    residential: TrackedValue
    inundation_depth: TrackedValue
    freeboard: TrackedValue
    affected_area_fraction: TrackedValue
    gis_status: str
    gis_reason: str
    warnings: list[str] = field(default_factory=list)
    methods: dict[str, str] = field(default_factory=dict)


def analyze_affected_area(
    geography: GeoInfo,
    population: PopulationInfo,
    economics: EconomicInfo,
    flood: FloodAnalysis,
    locations: Sequence[LocationInput] = (),
    config: EngineConfig = CONFIG,
) -> AreaAnalysis:
    """
    Estimate the affected area, its land-use split and the inundation depth.

    Args:
        geography: Geographic record.
        population: Population record.
        economics: Economic record (agricultural / crop areas).
        flood: Flood analysis (excess volume).
        locations: Response locations, used for the GIS availability check.
        config: Engine configuration.

    Returns:
        An `AreaAnalysis`.
    """
    methods: dict[str, str] = {}

    if geography.affected_area is not None and float(geography.affected_area) > 0:
        total_area = float(geography.affected_area)
        area_value = user_value(total_area, UNIT_KM2)
        methods["total_area"] = "supplied in the runtime input (geography.affected_area)"
    elif population.total_population:
        total_area = float(population.total_population) / config.area.population_density_per_km2
        area_value = estimated(
            total_area,
            UNIT_KM2,
            f"total_population / AreaConfig.population_density_per_km2 "
            f"({config.area.population_density_per_km2:g} people/km2)",
            Confidence.LOW,
        )
        methods["total_area"] = (
            f"DERIVED from population at a configured density of "
            f"{config.area.population_density_per_km2:g} people/km2 - low confidence"
        )
    else:
        total_area = 0.0
        area_value = unavailable(
            UNIT_KM2, "neither affected_area nor a population figure was supplied"
        )
        methods["total_area"] = "no area and no population -> DATA_UNAVAILABLE"

    land_use: dict[str, TrackedValue] = {}
    for name, fraction in config.area.land_use_fractions.items():
        land_use[name] = calculated(
            total_area * fraction,
            UNIT_KM2,
            f"total_area x AreaConfig.land_use_fractions['{name}'] = {fraction:g}",
        )
    methods["land_use"] = (
        "AreaConfig.land_use_fractions (a planning split, NOT a surveyed land-use map)"
    )

    excess_m3 = float(flood.excess_volume.value or 0.0) * M3_PER_TMC
    warnings: list[str] = []
    if total_area > 0 and excess_m3 > 0:
        severity_fraction = float(flood.severity_score.value or 0.0) / 100.0
        # Two independent readings of the same volume, because either one alone
        # can mislead: a depth is only meaningful at a plausible magnitude, and
        # an area is only meaningful at a plausible depth.
        volumetric_depth = excess_m3 / (total_area * 1.0e6)
        characteristic_depth = (
            severity_fraction * config.area.max_plausible_inundation_depth_m
        )
        implied_area_km2 = (
            excess_m3 / characteristic_depth / 1.0e6 if characteristic_depth > 0 else 0.0
        )
        if volumetric_depth > config.area.max_plausible_inundation_depth_m:
            inundation = calculated(
                config.area.max_plausible_inundation_depth_m,
                UNIT_M,
                f"volumetric average would be {volumetric_depth:,.1f} m, which is "
                f"not physically plausible; reported depth is capped at "
                f"AreaConfig.max_plausible_inundation_depth_m "
                f"({config.area.max_plausible_inundation_depth_m:g} m) and the "
                f"capped value means the true depth exceeds it",
            )
            warnings.append(
                f"INUNDATION DEPTH SATURATED: {excess_m3 / M3_PER_TMC:,.1f} TMC of "
                f"excess volume over {total_area:,.1f} km2 would need a depth of "
                f"{volumetric_depth:,.1f} m. The reported depth is capped at "
                f"{config.area.max_plausible_inundation_depth_m:g} m, so the true "
                "inundation is deeper than that and the excess volume is not a "
                "local inundation volume for the reported area."
            )
        else:
            inundation = calculated(
                max(0.0, volumetric_depth - config.area.freeboard_depth_m),
                UNIT_M,
                f"excess volume {excess_m3:,.0f} m3 / area {total_area:,.1f} km2 "
                f"= {volumetric_depth:.2f} m, minus "
                f"{config.area.freeboard_depth_m:g} m freeboard",
            )
        methods["inundation_depth"] = (
            f"volumetric average (excess volume / reported area), capped at "
            f"AreaConfig.max_plausible_inundation_depth_m; this is an average, not "
            "a spatially varying depth from a hydraulic model"
        )

        if implied_area_km2 > total_area * 1.05:
            warnings.append(
                f"EXCESS VOLUME EXCEEDS REPORTED AREA: at a plausible depth of "
                f"{characteristic_depth:.2f} m the {excess_m3 / M3_PER_TMC:,.1f} TMC "
                f"excess volume would cover {implied_area_km2:,.0f} km2, far more "
                f"than the reported {total_area:,.1f} km2. Either the affected area "
                "is under-reported or the excess volume is a basin routing volume "
                "rather than a local inundation volume."
            )
        affected_fraction = min(1.0, implied_area_km2 / total_area) if total_area else 0.0
        methods["affected_area_fraction"] = (
            f"implied inundated area {implied_area_km2:,.0f} km2 (excess volume / "
            f"characteristic depth) / reported area {total_area:,.1f} km2, capped at 1.0"
        )
    else:
        inundation = unavailable(
            UNIT_M,
            "no excess volume or no affected area, so no depth can be derived",
        )
        affected_fraction = 0.0
        methods["inundation_depth"] = "no excess volume / no area -> DATA_UNAVAILABLE"

    located = sum(
        1
        for location in locations
        if location.latitude is not None and location.longitude is not None
    )
    if (geography.latitude is None or geography.longitude is None) and located == 0:
        gis_status = GIS_UNAVAILABLE
        gis_reason = "no coordinates supplied for the region or for any location"
    elif located < len(locations) or geography.affected_area is None:
        gis_status = GIS_LIMITED
        gis_reason = (
            f"{located} of {len(locations)} locations carry coordinates; "
            "the affected area is not a surveyed layer"
        )
    else:
        gis_status = GIS_AVAILABLE
        gis_reason = "coordinates supplied for the region and for every location"

    return AreaAnalysis(
        total_area=area_value,
        urban=land_use["urban"],
        rural=land_use["rural"],
        agricultural=land_use["agricultural"],
        residential=land_use["residential"],
        inundation_depth=inundation,
        freeboard=user_value(config.area.freeboard_depth_m, UNIT_M),
        affected_area_fraction=calculated(
            affected_fraction * 100.0,
            "%",
            "inundated share of the reported area, from the volumetric depth",
        ),
        gis_status=gis_status,
        gis_reason=gis_reason,
        warnings=warnings,
        methods=methods,
    )


AREA = analyze_affected_area(
    RUNTIME_EVENT.geography,
    RUNTIME_EVENT.population,
    RUNTIME_EVENT.economics,
    FLOOD,
    RUNTIME_EVENT.locations,
)

print("=" * 60)
print("AFFECTED AREA ANALYSIS")
print("=" * 60)
print(f"Total affected area   : {AREA.total_area.display()}")
print(f"Urban area            : {AREA.urban.display()}")
print(f"Rural area            : {AREA.rural.display()}")
print(f"Agricultural area     : {AREA.agricultural.display()}")
print(f"Residential area      : {AREA.residential.display()}")
print(f"Inundation depth      : {AREA.inundation_depth.display()}")
print(f"Freeboard applied     : {AREA.freeboard.display()}")
print(f"Affected area fraction: {AREA.affected_area_fraction.display()}")
print(f"GIS STATUS            : {AREA.gis_status} ({AREA.gis_reason})")
print(f"GIS layers populated  : {GIS_LAYERS.is_populated} (placeholder container)")
print("")
print("How each figure was obtained:")
for name, method in AREA.methods.items():
    print(f"  - {name}: {method}")
print("")
if AREA.warnings:
    print("Consistency warnings:")
    for warning in AREA.warnings:
        print(f"  - {warning}")
else:
    print("Consistency warnings: none")
print("")
print(f"Inundation depth detail: {AREA.inundation_depth.assumption}")
print("Affected area status: COMPLETED")
print("=" * 60)


## SECTION 8 — POPULATION IMPACT

Exposure is a **continuous function of flood severity**, not a lookup:

```
exposed_population = total_population x max_exposure_ratio x (severity / 100)
```

so a change in the incoming flow changes the severity, which changes the exposed
population, which changes every demand figure downstream. At severity 0 nothing
is exposed; at severity 100 up to `PopulationConfig.max_exposure_ratio` of the
population is exposed.

`evacuation_population` adds the people who must be moved regardless of
inundation — the vulnerable, the children, the elderly and anyone who cannot
self-evacuate — and is capped at the total population.

If the runtime input carries no population at all, the section prints
`DATA UNAVAILABLE`. It never fabricates a headcount.


In [ ]:
# =============================================================================
# SECTION 8 - POPULATION IMPACT
# =============================================================================
#
# Derives exposed, evacuated, high-risk and vulnerable populations from the
# flood severity. Reports DATA_UNAVAILABLE when no population data exists.

DATA_UNAVAILABLE_LABEL = "DATA UNAVAILABLE"


@dataclass
class PopulationAnalysis:
    """
    Population-impact result.

    Attributes:
        total_population: Regional population, or None.
        exposed: Population inside the inundated footprint.
        evacuation: Population that must be moved.
        high_risk: Population in the worst-affected share of the area.
        vulnerable: Vulnerable population among the exposed.
        children / elderly / requiring_assistance: Special groups.
        risk_level: LOW / MEDIUM / HIGH / DANGER.
        available: False when no population data was supplied.
    """

    total_population: TrackedValue
    exposed: TrackedValue
    evacuation: TrackedValue
    high_risk: TrackedValue
    vulnerable: TrackedValue
    children: TrackedValue
    elderly: TrackedValue
    requiring_assistance: TrackedValue
    risk_level: str
    exposure_ratio: TrackedValue
    available: bool


def analyze_population_impact(
    population: PopulationInfo,
    flood: FloodAnalysis,
    config: EngineConfig = CONFIG,
) -> PopulationAnalysis:
    """
    Derive the population impact of the flood.

    Args:
        population: The runtime population record.
        flood: The flood analysis (supplies the severity score).
        config: Engine configuration.

    Returns:
        A `PopulationAnalysis`; `available` is False when no population was
        supplied, in which case every quantity is reported as unavailable.
    """
    if not population.total_population or population.total_population <= 0:
        placeholder = unavailable(UNIT_PEOPLE, "no population supplied")
        return PopulationAnalysis(
            total_population=placeholder,
            exposed=placeholder,
            evacuation=placeholder,
            high_risk=placeholder,
            vulnerable=placeholder,
            children=placeholder,
            elderly=placeholder,
            requiring_assistance=placeholder,
            risk_level=DATA_UNAVAILABLE_LABEL,
            exposure_ratio=placeholder,
            available=False,
        )

    total = float(population.total_population)
    severity = float(flood.severity_score.value or 0.0) / 100.0
    exposure_ratio = min(config.population.max_exposure_ratio, severity * config.population.max_exposure_ratio)
    exposed = float(round(total * exposure_ratio))

    vulnerable = min(
        total, float(population.vulnerable_population or 0) * config.population.vulnerable_multiplier
    )
    children = min(total, float(population.children or 0))
    elderly = min(total, float(population.elderly or 0))
    assistance = min(total, float(population.persons_requiring_assistance or 0))

    # Evacuation load is whichever is larger: everyone inside the inundated
    # footprint, or the special groups that must be moved regardless of depth.
    evacuation = min(total, max(exposed, vulnerable, assistance))
    high_risk = float(round(exposed * config.population.high_risk_fraction))

    population_risk_score = clamp(
        100.0
        * (
            0.55 * exposure_ratio / max(config.population.max_exposure_ratio, 1e-9)
            + 0.25 * (vulnerable / total if total else 0.0)
            + 0.20 * (assistance / total if total else 0.0)
        )
    )

    assumption = (
        f"exposure = total_population x PopulationConfig.max_exposure_ratio "
        f"({config.population.max_exposure_ratio:g}) x severity/100 "
        f"({severity:.4f})"
    )
    return PopulationAnalysis(
        total_population=user_value(int(total), UNIT_PEOPLE),
        exposed=calculated(int(exposed), UNIT_PEOPLE, assumption),
        evacuation=calculated(
            int(evacuation),
            UNIT_PEOPLE,
            f"max(exposed, vulnerable, assistance-requiring) = {evacuation:,.0f}",
        ),
        high_risk=calculated(
            int(high_risk),
            UNIT_PEOPLE,
            f"exposed x PopulationConfig.high_risk_fraction "
            f"({config.population.high_risk_fraction:g})",
        ),
        vulnerable=calculated(
            vulnerable,
            UNIT_PEOPLE,
            "min(total, vulnerable_population x PopulationConfig.vulnerable_multiplier)",
        ),
        children=user_value(int(children), UNIT_PEOPLE)
        if population.children is not None
        else unavailable(UNIT_PEOPLE, "children not supplied"),
        elderly=user_value(int(elderly), UNIT_PEOPLE)
        if population.elderly is not None
        else unavailable(UNIT_PEOPLE, "elderly not supplied"),
        requiring_assistance=user_value(int(assistance), UNIT_PEOPLE)
        if population.persons_requiring_assistance is not None
        else unavailable(UNIT_PEOPLE, "persons_requiring_assistance not supplied"),
        risk_level=risk_level(population_risk_score),
        exposure_ratio=calculated(exposure_ratio, "ratio", assumption),
        available=True,
    )


POPULATION = analyze_population_impact(RUNTIME_EVENT.population, FLOOD)

print("=" * 60)
print("POPULATION IMPACT")
print("=" * 60)
if POPULATION.available:
    print(f"Total population exposed : {POPULATION.exposed.display()}")
    print(f"Potentially evacuated    : {POPULATION.evacuation.display()}")
    print(f"High-risk population     : {POPULATION.high_risk.display()}")
    print(f"Vulnerable population    : {POPULATION.vulnerable.display()}")
    print(f"Children                 : {POPULATION.children.display()}")
    print(f"Elderly                  : {POPULATION.elderly.display()}")
    print(f"Requiring assistance     : {POPULATION.requiring_assistance.display()}")
    print(f"Exposure ratio           : {POPULATION.exposure_ratio.display()} "
          f"of {POPULATION.total_population.display()}")
    print(f"Population risk          : {POPULATION.risk_level}")
    print(f"Assumption               : {POPULATION.exposed.assumption}")
else:
    print("Total population exposed : DATA UNAVAILABLE")
    print("Potentially evacuated    : DATA UNAVAILABLE")
    print("High-risk population     : DATA UNAVAILABLE")
    print("Vulnerable population    : DATA UNAVAILABLE")
    print("Population risk          : DATA UNAVAILABLE")
    print("Reason                   : no population data was supplied with the runtime input")
print("")
print("Population impact status: COMPLETED")
print("=" * 60)


## SECTION 9 — ROAD AND TRANSPORT IMPACT

Road impact is computed from the supplied inventory wherever it exists, and
completed with a **severity-driven blockage estimate** only for the fields the
runtime input did not provide. The two are never mixed silently: an estimated
field is marked `MODEL_ESTIMATED` and names the ratio that produced it.

* `blocked_ratio = severity / 100 x RoadConfig.max_blocked_ratio`
* `accessibility_score` drives the road component of the risk engine and the
  accessibility term of the response-priority ranking.

When no road inventory exists at all, the section returns a
`DATA_UNAVAILABLE` warning and the downstream road risk is scored from the flood
severity alone, which the report states explicitly.


In [ ]:
# =============================================================================
# SECTION 9 - ROAD AND TRANSPORT IMPACT
# =============================================================================
#
# Combines the supplied road inventory with a severity-driven estimate for the
# fields the runtime input did not provide, and scores road accessibility.

# Maximum share of the road network that can be impassable at severity 100.
MAX_BLOCKED_ROAD_RATIO = 0.60
# Maximum share of the road network that can be damaged (not impassable).
MAX_DAMAGED_ROAD_RATIO = 0.35


@dataclass
class RoadAnalysis:
    """
    Road and transport impact.

    Attributes:
        total_roads: Total road length in km.
        blocked_roads: Impassable road length in km.
        damaged_roads: Damaged but passable road length in km.
        critical_roads: Life-line road length in km.
        critical_roads_blocked: Life-line length that is impassable.
        bridges / bridges_affected: Bridge counts.
        alternative_routes: Count of usable alternative routes.
        blocked_ratio: Blocked share of the network, 0-1.
        accessibility_score: 0-100; 100 means fully accessible.
        transportation_risk: LOW / MEDIUM / HIGH / DANGER.
        warnings: Data-availability warnings.
        available: False when no road inventory was supplied.
    """

    total_roads: TrackedValue
    blocked_roads: TrackedValue
    damaged_roads: TrackedValue
    critical_roads: TrackedValue
    critical_roads_blocked: TrackedValue
    bridges: TrackedValue
    bridges_affected: TrackedValue
    alternative_routes: TrackedValue
    blocked_ratio: TrackedValue
    accessibility_score: TrackedValue
    transportation_risk: str
    warnings: list[str] = field(default_factory=list)
    available: bool = False


def analyze_road_impact(
    infrastructure: InfrastructureInfo,
    flood: FloodAnalysis,
    config: EngineConfig = CONFIG,
) -> RoadAnalysis:
    """
    Compute the road and transport impact.

    Args:
        infrastructure: The runtime infrastructure inventory.
        flood: The flood analysis (supplies severity).
        config: Engine configuration.

    Returns:
        A `RoadAnalysis`.
    """
    warnings: list[str] = []
    severity_fraction = float(flood.severity_score.value or 0.0) / 100.0
    inventory_present = infrastructure.roads is not None

    if not inventory_present:
        warnings.append(
            "no road inventory supplied - road risk is derived from flood severity alone"
        )
        total_km = 0.0
        total = unavailable(UNIT_KM, "infrastructure.roads not supplied")
        blocked = unavailable(UNIT_KM, "no road network to block")
        damaged = unavailable(UNIT_KM, "no road network to damage")
        blocked_ratio_value = severity_fraction * MAX_BLOCKED_ROAD_RATIO
        blocked_ratio = estimated(
            blocked_ratio_value,
            "ratio",
            f"severity/100 x MAX_BLOCKED_ROAD_RATIO ({MAX_BLOCKED_ROAD_RATIO:g}) - "
            "assumed network length because none was supplied",
            Confidence.LOW,
        )
    else:
        total_km = float(infrastructure.roads)
        total = user_value(total_km, UNIT_KM)
        blocked_ratio_value = severity_fraction * MAX_BLOCKED_ROAD_RATIO
        if infrastructure.blocked_roads is not None:
            blocked_km = float(infrastructure.blocked_roads)
            blocked = user_value(blocked_km, UNIT_KM)
            blocked_ratio_value = (blocked_km / total_km) if total_km > 0 else 0.0
            blocked_ratio = calculated(
                blocked_ratio_value, "ratio", "blocked_roads / roads"
            )
        else:
            blocked_km = total_km * blocked_ratio_value
            blocked = estimated(
                blocked_km,
                UNIT_KM,
                f"roads x severity/100 x MAX_BLOCKED_ROAD_RATIO "
                f"({MAX_BLOCKED_ROAD_RATIO:g})",
            )
            warnings.append("blocked_roads not supplied - estimated from flood severity")

        if infrastructure.damaged_roads is not None:
            damaged = user_value(float(infrastructure.damaged_roads), UNIT_KM)
        else:
            damaged = estimated(
                total_km * severity_fraction * MAX_DAMAGED_ROAD_RATIO,
                UNIT_KM,
                f"roads x severity/100 x MAX_DAMAGED_ROAD_RATIO "
                f"({MAX_DAMAGED_ROAD_RATIO:g})",
            )
            warnings.append("damaged_roads not supplied - estimated from flood severity")

    if infrastructure.critical_roads is not None:
        critical_km = float(infrastructure.critical_roads)
        critical = user_value(critical_km, UNIT_KM)
    else:
        critical_km = total_km * 0.30
        critical = estimated(
            critical_km,
            UNIT_KM,
            "roads x 0.30 - assumed life-line share when not supplied",
            Confidence.LOW,
        )
        warnings.append("critical_roads not supplied - assumed 30% of the network")

    critical_blocked = critical_km * blocked_ratio_value
    critical_blocked_value = calculated(
        critical_blocked,
        UNIT_KM,
        "critical_roads x blocked_ratio",
    )

    if infrastructure.bridges is not None:
        bridges = user_value(int(infrastructure.bridges), "bridges")
    else:
        bridges = unavailable("bridges", "infrastructure.bridges not supplied")
    if infrastructure.bridges_affected is not None:
        bridges_affected = user_value(int(infrastructure.bridges_affected), "bridges")
        bridge_impact = estimated(
            float(infrastructure.bridges_affected)
            / float(infrastructure.bridges or 1)
            * severity_fraction,
            "ratio",
            "affected bridges / total bridges x severity/100",
        )
    else:
        bridges_affected = unavailable("bridges", "infrastructure.bridges_affected not supplied")
        bridge_impact = estimated(
            severity_fraction,
            "ratio",
            "severity/100 used as the bridge-impact ratio when no count was supplied",
            Confidence.LOW,
        )
        warnings.append("bridge impact not supplied - estimated from flood severity")

    if infrastructure.alternative_routes is not None:
        alternative_routes = user_value(int(infrastructure.alternative_routes), "routes")
    else:
        alternative_routes = unavailable("routes", "infrastructure.alternative_routes not supplied")
        warnings.append("no alternative-route inventory supplied")

    # Accessibility blends the network blockage with the number of alternatives:
    # a blocked network with alternatives is better than one without, but the
    # alternatives can restore at most a quarter of the lost accessibility.
    alternatives = (
        float(infrastructure.alternative_routes)
        if infrastructure.alternative_routes is not None
        else 0.0
    )
    alternative_relief = min(1.0, alternatives / 3.0) if alternatives else 0.0
    accessibility = clamp(
        100.0 * (1.0 - blocked_ratio_value) * (0.75 + 0.25 * alternative_relief)
    )
    accessibility_value = calculated(
        accessibility,
        "score",
        f"100 x (1 - blocked_ratio) x (0.75 + 0.25 x min(1, alternative_routes/3)); "
        f"blocked_ratio={blocked_ratio_value:.4f}, alternative_relief={alternative_relief:.4f}",
    )

    return RoadAnalysis(
        total_roads=total,
        blocked_roads=blocked,
        damaged_roads=damaged,
        critical_roads=critical,
        critical_roads_blocked=critical_blocked_value,
        bridges=bridges,
        bridges_affected=bridges_affected,
        alternative_routes=alternative_routes,
        blocked_ratio=blocked_ratio,
        accessibility_score=accessibility_value,
        transportation_risk=risk_level(100.0 - accessibility),
        warnings=warnings,
        available=inventory_present,
    )


ROADS = analyze_road_impact(RUNTIME_EVENT.infrastructure, FLOOD)

print("=" * 60)
print("ROAD AND TRANSPORT IMPACT")
print("=" * 60)
print(f"Total roads analyzed    : {ROADS.total_roads.display()}")
print(f"Blocked roads           : {ROADS.blocked_roads.display()} "
      f"({ROADS.blocked_ratio.display()})")
print(f"Damaged (passable) roads: {ROADS.damaged_roads.display()}")
print(f"Critical roads          : {ROADS.critical_roads.display()}")
print(f"Critical roads blocked  : {ROADS.critical_roads_blocked.display()}")
print(f"Bridges affected        : {ROADS.bridges_affected.display()} of "
      f"{ROADS.bridges.display()}")
print(f"Alternative routes      : {ROADS.alternative_routes.display()}")
print(f"Accessibility score     : {ROADS.accessibility_score.display()}")
print(f"Transportation risk     : {ROADS.transportation_risk}")
print("")
if ROADS.warnings:
    print("Data availability warnings:")
    for warning in ROADS.warnings:
        print(f"  - {warning}")
else:
    print("Data availability warnings: none")
print("")
print("Road impact status: COMPLETED")
print("=" * 60)


## SECTION 10 — CRITICAL INFRASTRUCTURE IMPACT

Each infrastructure class is exposed according to the flood severity, weighted
by its **criticality** (a hospital matters more than a school), and the result is
a structured impact table plus a single critical-infrastructure risk score.

```
exposed_units  = total_units x severity / 100 x exposure_floor
critical_score = 100 x Σ (criticality_w x exposed_units) / Σ (criticality_w x total_units)
```

When a class has no inventory at all, its row is reported as
`DATA_UNAVAILABLE` rather than as zero facilities.


In [ ]:
# =============================================================================
# SECTION 10 - CRITICAL INFRASTRUCTURE IMPACT
# =============================================================================
#
# Builds a structured exposure table for every critical facility class and a
# single critical-infrastructure risk score.

# Classes reported in the impact table, in report order.
INFRASTRUCTURE_CLASSES: tuple[str, ...] = (
    "hospitals",
    "emergency_centers",
    "fire_stations",
    "police_stations",
    "water_facilities",
    "power_stations",
    "communication_towers",
    "bridges",
    "schools",
    "shelters",
)

# Bridges are counted as assets, not kilometres, and the field name differs.
INFRASTRUCTURE_FIELD: dict[str, str] = {
    "hospitals": "hospitals",
    "emergency_centers": "emergency_centers",
    "fire_stations": "fire_stations",
    "police_stations": "police_stations",
    "water_facilities": "water_facilities",
    "power_stations": "power_stations",
    "communication_towers": "communication_towers",
    "bridges": "bridges",
    "schools": "schools",
    "shelters": "shelters",
}


@dataclass
class InfrastructureImpact:
    """
    One facility class row of the impact table.

    Attributes:
        name: Class name.
        total: Installed units, or None when not supplied.
        exposed: Units estimated to be affected.
        exposure_ratio: Share of the class that is affected, 0-1.
        criticality: Configured criticality weight of the class.
        weighted_exposure: exposed x criticality, the risk contribution.
        source: Provenance of the `exposed` figure.
    """

    name: str
    total: int | None
    exposed: float | None
    exposure_ratio: float
    criticality: float
    weighted_exposure: float
    source: DataSource


@dataclass
class InfrastructureAnalysis:
    """
    Aggregated critical-infrastructure impact.

    Attributes:
        rows: One `InfrastructureImpact` per class.
        critical_score: 0-100 infrastructure risk score.
        risk_level: LOW / MEDIUM / HIGH / DANGER.
        total_exposed_units: Sum of exposed units over classes with inventory.
        critical_facilities_affected: Hospitals, emergency centres, fire and
            police stations that are affected.
        warnings: Data-availability warnings.
    """

    rows: list[InfrastructureImpact]
    critical_score: TrackedValue
    risk_level: str
    total_exposed_units: float
    critical_facilities_affected: float
    warnings: list[str] = field(default_factory=list)


def analyze_critical_infrastructure(
    infrastructure: InfrastructureInfo,
    flood: FloodAnalysis,
    config: EngineConfig = CONFIG,
) -> InfrastructureAnalysis:
    """
    Compute the critical-infrastructure exposure table and score.

    Args:
        infrastructure: The runtime infrastructure inventory.
        flood: The flood analysis (supplies severity).
        config: Engine configuration (currently unused directly; kept for
            signature stability and future thresholds).

    Returns:
        An `InfrastructureAnalysis`.
    """
    severity_fraction = float(flood.severity_score.value or 0.0) / 100.0
    rows: list[InfrastructureImpact] = []
    warnings: list[str] = []
    weighted_exposed_total = 0.0
    weighted_installed_total = 0.0
    total_exposed_units = 0.0
    critical_exposed = 0.0

    for name in INFRASTRUCTURE_CLASSES:
        field_name = INFRASTRUCTURE_FIELD[name]
        total_units = getattr(infrastructure, field_name, None)
        criticality = INFRASTRUCTURE_CRITICALITY.get(name, 0.5)
        if total_units is None or float(total_units) <= 0:
            rows.append(
                InfrastructureImpact(
                    name=name,
                    total=None,
                    exposed=None,
                    exposure_ratio=0.0,
                    criticality=criticality,
                    weighted_exposure=0.0,
                    source=DataSource.DATA_UNAVAILABLE,
                )
            )
            warnings.append(f"{name}: no inventory supplied")
            continue
        total_units = int(total_units)
        exposed = total_units * severity_fraction
        weighted_exposed_total += exposed * criticality
        weighted_installed_total += total_units * criticality
        total_exposed_units += exposed
        if name in {"hospitals", "emergency_centers", "fire_stations", "police_stations"}:
            critical_exposed += exposed
        rows.append(
            InfrastructureImpact(
                name=name,
                total=total_units,
                exposed=exposed,
                exposure_ratio=severity_fraction,
                criticality=criticality,
                weighted_exposure=exposed * criticality,
                source=DataSource.MODEL_ESTIMATED,
            )
        )

    if weighted_installed_total > 0:
        score = clamp(100.0 * weighted_exposed_total / weighted_installed_total)
        critical_score = calculated(
            score,
            "score",
            "100 x sum(exposed x criticality) / sum(installed x criticality)",
        )
    else:
        critical_score = unavailable(
            "score", "no infrastructure inventory supplied at all"
        )
        score = severity_fraction * 100.0

    return InfrastructureAnalysis(
        rows=rows,
        critical_score=critical_score,
        risk_level=risk_level(score),
        total_exposed_units=total_exposed_units,
        critical_facilities_affected=critical_exposed,
        warnings=warnings,
    )


INFRASTRUCTURE = analyze_critical_infrastructure(RUNTIME_EVENT.infrastructure, FLOOD)

print("=" * 60)
print("CRITICAL INFRASTRUCTURE IMPACT")
print("=" * 60)
print(
    f"{'Class':<22}{'Installed':>10}{'Exposed':>10}{'Ratio':>8}"
    f"{'Criticality':>13}{'Source':>22}"
)
print("-" * 85)
for row in INFRASTRUCTURE.rows:
    installed = f"{row.total:,}" if row.total is not None else "N/A"
    exposed = f"{row.exposed:,.1f}" if row.exposed is not None else "N/A"
    print(
        f"{row.name:<22}{installed:>10}{exposed:>10}"
        f"{row.exposure_ratio:>8.2f}{row.criticality:>13.2f}{row.source.value:>22}"
    )
print("-" * 85)
print(f"Total exposed units        : {INFRASTRUCTURE.total_exposed_units:,.1f}")
print(f"Critical response facilities affected: "
      f"{INFRASTRUCTURE.critical_facilities_affected:,.1f}")
print(f"Critical infrastructure risk: {INFRASTRUCTURE.risk_level} "
      f"({INFRASTRUCTURE.critical_score.display()})")
if INFRASTRUCTURE.warnings:
    print("")
    print("Data availability warnings:")
    for warning in INFRASTRUCTURE.warnings:
        print(f"  - {warning}")
print("")
print("Infrastructure impact status: COMPLETED")
print("=" * 60)


## SECTION 11 — AGRICULTURE AND LIVESTOCK IMPACT

Crop and livestock losses are **estimates** produced from two configurable
fractions of the exposed value:

```
crop_damage     = crop_value x severity/100 x CropConfig.crop_damage_fraction
livestock_damage = livestock_value x severity/100 x CropConfig.livestock_damage_fraction
```

Both are labelled `MODEL_ESTIMATED`, name the fraction that produced them, and
are reported as `DATA_UNAVAILABLE` when the value was not supplied. Nothing in
this section is a measured loss.


In [ ]:
# =============================================================================
# SECTION 11 - AGRICULTURE AND LIVESTOCK IMPACT
# =============================================================================
#
# Estimates crop and livestock losses from the exposed value and the flood
# severity. Every figure is an explicitly labelled estimate.


@dataclass
class AgricultureAnalysis:
    """
    Agricultural impact result.

    Attributes:
        agricultural_area: Affected agricultural area in km2.
        crop_area: Affected crop area in km2.
        crop_damage: Estimated crop loss in INR.
        livestock_damage: Estimated livestock loss in INR.
        livestock_affected: Estimated number of livestock affected.
        total_damage: crop_damage + livestock_damage.
        value_per_km2_check: Cross-check of the crop value against the area.
    """

    agricultural_area: TrackedValue
    crop_area: TrackedValue
    crop_damage: TrackedValue
    livestock_damage: TrackedValue
    livestock_affected: TrackedValue
    total_damage: TrackedValue
    value_per_km2_check: TrackedValue


def analyze_agriculture(
    economics: EconomicInfo,
    area: AreaAnalysis,
    flood: FloodAnalysis,
    config: EngineConfig = CONFIG,
) -> AgricultureAnalysis:
    """
    Estimate crop and livestock damage.

    Args:
        economics: The runtime economic record.
        area: The affected-area analysis (for the agricultural area split).
        flood: The flood analysis (supplies severity).
        config: Engine configuration.

    Returns:
        An `AgricultureAnalysis`.
    """
    severity_fraction = float(flood.severity_score.value or 0.0) / 100.0
    agri = config.agriculture

    if economics.agricultural_area is not None:
        agricultural_area = user_value(float(economics.agricultural_area), UNIT_KM2)
    else:
        agricultural_area = estimated(
            float(area.agricultural.value or 0.0),
            UNIT_KM2,
            "AreaConfig.land_use_fractions['agricultural'] x total area",
            Confidence.LOW,
        )

    if economics.crop_area is not None:
        crop_area = user_value(float(economics.crop_area), UNIT_KM2)
    else:
        crop_area = estimated(
            float(agricultural_area.value or 0.0),
            UNIT_KM2,
            "whole agricultural area assumed to be under crop",
            Confidence.LOW,
        )

    if economics.crop_value is not None:
        crop_damage = estimated(
            float(economics.crop_value) * severity_fraction * agri.crop_damage_fraction,
            UNIT_INR,
            f"crop_value x severity/100 x AgricultureConfig.crop_damage_fraction "
            f"({agri.crop_damage_fraction:g})",
        )
    else:
        crop_damage = unavailable(
            UNIT_INR, "economics.crop_value not supplied - no crop value to damage"
        )

    if economics.livestock_value is not None:
        livestock_damage = estimated(
            float(economics.livestock_value)
            * severity_fraction
            * agri.livestock_damage_fraction,
            UNIT_INR,
            f"livestock_value x severity/100 x "
            f"AgricultureConfig.livestock_damage_fraction "
            f"({agri.livestock_damage_fraction:g})",
        )
    else:
        livestock_damage = unavailable(
            UNIT_INR, "economics.livestock_value not supplied"
        )

    if economics.livestock is not None:
        livestock_affected = estimated(
            int(float(economics.livestock) * severity_fraction),
            "livestock",
            f"livestock x severity/100 ({severity_fraction:.4f})",
        )
    else:
        livestock_affected = unavailable("livestock", "economics.livestock not supplied")

    crop_loss = float(crop_damage.value or 0.0)
    livestock_loss = float(livestock_damage.value or 0.0)
    total_damage = calculated(
        crop_loss + livestock_loss,
        UNIT_INR,
        "crop_damage + livestock_damage",
    )

    crop_area_km2 = float(crop_area.value or 0.0)
    if economics.crop_value is not None and crop_area_km2 > 0:
        per_km2 = calculated(
            float(economics.crop_value) / crop_area_km2,
            UNIT_INR + "/km2",
            f"crop_value / crop_area; "
            f"AgricultureConfig.standing_crop_value_per_km2 reference is "
            f"{agri.standing_crop_value_per_km2:,.0f}",
        )
    else:
        per_km2 = unavailable(UNIT_INR + "/km2", "crop value or crop area not supplied")

    return AgricultureAnalysis(
        agricultural_area=agricultural_area,
        crop_area=crop_area,
        crop_damage=crop_damage,
        livestock_damage=livestock_damage,
        livestock_affected=livestock_affected,
        total_damage=total_damage,
        value_per_km2_check=per_km2,
    )


AGRICULTURE = analyze_agriculture(RUNTIME_EVENT.economics, AREA, FLOOD)

print("=" * 60)
print("AGRICULTURE AND LIVESTOCK IMPACT")
print("=" * 60)
print(f"Agricultural area   : {AGRICULTURE.agricultural_area.display()}")
print(f"Crop area           : {AGRICULTURE.crop_area.display()}")
print(f"Crop value per km2  : {AGRICULTURE.value_per_km2_check.display()}")
print(f"Crop damage (EST.)  : {AGRICULTURE.crop_damage.display()}")
print(f"Livestock affected  : {AGRICULTURE.livestock_affected.display()}")
print(f"Livestock damage    : {AGRICULTURE.livestock_damage.display()}")
print(f"Total agri damage   : {AGRICULTURE.total_damage.display()}")
print("")
print("Labels:")
print(f"  crop_damage        -> {AGRICULTURE.crop_damage.source.value} "
      f"({AGRICULTURE.crop_damage.assumption})")
print(f"  livestock_damage   -> {AGRICULTURE.livestock_damage.source.value} "
      f"({AGRICULTURE.livestock_damage.assumption})")
print("")
print("Agriculture impact status: COMPLETED")
print("=" * 60)


## SECTION 12 — RISK ENGINE

The risk engine is deliberately **transparent**: seven independent components,
each scored 0–100 from an already-computed analysis, combined with configurable
weights. No component is hidden, and the printed block always shows the
components next to the total.

```
risk_score = Σ wᵢ · componentᵢ            (weights normalised to 1.0)
```

| Component | Driven by |
| --- | --- |
| Flood risk | flood severity score |
| Population risk | exposure ratio, vulnerable and assistance-requiring shares |
| Infrastructure risk | criticality-weighted facility exposure |
| Road risk | `100 - road accessibility score` |
| Medical risk | medical demand versus available capacity |
| Evacuation risk | available response time versus required mobilisation time |
| Economic risk | exposed asset value versus a configured reference value |

A component whose input is unavailable is reported as `DATA_UNAVAILABLE` and is
**excluded from the weighted sum**, with the weights renormalised over the
available components. The report states how many components were available, so a
score computed from two components is never mistaken for a seven-component score.

### Band mapping

| Score | Level |
| --- | --- |
| 0 – 24 | LOW |
| 25 – 49 | MEDIUM |
| 50 – 74 | HIGH |
| 75 – 100 | DANGER |


In [ ]:
# =============================================================================
# SECTION 12 - RISK ENGINE
# =============================================================================
#
# Seven transparent 0-100 risk components, combined with configurable weights.
# Unavailable components are excluded and the weights are renormalised.

# Reference value used to scale the exposed economic asset base to 0-100.
ECONOMIC_RISK_REFERENCE_INR = 20.0e9


@dataclass(frozen=True)
class RiskComponent:
    """
    One risk dimension.

    Attributes:
        name: Component name, matching the key in `RiskConfig.weights`.
        score: 0-100 score, or None when the input is unavailable.
        weight: Configured weight.
        rationale: The formula that produced the score.
        available: False when the underlying data is missing.
    """

    name: str
    score: float | None
    weight: float
    rationale: str
    available: bool

    def effective_weight(self, weight_sum: float) -> float:
        """Normalised weight within the set of available components."""
        if not self.available or weight_sum <= 0:
            return 0.0
        return self.weight / weight_sum


@dataclass
class RiskAssessment:
    """
    Aggregated risk assessment.

    Attributes:
        components: The seven `RiskComponent` records.
        overall_score: Weighted 0-100 risk score.
        overall_level: LOW / MEDIUM / HIGH / DANGER.
        available_components: How many components contributed.
        total_components: How many components were defined.
        formula: The literal formula string, printed in the report.
    """

    components: list[RiskComponent]
    overall_score: TrackedValue
    overall_level: str
    available_components: int
    total_components: int
    formula: str

    def component(self, name: str) -> RiskComponent:
        """Return the named component, raising if it does not exist."""
        for item in self.components:
            if item.name == name:
                return item
        raise KeyError(f"no risk component named '{name}'")


def required_medical_teams(
    population: PopulationAnalysis, config: EngineConfig = CONFIG
) -> float | None:
    """
    Number of medical teams the evacuation load requires.

    Shared by the risk engine (medical-risk component) and the demand estimator,
    so both always report the same requirement.

    Args:
        population: The population analysis.
        config: Engine configuration.

    Returns:
        The requirement, or None when no evacuation load is available.
    """
    if not population.available:
        return None
    evacuees = float(population.evacuation.value or 0.0)
    if evacuees <= 0:
        return 0.0
    return float(
        math.ceil(
            evacuees
            / 10000.0
            * config.demand.medical_teams_per_10000_evacuees
        )
    )


def medical_capacity_ratio(
    population: PopulationAnalysis,
    resources: ResourceAvailability,
    config: EngineConfig = CONFIG,
) -> float | None:
    """
    Ratio of required to available medical teams.

    Args:
        population: The population analysis.
        resources: The runtime resource availability.
        config: Engine configuration.

    Returns:
        `required / available`, or None when the requirement or the availability
        is unknown (an unknown availability is never treated as zero).
    """
    required = required_medical_teams(population, config)
    if required is None:
        return None
    if resources.medical_teams is None or float(resources.medical_teams) <= 0:
        return None
    return required / float(resources.medical_teams)


def assess_risk(
    flood: FloodAnalysis,
    population: PopulationAnalysis,
    infrastructure: InfrastructureAnalysis,
    roads: RoadAnalysis,
    medical_capacity_ratio: float | None,
    arrival: ArrivalAnalysis,
    economics: EconomicInfo,
    config: EngineConfig = CONFIG,
) -> RiskAssessment:
    """
    Compute the seven risk components and the overall risk score.

    Args:
        flood: Flood analysis (severity).
        population: Population analysis.
        infrastructure: Infrastructure analysis.
        roads: Road analysis (accessibility).
        medical_capacity_ratio: required / available medical teams, or None
            when medical data is unavailable.
        arrival: Arrival analysis (response window).
        economics: Economic record.
        config: Engine configuration.

    Returns:
        A `RiskAssessment`.
    """
    weights = dict(config.risk.weights)
    components: list[RiskComponent] = []

    # -- Flood risk ----------------------------------------------------------
    flood_score = float(flood.severity_score.value or 0.0)
    components.append(
        RiskComponent(
            name="flood",
            score=flood_score,
            weight=weights.get("flood", 0.0),
            rationale="flood severity score (SECTION 5)",
            available=True,
        )
    )

    # -- Population risk -----------------------------------------------------
    if population.available and population.total_population.value:
        total = float(population.total_population.value)
        exposed_ratio = float(population.exposed.value or 0) / total
        vulnerable_ratio = float(population.vulnerable.value or 0) / total
        assistance_ratio = float(
                            population.requiring_assistance.value or 0
                        ) / total
        score = clamp(
            100.0
            * (
                0.60 * exposed_ratio / max(config.population.max_exposure_ratio, 1e-9)
                + 0.25 * vulnerable_ratio
                + 0.15 * assistance_ratio
            )
        )
        components.append(
            RiskComponent(
                name="population",
                score=score,
                weight=weights.get("population", 0.0),
                rationale=(
                    f"0.60 x exposed ratio ({exposed_ratio:.4f}) / max exposure ratio "
                    f"+ 0.25 x vulnerable ratio ({vulnerable_ratio:.4f}) "
                    f"+ 0.15 x assistance ratio ({assistance_ratio:.4f})"
                ),
                available=True,
            )
        )
    else:
        components.append(
            RiskComponent(
                name="population",
                score=None,
                weight=weights.get("population", 0.0),
                rationale="no population data supplied",
                available=False,
            )
        )

    # -- Infrastructure risk -------------------------------------------------
    infra_score = (
        float(infrastructure.critical_score.value)
        if infrastructure.critical_score.available
        else None
    )
    components.append(
        RiskComponent(
            name="infrastructure",
            score=infra_score,
            weight=weights.get("infrastructure", 0.0),
            rationale=infrastructure.critical_score.assumption
            or "no infrastructure inventory supplied",
            available=infra_score is not None,
        )
    )

    # -- Road risk -----------------------------------------------------------
    road_score = 100.0 - float(roads.accessibility_score.value or 0.0)
    components.append(
        RiskComponent(
            name="road",
            score=road_score,
            weight=weights.get("road", 0.0),
            rationale="100 - road accessibility score (SECTION 9); "
            + str(roads.accessibility_score.assumption),
            available=roads.accessibility_score.available,
        )
    )

    # -- Medical risk --------------------------------------------------------
    if medical_capacity_ratio is None:
        components.append(
            RiskComponent(
                name="medical",
                score=None,
                weight=weights.get("medical", 0.0),
                rationale="medical demand or availability is unavailable",
                available=False,
            )
        )
    else:
        score = clamp(100.0 * (float(medical_capacity_ratio) - 1.0) + 50.0)
        components.append(
            RiskComponent(
                name="medical",
                score=score,
                weight=weights.get("medical", 0.0),
                rationale=(
                    f"100 x (required/available medical teams - 1) + 50, with "
                    f"required/available = {float(medical_capacity_ratio):.4f}; "
                    "a ratio of 1 is the neutral 50-point point"
                ),
                available=True,
            )
        )

    # -- Evacuation risk -----------------------------------------------------
    available_hours = arrival.available_response_time_hours.value
    required_hours = float(arrival.required_response_time_hours.value or 0.0)
    if available_hours is None or required_hours <= 0:
        components.append(
            RiskComponent(
                name="evacuation",
                score=None,
                weight=weights.get("evacuation", 0.0),
                rationale="arrival time unavailable, so the response window is unknown",
                available=False,
            )
        )
    else:
        ratio = float(available_hours) / required_hours
        score = clamp(100.0 * (1.0 - min(1.0, ratio / 4.0)))
        components.append(
            RiskComponent(
                name="evacuation",
                score=score,
                weight=weights.get("evacuation", 0.0),
                rationale=(
                    f"100 x (1 - min(1, available/required / 4)) with "
                    f"available={float(available_hours):.2f} h, "
                    f"required={required_hours:.2f} h, ratio={ratio:.3f}; "
                    "a ratio of 4 or more scores 0"
                ),
                available=True,
            )
        )

    # -- Economic risk -------------------------------------------------------
    exposed_assets = sum(
        float(value or 0.0)
        for value in (
            economics.land_value,
            economics.crop_value,
            economics.infrastructure_value,
            economics.business_value,
            economics.industrial_value,
        )
    )
    has_assets = any(
        value is not None
        for value in (
            economics.land_value,
            economics.crop_value,
            economics.infrastructure_value,
            economics.business_value,
            economics.industrial_value,
        )
    )
    if has_assets and exposed_assets > 0:
        ratio = exposed_assets / ECONOMIC_RISK_REFERENCE_INR
        score = clamp(100.0 * ratio ** 0.5)
        components.append(
            RiskComponent(
                name="economic",
                score=score,
                weight=weights.get("economic", 0.0),
                rationale=(
                    f"sqrt(100 x exposed assets / ECONOMIC_RISK_REFERENCE_INR) with "
                    f"exposed assets = {exposed_assets:,.0f} {UNIT_INR} and reference "
                    f"= {ECONOMIC_RISK_REFERENCE_INR:,.0f} {UNIT_INR}; the square root "
                    "compresses the very large dynamic range of asset values"
                ),
                available=True,
            )
        )
    else:
        components.append(
            RiskComponent(
                name="economic",
                score=None,
                weight=weights.get("economic", 0.0),
                rationale="no economic values supplied",
                available=False,
            )
        )

    available_components = [item for item in components if item.available]
    weight_sum = sum(item.weight for item in available_components)
    if available_components and weight_sum > 0:
        overall = sum(
            item.score * item.effective_weight(weight_sum)
            for item in available_components
            if item.score is not None
        )
        overall_value = calculated(
            overall,
            "score",
            f"weighted sum over {len(available_components)} available components, "
            f"weights renormalised to sum to 1.0 (raw weight sum {weight_sum:.3f})",
        )
    else:
        overall_value = unavailable("score", "no risk component could be computed")

    formula = (
        "risk_score = sum(weight_i x component_i) / sum(weight_i) over the available "
        "components; weights are the normalised RiskConfig.weights"
    )
    return RiskAssessment(
        components=components,
        overall_score=overall_value,
        overall_level=risk_level(float(overall_value.value or 0.0))
        if overall_value.available
        else DATA_UNAVAILABLE_LABEL,
        available_components=len(available_components),
        total_components=len(components),
        formula=formula,
    )


def display_risk(assessment: RiskAssessment) -> None:
    """Print the required RISK ASSESSMENT block."""
    print("=" * 60)
    print("RISK ASSESSMENT")
    print("=" * 60)
    print(f"{'Component':<20}{'Score':>9}{'Weight':>9}{'Norm.':>8}   Rationale")
    print("-" * 60)
    weight_sum = sum(item.weight for item in assessment.components if item.available)
    for item in assessment.components:
        score_text = f"{item.score:8.2f}" if item.score is not None else "   N/A  "
        print(
            f"{item.name:<20}{score_text}{item.weight:>9.3f}"
            f"{item.effective_weight(weight_sum):>8.3f}   {item.rationale}"
        )
    print("-" * 60)
    print(f"Components available     : {assessment.available_components} of "
          f"{assessment.total_components}")
    print(f"Overall Risk            : {assessment.overall_score.display()}")
    print(f"Risk Level              : {assessment.overall_level}")
    print(f"Formula                 : {assessment.formula}")
    print("Risk assessment status  : COMPLETED")
    print("=" * 60)


In [ ]:
# ---- Run the risk engine -----------------------------------------------------

MEDICAL_CAPACITY_RATIO = medical_capacity_ratio(POPULATION, RUNTIME_EVENT.resources, CONFIG)

RISK = assess_risk(
    FLOOD,
    POPULATION,
    INFRASTRUCTURE,
    ROADS,
    MEDICAL_CAPACITY_RATIO,
    ARRIVAL,
    RUNTIME_EVENT.economics,
    CONFIG,
)

display_risk(RISK)

print("")
print("The medical-risk component and SECTION 13 share the helper")
print("`required_medical_teams`, so both report an identical requirement.")
print(f"Required medical teams  : {required_medical_teams(POPULATION, CONFIG):,.0f}")
if MEDICAL_CAPACITY_RATIO is None:
    print("Medical capacity ratio   : DATA_UNAVAILABLE")
else:
    print(f"Medical capacity ratio   : {MEDICAL_CAPACITY_RATIO:.4f}")


## SECTION 13 — RESOURCE DEMAND ESTIMATION

Demand is computed from the **evacuation load**, not from the raw population, and
from the **relief horizon**, not from a single day. Every ratio is configurable
and is printed with the value it produced.

| Category | Quantity | Formula |
| --- | --- | --- |
| Rescue | teams | `evacuees/1000 x rescue_teams_per_1000_evacuees` |
| Rescue | vehicles | `teams x rescue_vehicles_per_team` |
| Rescue | boats | `evacuees/1000 x boats_per_1000_evacuees` |
| Medical | teams | `evacuees/10000 x medical_teams_per_10000_evacuees` |
| Medical | doctors | `evacuees/10000 x doctors_per_10000_evacuees` |
| Medical | nurses | `doctors x nurses_per_doctor` |
| Medical | ambulances | `teams x ambulances_per_medical_team` |
| Relief | water | `evacuees x days x water_litres_per_person_per_day` |
| Relief | food | `evacuees x days x food_meals_per_person_per_day` |
| Relief | medicines | `evacuees/1000 x days x medicines_kits_per_1000_persons` |
| Relief | blankets | `evacuees x blankets_per_person` |
| Relief | sanitation kits | `evacuees/100 x days x sanitation_kits_per_100_persons` |
| Relief | generators | `shelters_required x generators_per_shelter` |
| Relief | fuel | `rescue teams x days x fuel_litres_per_team_per_day` |
| Shelter | capacity | `evacuees / shelter_occupancy_ratio` |
| Shelter | units | `capacity / shelter_capacity_persons` |

Every row is printed as `REQUIRED / AVAILABLE / DEFICIT`, and the deficit
percentage is shown next to it. When availability is unknown the row reports
`DATA_UNAVAILABLE` for the available column instead of pretending it is zero.


In [ ]:
# =============================================================================
# SECTION 13 - RESOURCE DEMAND ESTIMATION
# =============================================================================
#
# Converts the evacuation load and the relief horizon into a REQUIRED / AVAILABLE
# / DEFICIT line for every resource class. All ratios come from DemandConfig.


@dataclass
class ResourceLine:
    """
    One required / available / deficit line.

    Attributes:
        name: Resource name, e.g. "rescue_teams".
        category: RESCUE / MEDICAL / RELIEF / SHELTER.
        unit: Unit label of the quantity.
        required: Requirement, or None when the load could not be computed.
        available: Availability, or None when not reported.
        deficit: max(0, required - available), or None.
        deficit_pct: Deficit as a share of the requirement, or None.
        formula: The arithmetic that produced `required`.
        ratio_source: The configuration field that supplied the ratio.
    """

    name: str
    category: str
    unit: str
    required: float | None
    available: float | None
    deficit: float | None
    deficit_pct: float | None
    formula: str
    ratio_source: str

    @property
    def has_deficit(self) -> bool:
        """True when a positive deficit was detected."""
        return self.deficit is not None and self.deficit > 0

    @property
    def available_known(self) -> bool:
        """True when the availability figure was reported."""
        return self.available is not None


@dataclass
class DemandEstimate:
    """
    Full demand estimate.

    Attributes:
        lines: Every `ResourceLine`, in report order.
        categories: Lines grouped by category, in report order.
        evacuees: The evacuation load the demand is based on.
        relief_days: The relief horizon.
        total_deficit_count: Number of lines with a positive deficit.
        shelter_deficit: Shelter capacity shortfall in people, or None.
        additional_shelters: Additional shelter units required, or None.
    """

    lines: list[ResourceLine]
    evacuees: float
    relief_days: float
    total_deficit_count: int
    shelter_deficit: float | None
    additional_shelters: float | None

    def category(self, name: str) -> list[ResourceLine]:
        """Return the lines of one category."""
        return [line for line in self.lines if line.category == name]

    def line(self, name: str) -> ResourceLine:
        """Return one line by resource name."""
        for item in self.lines:
            if item.name == name:
                return item
        raise KeyError(f"no demand line named '{name}'")


CATEGORY_ORDER: tuple[str, ...] = ("RESCUE", "MEDICAL", "RELIEF", "SHELTER")


def estimate_resource_demand(
    population: PopulationAnalysis,
    resources: ResourceAvailability,
    config: EngineConfig = CONFIG,
) -> DemandEstimate:
    """
    Estimate the required resources and compare them with availability.

    Args:
        population: The population analysis (supplies the evacuation load).
        resources: The runtime resource availability record.
        config: Engine configuration.

    Returns:
        A `DemandEstimate`.
    """
    cfg = config.demand
    evacuees = float(population.evacuation.value or 0.0) if population.available else 0.0
    days = float(cfg.relief_horizon_days)

    def make(
        name: str,
        category: str,
        unit: str,
        required: float | None,
        available: float | None,
        formula: str,
        ratio_source: str,
    ) -> ResourceLine:
        if required is None:
            deficit = None
            deficit_pct = None
        elif available is None:
            deficit = None
            deficit_pct = None
        else:
            deficit = max(0.0, float(required) - float(available))
            deficit_pct = (deficit / float(required) * 100.0) if required > 0 else 0.0
        return ResourceLine(
            name=name,
            category=category,
            unit=unit,
            required=required,
            available=available,
            deficit=deficit,
            deficit_pct=deficit_pct,
            formula=formula,
            ratio_source=ratio_source,
        )

    if not population.available:
        no_data = "no evacuation load available (no population data supplied)"
    else:
        no_data = ""

    # -- Rescue --------------------------------------------------------------
    rescue_teams = evacuees / 1000.0 * cfg.rescue_teams_per_1000_evacuees
    rescue_vehicles = rescue_teams * cfg.rescue_vehicles_per_team
    boats = evacuees / 1000.0 * cfg.boats_per_1000_evacuees

    lines: list[ResourceLine] = [
        make(
            "rescue_teams", "RESCUE", "teams", math.ceil(rescue_teams) or None,
            resources.rescue_teams,
            f"ceil({evacuees:,.0f}/1000 x {cfg.rescue_teams_per_1000_evacuees:g})",
            "DemandConfig.rescue_teams_per_1000_evacuees",
        ),
        make(
            "rescue_vehicles", "RESCUE", "vehicles",
            math.ceil(rescue_vehicles) or None, resources.vehicles,
            f"ceil({rescue_teams:,.2f} x {cfg.rescue_vehicles_per_team:g})",
            "DemandConfig.rescue_vehicles_per_team",
        ),
        make(
            "boats", "RESCUE", "boats", math.ceil(boats) or None, resources.boats,
            f"ceil({evacuees:,.0f}/1000 x {cfg.boats_per_1000_evacuees:g})",
            "DemandConfig.boats_per_1000_evacuees",
        ),
        # -- Medical --------------------------------------------------------
        make(
            "medical_teams", "MEDICAL", "teams",
            math.ceil(evacuees / 10000.0 * cfg.medical_teams_per_10000_evacuees) or None,
            resources.medical_teams,
            f"ceil({evacuees:,.0f}/10000 x {cfg.medical_teams_per_10000_evacuees:g})",
            "DemandConfig.medical_teams_per_10000_evacuees",
        ),
        make(
            "doctors", "MEDICAL", "doctors",
            math.ceil(evacuees / 10000.0 * cfg.doctors_per_10000_evacuees) or None,
            resources.doctors,
            f"ceil({evacuees:,.0f}/10000 x {cfg.doctors_per_10000_evacuees:g})",
            "DemandConfig.doctors_per_10000_evacuees",
        ),
        make(
            "nurses", "MEDICAL", "nurses",
            math.ceil(evacuees / 10000.0 * cfg.doctors_per_10000_evacuees
                      * cfg.nurses_per_doctor) or None,
            resources.nurses,
            f"ceil(doctors x {cfg.nurses_per_doctor:g})",
            "DemandConfig.nurses_per_doctor",
        ),
        make(
            "ambulances", "MEDICAL", "ambulances",
            math.ceil(evacuees / 10000.0 * cfg.medical_teams_per_10000_evacuees
                      * cfg.ambulances_per_medical_team) or None,
            resources.ambulances,
            f"ceil(medical teams x {cfg.ambulances_per_medical_team:g})",
            "DemandConfig.ambulances_per_medical_team",
        ),
        # -- Relief ---------------------------------------------------------
        make(
            "water_litres", "RELIEF", "litres",
            evacuees * days * cfg.water_litres_per_person_per_day or None,
            resources.water_litres,
            f"{evacuees:,.0f} x {days:g} x {cfg.water_litres_per_person_per_day:g}",
            "DemandConfig.water_litres_per_person_per_day",
        ),
        make(
            "food_meals", "RELIEF", "meals",
            evacuees * days * cfg.food_meals_per_person_per_day or None,
            resources.food_meals,
            f"{evacuees:,.0f} x {days:g} x {cfg.food_meals_per_person_per_day:g}",
            "DemandConfig.food_meals_per_person_per_day",
        ),
        make(
            "medicines_kits", "RELIEF", "kits",
            math.ceil(evacuees / 1000.0 * days * cfg.medicines_kits_per_1000_persons)
            or None,
            resources.medicines_kits,
            f"ceil({evacuees:,.0f}/1000 x {days:g} x {cfg.medicines_kits_per_1000_persons:g})",
            "DemandConfig.medicines_kits_per_1000_persons",
        ),
        make(
            "blankets", "RELIEF", "blankets",
            math.ceil(evacuees * cfg.blankets_per_person) or None, None,
            f"ceil({evacuees:,.0f} x {cfg.blankets_per_person:g})",
            "DemandConfig.blankets_per_person",
        ),
        make(
            "sanitation_kits", "RELIEF", "kits",
            math.ceil(evacuees / 100.0 * days * cfg.sanitation_kits_per_100_persons)
            or None,
            None,
            f"ceil({evacuees:,.0f}/100 x {days:g} x {cfg.sanitation_kits_per_100_persons:g})",
            "DemandConfig.sanitation_kits_per_100_persons",
        ),
        # -- Shelter --------------------------------------------------------
        make(
            "shelter_capacity", "SHELTER", "people",
            math.ceil(evacuees / cfg.shelter_occupancy_ratio) or None,
            resources.shelter_capacity,
            f"ceil({evacuees:,.0f} / {cfg.shelter_occupancy_ratio:g})",
            "DemandConfig.shelter_occupancy_ratio",
        ),
        make(
            "shelters", "SHELTER", "shelters",
            math.ceil(
                evacuees
                / (cfg.shelter_capacity_persons * cfg.shelter_occupancy_ratio)
            ) or None,
            resources.shelters,
            f"ceil({evacuees:,.0f} / ({cfg.shelter_capacity_persons} x "
            f"{cfg.shelter_occupancy_ratio:g}))",
            "DemandConfig.shelter_capacity_persons, DemandConfig.shelter_occupancy_ratio",
        ),
    ]

    required_shelters = math.ceil(
        evacuees / (cfg.shelter_capacity_persons * cfg.shelter_occupancy_ratio)
    )
    required_capacity = math.ceil(evacuees / cfg.shelter_occupancy_ratio)
    available_shelters = resources.shelters
    available_capacity = resources.shelter_capacity
    if available_shelters is None or available_capacity is None:
        shelter_deficit = None
        additional = None
    else:
        shelter_deficit = max(0.0, required_capacity - float(available_capacity))
        additional = max(
            0,
            required_shelters - int(available_shelters),
            math.ceil(
                shelter_deficit
                / (cfg.shelter_capacity_persons * cfg.shelter_occupancy_ratio)
            )
            if shelter_deficit > 0
            else 0,
        )

    generators_required = math.ceil(required_shelters * cfg.generators_per_shelter)
    lines.append(
        make(
            "generators", "RELIEF", "generators", generators_required or None, None,
            f"ceil({required_shelters} x {cfg.generators_per_shelter:g})",
            "DemandConfig.generators_per_shelter",
        )
    )
    lines.append(
        make(
            "fuel_litres", "RELIEF", "litres",
            math.ceil(rescue_teams * days * cfg.fuel_litres_per_team_per_day) or None,
            None,
            f"ceil({rescue_teams:,.2f} teams x {days:g} x "
            f"{cfg.fuel_litres_per_team_per_day:g})",
            "DemandConfig.fuel_litres_per_team_per_day",
        )
    )

    if not population.available:
        for line in lines:
            line.formula = f"{no_data}; {line.formula}"

    return DemandEstimate(
        lines=lines,
        evacuees=evacuees,
        relief_days=days,
        total_deficit_count=sum(1 for line in lines if line.has_deficit),
        shelter_deficit=shelter_deficit,
        additional_shelters=float(additional) if additional is not None else None,
    )


def display_demand(demand: DemandEstimate) -> None:
    """Print the REQUIRED / AVAILABLE / DEFICIT table for every resource."""
    print("=" * 60)
    print("RESOURCE DEMAND ESTIMATION")
    print("=" * 60)
    print(f"Evacuation load     : {demand.evacuees:,.0f} people")
    print(f"Relief horizon      : {demand.relief_days:g} days")
    print("")
    header = f"{'Category':<9}{'Resource':<18}{'Required':>14}{'Available':>14}{'Deficit':>12}{'Def %':>8}"
    print(header)
    print("-" * 75)
    for category in CATEGORY_ORDER:
        for line in demand.category(category):
            required = f"{line.required:,.0f}" if line.required is not None else "N/A"
            available = f"{line.available:,.0f}" if line.available is not None else "N/A"
            deficit = f"{line.deficit:,.0f}" if line.deficit is not None else "N/A"
            deficit_pct = f"{line.deficit_pct:,.1f}%" if line.deficit_pct is not None else "N/A"
            print(
                f"{category:<9}{line.name:<18}{required:>14}{available:>14}"
                f"{deficit:>12}{deficit_pct:>8}"
            )
    print("-" * 75)
    print(f"Lines with a deficit : {demand.total_deficit_count} of {len(demand.lines)}")
    if demand.shelter_deficit is not None:
        print(f"Shelter deficit      : {demand.shelter_deficit:,.0f} people")
        print(f"Additional shelters  : {demand.additional_shelters:,.0f}")
    else:
        print("Shelter deficit      : DATA_UNAVAILABLE")
    print("")
    print("Configurable ratios used:")
    for line in demand.lines:
        print(f"  {line.name:<18} {line.formula}")
        print(f"  {'':<18} ratio source: {line.ratio_source}")
    print("")
    print("Demand estimation status: COMPLETED")
    print("=" * 60)


In [ ]:
# ---- Run the demand estimator ------------------------------------------------

DEMAND = estimate_resource_demand(POPULATION, RUNTIME_EVENT.resources, CONFIG)

display_demand(DEMAND)

print("")
print("Available figures that the runtime input did not report are shown as N/A,")
print("never as zero: a missing inventory is not evidence of a shortage, and it is")
print("not evidence of sufficiency either.")
missing = [line.name for line in DEMAND.lines if not line.available_known]
print(f"Availability not reported for : {', '.join(missing) if missing else 'nothing'}")


## SECTION 14 — OPTIMIZATION PREPROCESSOR (RUNTIME → QUBO INPUTS)

This is the **only** place where runtime analysis becomes quantum-optimization
input. Nothing downstream of this section reinterprets raw flood data.

### 1. Deployment slots

The preserved quantum core constrains the number of **selected locations**, not
the number of teams. The runtime engine therefore converts teams into slots:

```
deployment_slots = floor(rescue_teams / OptimizationConfig.units_per_deployment_slot)
deployment_slots = clamp(deployment_slots, 1, n_locations)
```

A location is a unit of rescue: one deployment slot is a coordinated team group
capable of reaching and evacuating one location. This is stated explicitly, and
it is what makes the "30 teams → 3 slots" and "50 teams → 4 slots" sensitivity
cases in SECTION 31 meaningful.

### 2. Per-location population

Runtime input supplies one regional population, so each location's share is
allocated by an explicit, configurable basis:

```
share_i      = w_pop·(population_i / Σ population) + w_trav·(1 - travel_time_i / max travel) ...
location_population_i = floor(RegionalPopulation x share_i x urgency_relevance)
```

Shares are renormalised to sum to exactly 1.0 before allocation, and the sum of
the allocated populations is printed so the split is auditable.

### 3. Per-location urgency

```
urgency_i = 100 x Σ wⱼ·componentⱼ   (population, vulnerability, infrastructure,
                                   road access, flood severity, travel time)
```

Only normalized 0–100 features are passed on. **No TMC, no INR, no km, no m³/s
ever reaches QAOA.**

### 4. Handoff

The result is a list of `FloodLocation` records — the exact type the preserved
core validates — plus an `InputBundle`. Nothing about the runtime input is
discarded: the urgency components remain in `DECISION_MATRIX` for the report.


In [ ]:
# =============================================================================
# SECTION 14 - OPTIMIZATION PREPROCESSOR
# =============================================================================
#
# The single boundary between runtime analysis and quantum optimization.
# Produces `FloodLocation` records for the preserved quantum core.

# --- The preserved quantum core's location type ------------------------------
#
# Only the data structure is imported here. All quantum mathematics (QUBO,
# Hamiltonian, QAOA, sampling, decoding) is carried over unchanged in SECTION 15.


@dataclass(frozen=True)
class FloodLocation:
    """
    One flood-affected location considered for rescue allocation.

    This is the preserved core's type. Only its fields are listed here; the
    validation rules, the score function and every quantum routine that consume
    it are carried over unchanged from `q_flare_quantum_core.py`.

    Attributes:
        location_id: Unique identifier, e.g. "L1".
        name: Human readable place name, e.g. "Riverbank-A".
        affected_population: Number of people affected (non-negative integer).
        urgency: Urgency / severity score in [0, 100] (higher = more urgent).
        latitude: Optional illustrative latitude (GIS hand-off).
        longitude: Optional illustrative longitude (GIS hand-off).
    """

    location_id: str
    name: str
    affected_population: int
    urgency: float
    latitude: float | None = None
    longitude: float | None = None


@dataclass
class LocationFeatureRow:
    """
    One row of the auditable decision matrix.

    Attributes:
        location: The `LocationInput` this row was derived from.
        components: Named 0-100 urgency components.
        weights: The weight applied to each component.
        urgency: The weighted composite urgency in [0, 100].
        population_share: Share of the regional evacuation load.
        population: The allocated population.
    """

    location: LocationInput
    components: dict[str, float]
    weights: dict[str, float]
    urgency: float
    population_share: float
    population: int


@dataclass
class OptimizationInput:
    """
    Everything the quantum stage receives, plus the audit trail.

    Attributes:
        locations: `FloodLocation` records for the preserved core.
        rescue_teams: Raw rescue teams from the runtime input.
        deployment_slots: The hard constraint actually used by the core.
        key_resource: Name of the resource the slots were derived from.
        matrix: The auditable per-location feature table.
        severity_score: Regional flood severity used by every urgency component.
        warnings: Non-fatal notes about the handoff.
    """

    locations: list[FloodLocation]
    rescue_teams: int
    deployment_slots: int
    key_resource: str
    matrix: list[LocationFeatureRow]
    severity_score: float
    warnings: list[str] = field(default_factory=list)

    @property
    def n_locations(self) -> int:
        """Number of binary decision variables (= number of qubits)."""
        return len(self.locations)

    @property
    def total_population(self) -> int:
        """Sum of the allocated populations."""
        return sum(row.population for row in self.matrix)

    def urgency_formula(self) -> str:
        """Human readable urgency formula with the actual weights."""
        parts = ", ".join(
            f"{name} x {self.matrix[0].weights[name]:g}"
            for name in self.matrix[0].weights
            if name in self.matrix[0].components
        )
        return f"urgency_i = sum({parts}) / sum(weights), clamped to [0, 100]"


def deployment_slots_for(
    key_resource_units: int | None,
    n_locations: int,
    config: EngineConfig = CONFIG,
) -> tuple[int, list[str]]:
    """
    Convert available resource units into deployment slots.

    Args:
        key_resource_units: Available units of `OptimizationConfig.key_resource`.
        n_locations: Number of candidate locations.
        config: Engine configuration.

    Returns:
        `(slots, warnings)`. `slots` is always at least 1 and at most
        `n_locations`, because a zero-slot problem has no solution to optimize.
    """
    warnings: list[str] = []
    if key_resource_units is None:
        slots = 1
        warnings.append(
            "key resource availability was not reported, so exactly one deployment "
            "slot is assumed; this is a placeholder, not an assessment of capacity"
        )
        return slots, warnings

    units = int(key_resource_units)
    slots = units // max(1, config.optimization.units_per_deployment_slot)
    if slots < 1:
        slots = 1
        warnings.append(
            f"{units} {config.optimization.key_resource} is fewer than "
            f"OptimizationConfig.units_per_deployment_slot "
            f"({config.optimization.units_per_deployment_slot}), so the slot count "
            "would be 0; it is clamped to 1 so the problem remains solvable"
        )
    if slots > n_locations:
        warnings.append(
            f"{slots} deployment slots exceed the {n_locations} candidate locations; "
            f"the slot count is clamped to {n_locations}"
        )
        slots = n_locations
    return slots, warnings


def build_optimization_input(
    event: RuntimeEvent,
    flood: FloodAnalysis,
    roads: RoadAnalysis,
    infrastructure: InfrastructureAnalysis,
    config: EngineConfig = CONFIG,
) -> OptimizationInput:
    """
    Convert the runtime analysis into the preserved core's input format.

    Args:
        event: The runtime event.
        flood: Flood analysis (supplies the regional severity).
        roads: Road analysis (supplies regional accessibility).
        infrastructure: Critical-infrastructure analysis.
        config: Engine configuration.

    Returns:
        An `OptimizationInput`.
    """
    cfg = config.optimization
    warnings: list[str] = []
    severity = float(flood.severity_score.value or 0.0)
    accessibility = float(roads.accessibility_score.value or 50.0)

    travel_times = [
        float(location.travel_time_hours)
        for location in event.locations
        if location.travel_time_hours is not None and float(location.travel_time_hours) > 0
    ]
    max_travel = max(travel_times) if travel_times else 0.0

    raw_shares: dict[str, float] = {}
    for location in event.locations:
        population_term = float(max(0.0, location.population))
        if location.travel_time_hours is not None and max_travel > 0:
            travel_term = 1.0 - float(location.travel_time_hours) / max_travel
        else:
            travel_term = 0.0
        w = cfg.population_share_weights
        raw_shares[location.location_id] = (
            w.get("population", 0.0) * population_term
            + w.get("travel", 0.0) * travel_term
        )
    share_total = sum(raw_shares.values())
    if share_total <= 0:
        raw_shares = {location.location_id: 1.0 for location in event.locations}
        share_total = float(len(event.locations))
        warnings.append(
            "every location reported zero population and zero usable travel time, so "
            "the population split falls back to an equal share"
        )
    shares = {
        key: max(cfg.population_share_floor, value / share_total)
        for key, value in raw_shares.items()
    }
    share_sum = sum(shares.values())
    shares = {key: value / share_sum for key, value in shares.items()}

    evacuation_load = POPULATION.evacuation.value if POPULATION.available else None
    regional_population = float(evacuation_load) if evacuation_load else 0.0
    if regional_population <= 0:
        regional_population = float(event.population.total_population or 0)
        warnings.append(
            "no evacuation load was computed, so the raw regional population is "
            "split across locations instead of the evacuation load"
        )

    max_location_population = max(
        (float(max(0.0, loc.population)) for loc in event.locations), default=0.0
    )
    max_location_infrastructure = max(
        (float(loc.critical_infrastructure_count) for loc in event.locations), default=0.0
    )

    rows: list[LocationFeatureRow] = []
    for location in event.locations:
        access_penalty = {
            "OPEN": 0.0,
            "PARTIAL": 25.0,
            "BLOCKED": 60.0,
        }.get(str(location.road_access).upper(), 40.0)
        components = {
            "population": clamp(
                100.0
                * float(max(0.0, location.population))
                / max(1.0, max_location_population)
            ),
            "vulnerability": clamp(100.0 * float(location.vulnerability_index)),
            "infrastructure": clamp(
                100.0
                * float(location.critical_infrastructure_count)
                / max(1.0, max_location_infrastructure)
            ),
            "road_access": clamp(100.0 - access_penalty),
            "flood_severity": severity,
            "accessibility": clamp(accessibility),
        }
        weights = {
            name: cfg.location_weights.get(name, 0.0) for name in components
        }
        weight_total = sum(weights.values())
        if weight_total <= 0:
            raise QFlareValidationError(
                "OptimizationConfig.location_weights sums to "
                f"{weight_total}; it must be positive"
            )
        # The components are already 0-100 and the weights sum to 1, so the
        # weighted average is already 0-100. Dividing by the weight total keeps
        # the scale correct even if an operator edits the weights without
        # renormalising them.
        urgency = clamp(
            sum(components[key] * weights[key] for key in components) / weight_total
        )
        rows.append(
            LocationFeatureRow(
                location=location,
                components=components,
                weights=weights,
                urgency=urgency,
                population_share=shares[location.location_id],
                population=int(math.floor(regional_population * shares[location.location_id])),
            )
        )

    slots, slot_warnings = deployment_slots_for(
        getattr(event.resources, cfg.key_resource, None),
        len(event.locations),
        config,
    )
    warnings.extend(slot_warnings)

    return OptimizationInput(
        locations=[
            FloodLocation(
                location_id=row.location.location_id,
                name=row.location.name,
                affected_population=row.population,
                urgency=row.urgency,
                latitude=row.location.latitude,
                longitude=row.location.longitude,
            )
            for row in rows
        ],
        rescue_teams=int(
            getattr(event.resources, cfg.key_resource, 0) or 0
        ),
        deployment_slots=slots,
        key_resource=cfg.key_resource,
        matrix=rows,
        severity_score=severity,
        warnings=warnings,
    )


def display_optimization_input(optimization: OptimizationInput) -> None:
    """Print the OPTIMIZATION INPUTS block."""
    print("=" * 60)
    print("OPTIMIZATION INPUTS")
    print("=" * 60)
    print(f"Candidate locations    : {optimization.n_locations} "
          f"(=> {optimization.n_locations} qubits, "
          f"2**{optimization.n_locations} = {1 << optimization.n_locations} classical "
          "assignments)")
    print(f"Regional severity      : {optimization.severity_score:.2f} / 100")
    print(f"Key resource           : {optimization.key_resource}")
    print(f"Available              : {optimization.rescue_teams}")
    print(f"Units per slot         : {CONFIG.optimization.units_per_deployment_slot}")
    print(f"DEPLOYMENT SLOTS       : {optimization.deployment_slots}")
    print("")
    print(f"{'ID':<5}{'Name':<26}{'Pop':>9}{'Share':>8}{'Urgency':>9}   "
          f"{'Lat':>9}{'Lon':>10}")
    print("-" * 78)
    for row in optimization.matrix:
        location = row.location
        print(
            f"{location.location_id:<5}{location.name[:25]:<26}"
            f"{row.population:>9,}{row.population_share:>8.3f}{row.urgency:>9.2f}   "
            f"{(location.latitude if location.latitude is not None else float('nan')):>9.4f}"
            f"{(location.longitude if location.longitude is not None else float('nan')):>10.4f}"
        )
    print("-" * 78)
    print(f"Allocated population   : {optimization.total_population:,} "
          f"(regional evacuation load {POPULATION.evacuation.value:,.0f})")
    print("")
    print("Urgency formula        : " + optimization.urgency_formula())
    print("")
    print("Per-location urgency components (all 0-100):")
    component_names = list(optimization.matrix[0].components) if optimization.matrix else []
    header = f"{'ID':<5}" + "".join(f"{name[:13]:>15}" for name in component_names)
    print(header)
    print("-" * len(header))
    for row in optimization.matrix:
        line = f"{row.location.location_id:<5}" + "".join(
            f"{row.components[name]:>15.2f}" for name in component_names
        )
        print(line)
    print("")
    print("Only normalized 0-100 features and integer populations cross into QAOA.")
    print("No TMC, INR, km, m or m3/s value is sent to the quantum stage.")
    if optimization.warnings:
        print("")
        print("Handoff warnings:")
        for warning in optimization.warnings:
            print(f"  - {warning}")
    print("Optimization input status: READY")
    print("=" * 60)


In [ ]:
# ---- Run the optimization preprocessor ---------------------------------------

OPTIMIZATION_INPUT = build_optimization_input(
    RUNTIME_EVENT, FLOOD, ROADS, INFRASTRUCTURE, CONFIG
)

display_optimization_input(OPTIMIZATION_INPUT)

print("")
print("Hard constraint handed to the preserved core:")
print(f"  sum(x_i) <= {OPTIMIZATION_INPUT.deployment_slots} deployment slots "
      f"derived from {OPTIMIZATION_INPUT.rescue_teams} "
      f"{OPTIMIZATION_INPUT.key_resource} "
      f"/ {CONFIG.optimization.units_per_deployment_slot} per slot")


## SECTION 15 — QUBO FORMULATION (PRESERVED QUANTUM CORE)

Everything in this section is **carried over unchanged** from
`q_flare_quantum_core.py`. The mathematics is identical; only the *inputs* are
now runtime-derived (SECTION 14).

### Decision variables

```
x_i ∈ {0, 1}     i = 1 … n          n = number of locations (≤ 8)
x_i = 1  ⇔  location i receives a deployment slot
```

### Objective (benefit to be maximised)

```
B(x) = Σ_i  s_i · x_i

s_i = urgency_i · (1 + population_weight · population_i / max_j population_j)
```

`s_i` is the single source of truth: the QUBO builder and the classical scorer
both read `ProblemDefinition.scores`, so the quantum objective and the classical
baseline can never drift apart.

### QUBO cost function (minimised)

```
E(x) = − Σ_i  s_i · x_i   +   λ · ( Σ_i x_i − k )²

k = deployment_slots
λ = int(Σ_i s_i) + 1        (automatic, data-derived)
```

Expanding the penalty and using `(Σx)² = Σx_i + 2Σ_{i<j} x_i x_j`:

```
E(x) = Σ_i (λ(1 − 2k) − s_i) · x_i  +  2λ Σ_{i<j} x_i x_j  +  λk²
```

so the QUBO matrix is

```
Q[i][i] = λ(1 − 2k) − s_i          (diagonal, objective + capacity)
Q[i][j] = λ            i ≠ j       (off-diagonal, pairwise capacity penalty)
offset  = λ · k²                   (constant, does not change the argmin)
```

### Why λ is sufficient

With `λ = int(Σ s) + 1 > Σ s`, moving from any infeasible assignment to the best
feasible one reduces the penalty term by at least `λ`, which strictly exceeds the
entire benefit term. So **no infeasible assignment can have a lower energy than
any feasible assignment** — the constraint is enforced by the QUBO itself, not
merely filtered afterwards. `validate_qubo` re-checks this numerically against
the closed form for every one of the `2**n` assignments.


In [ ]:
# =============================================================================
# PRESERVED QUANTUM CORE - NAME ADAPTATION
# =============================================================================
#
# The quantum mathematics that follows is carried over byte-for-byte from
# `q_flare_quantum_core.py`. Exactly two names are adapted, and only so that the
# preserved code runs unchanged inside this runtime notebook:
#
#   1. The core's eight-level error hierarchy collapses to one domain error.
#      Every preserved routine raises one of these. Nothing catches them and
#      substitutes a placeholder, so a defect still fails the run loudly.
#
#   2. `QFlareConfig` becomes this notebook's `QuantumConfig`. The field names
#      the preserved routines read - `depth`, `shots`, `seed`,
#      `optimizer_method`, `optimizer_maxiter`, `optimizer_restarts` - are
#      identical, so `CONFIG.quantum` is passed straight through with no
#      translation layer.

class QFlareError(RuntimeError):
    """
    Q-FLARE domain error.

    A single base class replaces the preserved core's eight-level hierarchy
    (`QFlareValidationError`, `QFlareQuboError`, ...). The preserved routines
    keep raising the names they always raised, so no preserved `except` clause
    changes behaviour.
    """


QFlareValidationError = QFlareError
QFlareQuboError = QFlareError
QFlareCircuitError = QFlareError
QFlareOptimizerError = QFlareError
QFlareSimulationError = QFlareError
QFlareDecodeError = QFlareError

# The runtime notebook's QAOA parameters. Field names match the preserved core,
# so `CONFIG.quantum` satisfies the preserved annotations unchanged.
QFlareConfig = QuantumConfig


In [ ]:
# =============================================================================
# PRESERVED QUANTUM CORE - extracted verbatim from q_flare_quantum_core.py
# =============================================================================
#
# Lines below are copied byte-for-byte from the standalone Q-FLARE quantum
# engine. No line of the quantum mathematics was edited, reordered or
# rewritten. Only the upstream inputs changed (SECTION 14).
#
# The preserved core comment blocks (WHAT / WHY / HOW / Q-FLARE USE / INPUT /
# OUTPUT / MATHEMATICAL TRANSFORMATION) were kept with the code, because they
# are the justification for the mathematics.



@dataclass
class ProblemDefinition:
    """
    Machine-readable description of the allocation problem.

    Attributes:
        locations: Validated location records.
        rescue_teams: Number of available rescue teams (the hard constraint).
        scores: Per-location response score (single source of truth for the
            objective; the QUBO builder and the scorer both consume it).
        score_weights: Human readable description of each score component.
    """

    locations: list[FloodLocation]
    rescue_teams: int
    scores: list[float]
    population_weight: float

    @property
    def n_variables(self) -> int:
        """Number of binary decision variables (== number of qubits)."""
        return len(self.locations)

    @property
    def search_space_size(self) -> int:
        """Size of the classical search space, 2**n."""
        return 1 << self.n_variables

    def location_label(self, index: int) -> str:
        """Return `ID (Name)` for a variable index."""
        location = self.locations[index]
        return f"{location.location_id} ({location.name})"

    def labels_for(self, x: Sequence[int]) -> list[str]:
        """Return the labels of the selected locations."""
        return [self.location_label(i) for i, bit in enumerate(x) if bit]


def build_score_vector(
    locations: Sequence[FloodLocation],
    population_weight: float = 0.0,
) -> list[float]:
    """
    Compute the per-location response score used by the objective.

    Args:
        locations: Validated location records.
        population_weight: Weight of the normalized population term.

    Returns:
        List of response scores, one per location.

    Raises:
        QFlareValidationError: If `population_weight` is negative.
    """
    if population_weight < 0:
        raise QFlareValidationError(
            f"population_weight cannot be negative (got {population_weight})"
        )
    max_population = max((loc.affected_population for loc in locations), default=0)
    scores: list[float] = []
    for location in locations:
        normalized_population = (
            location.affected_population / max_population if max_population else 0.0
        )
        score = float(location.urgency) * (
            1.0 + population_weight * normalized_population
        )
        scores.append(score)
    return scores


def define_problem(
    locations: Sequence[FloodLocation],
    rescue_teams: int,
    population_weight: float = 0.0,
) -> ProblemDefinition:
    """
    Build the optimization problem from validated inputs.

    Args:
        locations: Validated location records.
        rescue_teams: Number of available rescue teams.
        population_weight: Weight of the normalized population term.

    Returns:
        The explicit problem definition.
    """
    scores = build_score_vector(locations, population_weight)
    return ProblemDefinition(
        locations=list(locations),
        rescue_teams=rescue_teams,
        scores=scores,
        population_weight=population_weight,
    )


def compute_response_score(
    problem: ProblemDefinition,
    x: Sequence[int],
) -> float:
    """
    Compute the response benefit B(x) of a binary decision vector.

    Args:
        problem: The problem definition.
        x: Binary decision vector.

    Returns:
        Total response benefit of the selected locations.
    """
    return float(
        sum(score for score, bit in zip(problem.scores, x, strict=True) if bit)
    )


def direct_energy(
    problem: ProblemDefinition,
    x: Sequence[int],
    penalty_weight: float,
) -> float:
    """
    Compute the QUBO energy directly from its closed-form definition.

    This is the reference implementation used to cross-check the QUBO matrix:

        E(x) = -B(x) + penalty * (sum(x_i) - teams)^2

    Args:
        problem: The problem definition.
        x: Binary decision vector.
        penalty_weight: Constraint penalty coefficient.

    Returns:
        The QUBO energy of x.
    """
    return -compute_response_score(problem, x) + penalty_weight * (
        sum(x) - problem.rescue_teams
    ) ** 2



@dataclass
class QuboModel:
    """
    A QUBO instance in matrix form.

    Attributes:
        matrix: Symmetric QUBO matrix `Q` of shape (n, n).
        offset: Constant term (energy shift, does not change the argmin).
        penalty_weight: Penalty coefficient used for the resource constraint.
        rescue_teams: Constraint right-hand side K.
        scores: Per-location response scores used in the objective.
    """

    matrix: Any
    offset: float
    penalty_weight: float
    rescue_teams: int
    scores: list[float]

    @property
    def n_variables(self) -> int:
        """Number of binary variables (QUBO dimension)."""
        return int(self.matrix.shape[0])

    def energy(self, x: Sequence[int]) -> float:
        """Compute E(x) = x^T Q x + offset for a binary vector."""
        import numpy as np

        vector = np.asarray(x, dtype=float)
        if vector.shape != (self.n_variables,):
            raise QFlareQuboError(
                f"decision vector has length {vector.shape[0]}, "
                f"expected {self.n_variables}"
            )
        return float(vector @ self.matrix @ vector + self.offset)


def auto_penalty_weight(scores: Sequence[float]) -> int:
    """
    Compute a provably sufficient constraint penalty from the data.

    Args:
        scores: Per-location response scores.

    Returns:
        An integer penalty that guarantees the QUBO optimum is feasible.
    """
    total_benefit = float(sum(scores))
    return int(total_benefit) + 1


def build_qubo_model(
    problem: ProblemDefinition,
    penalty_weight: int | None = None,
) -> QuboModel:
    """
    Build the QUBO matrix of the rescue allocation problem.

    Args:
        problem: The problem definition.
        penalty_weight: Explicit penalty, or None for the automatic value.

    Returns:
        The QUBO model.

    Raises:
        QFlareQuboError: If a non-positive penalty is requested, or if the
            problem definition cannot be turned into a matrix.
    """
    import numpy as np

    n = problem.n_variables
    teams = problem.rescue_teams
    penalty = (
        auto_penalty_weight(problem.scores)
        if penalty_weight is None
        else penalty_weight
    )
    if penalty <= 0:
        raise QFlareQuboError(f"penalty weight must be positive (got {penalty})")
    if n <= 0:  # pragma: no cover - guarded by input validation
        raise QFlareQuboError("cannot build a QUBO without decision variables")

    matrix = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            if i == j:
                matrix[i, j] = -problem.scores[i] + penalty * (1 - 2 * teams)
            else:
                matrix[i, j] = penalty
    offset = penalty * teams**2

    return QuboModel(
        matrix=matrix,
        offset=float(offset),
        penalty_weight=float(penalty),
        rescue_teams=teams,
        scores=list(problem.scores),
    )



def vector_to_bitstring(x: Sequence[int]) -> str:
    """
    Encode a decision vector using the Qiskit / classical bit ordering.

    The most significant character represents the LAST variable
    (x_{n-1}), matching the little-endian convention of Qiskit's
    ``counts`` keys and of ``Statevector`` basis indices.

    Args:
        x: Binary decision vector.

    Returns:
        Bitstring of length n.
    """
    n = len(x)
    return "".join(str(int(x[n - 1 - i])) for i in range(n))


def bitstring_to_vector(bitstring: str) -> list[int]:
    """
    Decode a Qiskit bitstring into a decision vector (little-endian).

    Args:
        bitstring: Bitstring such as "0011".

    Returns:
        Decision vector such as [1, 1, 0, 0].

    Raises:
        QFlareValidationError: If the bitstring contains non-binary characters.
    """
    cleaned = bitstring.strip()
    if not cleaned or any(char not in "01" for char in cleaned):
        raise QFlareValidationError(
            f"invalid bitstring '{bitstring}': only '0' and '1' are allowed"
        )
    n = len(cleaned)
    return [int(cleaned[n - 1 - i]) for i in range(n)]


def enumerate_all_vectors(n: int) -> list[list[int]]:
    """
    Enumerate every binary decision vector of dimension n.

    Args:
        n: Number of variables.

    Returns:
        List of 2**n vectors in binary-counting order (x_{n-1} fastest).
    """
    return [[(index >> (n - 1 - i)) & 1 for i in range(n)] for index in range(1 << n)]




In [ ]:
# =============================================================================
# SECTION 15 - RUN: build the problem and the QUBO
# =============================================================================

# The hard constraint the preserved core enforces is on the number of SELECTED
# LOCATIONS. Runtime teams are converted to slots in SECTION 14.
PROBLEM = define_problem(
    OPTIMIZATION_INPUT.locations,
    OPTIMIZATION_INPUT.deployment_slots,
    CONFIG.optimization.population_weight,
)

QUBO = build_qubo_model(PROBLEM)

print("=" * 60)
print("QUBO MODEL")
print("=" * 60)
print(f"Variables (qubits)     : {PROBLEM.n_variables}")
print(f"Deployment slots (K)   : {PROBLEM.rescue_teams}")
print(f"Search space           : 2**{PROBLEM.n_variables} = "
      f"{PROBLEM.search_space_size} assignments")
print(f"Penalty weight (lambda): {QUBO.penalty_weight:g} "
      f"= int(sum(s_i)) + 1 = int({sum(QUBO.scores):.4f}) + 1")
print(f"Offset (lambda K^2)    : {QUBO.offset:g}")
print("")
print("Response scores s_i = urgency_i x (1 + population_weight x pop_i / max pop):")
print(f"  population_weight   : {PROBLEM.population_weight:g}")
print(f"  {'ID':<5}{'Location':<30}{'Urgency':>9}{'Pop':>9}{'Norm':>8}{'Score':>10}")
print("  " + "-" * 71)
max_population = max(
    (loc.affected_population for loc in PROBLEM.locations), default=0
)
for index, location in enumerate(PROBLEM.locations):
    normalized = (
        location.affected_population / max_population if max_population else 0.0
    )
    print(
        f"  {location.location_id:<5}{location.name[:29]:<30}"
        f"{location.urgency:>9.2f}{location.affected_population:>9,}"
        f"{normalized:>8.3f}{QUBO.scores[index]:>10.2f}"
    )
print("")
print("QUBO matrix Q (diagonal = lambda(1 - 2K) - s_i, off-diagonal = lambda):")
header = "        " + "".join(
    f"{'col ' + str(j):>12}" for j in range(QUBO.n_variables)
)
print(header)
for i in range(QUBO.n_variables):
    row = "".join(f"{QUBO.matrix[i, j]:>12.2f}" for j in range(QUBO.n_variables))
    print(f"  [{i}]   " + row)
print("")
print(f"Symmetry max |Q[i][j] - Q[j][i]| : "
      f"{max(abs(QUBO.matrix[i, j] - QUBO.matrix[j, i]) for i in range(QUBO.n_variables) for j in range(QUBO.n_variables)):.2e}")

# ---- Cross-check the matrix against the closed form, for every assignment -----
worst_matrix_error = 0.0
for vector in enumerate_all_vectors(PROBLEM.n_variables):
    matrix_energy = QUBO.energy(vector)
    closed_form = direct_energy(PROBLEM, vector, QUBO.penalty_weight)
    worst_matrix_error = max(worst_matrix_error, abs(matrix_energy - closed_form))
print(f"Matrix vs closed form  : worst absolute error over all "
      f"{PROBLEM.search_space_size} assignments = {worst_matrix_error:.3e}")

# ---- Verify the penalty really does force feasibility ------------------------
# For every infeasible assignment, its energy must exceed the best feasible one.
best_feasible_energy = min(
    QUBO.energy(vector)
    for vector in enumerate_all_vectors(PROBLEM.n_variables)
    if sum(vector) <= PROBLEM.rescue_teams
)
worst_infeasible_excess = min(
    (
        QUBO.energy(vector) - best_feasible_energy
        for vector in enumerate_all_vectors(PROBLEM.n_variables)
        if sum(vector) > PROBLEM.rescue_teams
    ),
    default=float("inf"),
)
print(f"Penalty margin         : the cheapest infeasible assignment still costs "
      f"{worst_infeasible_excess:+.2f} more than the best feasible one")
print(f"                        lambda - sum(s_i) = "
      f"{QUBO.penalty_weight - sum(QUBO.scores):+.2f} (must be > 0)")
if worst_infeasible_excess <= 0:
    raise QFlareQuboError(
        "the penalty weight is too small: an infeasible assignment is not more "
        "expensive than every feasible one"
    )

print("")
print("QUBO status: BUILT AND VERIFIED")
print("=" * 60)


## SECTION 16 — EXACT CLASSICAL BASELINE (GROUND TRUTH)

A quantum result cannot be interpreted without ground truth, so before any
quantum execution the engine enumerates **all** `2**n` assignments and takes the
provable optimum.

```
n = 4  →  2**4 = 16 assignments  →  enumeration cost is trivial
```

Two facts are established here and used by every later section:

1. **The optimum** — lowest QUBO energy over all `2**n` assignments.
2. **The optimum is feasible** — because the penalty is strong enough, the
   unconstrained QUBO optimum already satisfies `sum(x) ≤ K`. The preserved core
   proves this by comparing the global minimum with the minimum over the
   feasible subset; the exact difference is printed.

> **Note on the constraint shape.** The quadratic penalty is centred on exactly
> `K` selected locations, so it mildly prefers *using all slots* over leaving one
> idle. For a monotonically increasing benefit (`s_i ≥ 0`, always true here) the
> optimum is identical to the "at most `K`" rule, which SECTION 15 verified
> numerically. This is stated rather than hidden.


In [ ]:
# =============================================================================
# PRESERVED QUANTUM CORE - extracted verbatim from q_flare_quantum_core.py
# =============================================================================
#
# Lines below are copied byte-for-byte from the standalone Q-FLARE quantum
# engine. No line of the quantum mathematics was edited, reordered or
# rewritten. Only the upstream inputs changed (SECTION 14).
#
# The preserved core comment blocks (WHAT / WHY / HOW / Q-FLARE USE / INPUT /
# OUTPUT / MATHEMATICAL TRANSFORMATION) were kept with the code, because they
# are the justification for the mathematics.



@dataclass
class ClassicalBaseline:
    """
    Ground-truth result of exhaustive classical enumeration.

    Attributes:
        best_vector: Decision vector of the minimum-energy solution.
        best_bitstring: The same solution in Qiskit bit ordering.
        best_energy: Minimum QUBO energy over all 2**n assignments.
        best_score: Response benefit of the best solution.
        feasible: Whether the best solution satisfies the team constraint.
        best_feasible_energy: Minimum energy among feasible assignments only.
        optimum_is_feasible: True when the penalty is strong enough that the
            unconstrained QUBO optimum is already feasible.
        table: pandas DataFrame with one row per enumerated assignment.
    """

    best_vector: list[int]
    best_bitstring: str
    best_energy: float
    best_score: float
    feasible: bool
    best_feasible_energy: float
    optimum_is_feasible: bool
    table: Any


def brute_force_solve(
    problem: ProblemDefinition,
    qubo: QuboModel,
) -> ClassicalBaseline:
    """
    Exhaustive exact solver: enumerate all 2**n assignments.

    Args:
        problem: The problem definition.
        qubo: The QUBO model.

    Returns:
        The exact classical baseline plus the full enumeration table.

    Raises:
        QFlareQuboError: If the QUBO matrix energy and the closed-form energy
            disagree, which would make every later comparison meaningless.
    """
    import pandas as pd

    n = problem.n_variables
    rows: list[dict[str, Any]] = []
    for x in enumerate_all_vectors(n):
        selected = sum(x)
        feasible = selected <= problem.rescue_teams
        energy = qubo.energy(x)
        # Cross-check: the matrix energy must equal the closed-form energy.
        reference_energy = direct_energy(problem, x, qubo.penalty_weight)
        if abs(energy - reference_energy) > 1e-8:  # pragma: no cover - guarded by tests
            raise QFlareQuboError(
                f"QUBO matrix energy {energy} disagrees with closed form "
                f"{reference_energy} for {x}"
            )
        rows.append(
            {
                "Bitstring": vector_to_bitstring(x),
                "Selected Locations": ", ".join(problem.labels_for(x)) or "(none)",
                "Selected Count": selected,
                "Response Score": compute_response_score(problem, x),
                "QUBO Energy": energy,
                "Constraint Penalty": energy
                + compute_response_score(problem, x),
                "Feasible": feasible,
            }
        )

    table = pd.DataFrame(rows)
    feasible_rows = table[table["Feasible"]]
    best_index = int(table["QUBO Energy"].idxmin())
    best_feasible_energy = float(feasible_rows["QUBO Energy"].min())

    return ClassicalBaseline(
        best_vector=bitstring_to_vector(str(table.loc[best_index, "Bitstring"])),
        best_bitstring=str(table.loc[best_index, "Bitstring"]),
        best_energy=float(table.loc[best_index, "QUBO Energy"]),
        best_score=float(table.loc[best_index, "Response Score"]),
        feasible=bool(table.loc[best_index, "Feasible"]),
        best_feasible_energy=best_feasible_energy,
        optimum_is_feasible=abs(
            float(table.loc[best_index, "QUBO Energy"]) - best_feasible_energy
        )
        < 1e-9,
        table=table,
    )





In [ ]:
# =============================================================================
# SECTION 16 - RUN: exact classical baseline
# =============================================================================

CLASSICAL = brute_force_solve(PROBLEM, QUBO)

print("=" * 60)
print("CLASSICAL EXACT BASELINE")
print("=" * 60)
print(f"Enumerated assignments: {PROBLEM.search_space_size} "
      f"(every one of the 2**{PROBLEM.n_variables} possibilities)")
print(f"Best bitstring         : {CLASSICAL.best_bitstring}")
print(f"Best decision vector   : {CLASSICAL.best_vector}")
print(f"Selected locations     : {', '.join(PROBLEM.labels_for(CLASSICAL.best_vector)) or '(none)'}")
print(f"Selected count         : {sum(CLASSICAL.best_vector)} of {PROBLEM.rescue_teams} slots")
print(f"Best response score    : {CLASSICAL.best_score:,.4f}")
print(f"Best QUBO energy       : {CLASSICAL.best_energy:,.4f}")
print(f"Best feasible energy   : {CLASSICAL.best_feasible_energy:,.4f}")
print(f"Optimum is feasible    : {CLASSICAL.feasible}")
print(f"Global min == feasible min : {CLASSICAL.optimum_is_feasible} "
      f"(difference {abs(CLASSICAL.best_energy - CLASSICAL.best_feasible_energy):.2e})")
print("")
print("Full enumeration table, in binary-counting order (all 2**n rows):")
display_table = CLASSICAL.table.copy()
display_table["Selected Locations"] = display_table["Selected Locations"].map(
    lambda text: (
        ", ".join(label.split(" (")[0] for label in text.split(", "))
        if text != "(none)"
        else text
    )
)
print(display_table.to_string(index=False))
print("")
print("Reading the table:")
print("  Response Score     = B(x) = sum of selected s_i, higher is better")
print("  QUBO Energy        = E(x) = -B(x) + lambda (sum(x) - K)^2, lower is better")
print("  Constraint Penalty = E(x) + B(x), i.e. the penalty term on its own")
print("  Feasible           = sum(x) <= K; an infeasible row is never a decision")
print("")
print("Classical baseline status: EXACT OPTIMUM ESTABLISHED")
print("=" * 60)


## SECTION 17 — QUBO → COST HAMILTONIAN (PAULI OPERATOR)

A QUBO matrix is a classical object. A quantum circuit can only apply operators
built from the Pauli basis, so a translation layer is required.

### Substitution

```
x_i     = (1 − Z_i) / 2
x_i x_j = (1 − Z_i − Z_j + Z_i Z_j) / 4
```

Applying it to `E(x) = Σ_i Q_ii x_i + Σ_{i<j} (Q_ij + Q_ji) x_i x_j` and
demanding `⟨x| H_C |x⟩ = E(x)` gives:

```
H_C = Σ_i  (−Q_ii/2) Z_i
    + Σ_{i<j} [ (−(Q_ij+Q_ji)/4) Z_i + (−(Q_ij+Q_ji)/4) Z_j
                 + ((Q_ij+Q_ji)/4)  Z_i Z_j ]
    + offset
```

where the scalar part collected along the way is

```
offset = λK² + ½ Σ_i Q_ii + ¼ Σ_{i<j} (Q_ij + Q_ji)
```

> The scalar part becomes the **Hamiltonian offset**. Dropping it would silently
> shift every measured energy, which is exactly the class of bug that makes a
> quantum pipeline produce plausible but wrong numbers.

### Mixer

```
H_M = ½ Σ_i X_i
```

so the layer unitary is `exp(−i·2β·H_M) = exp(−i·β·Σ_i X_i)`, the standard
Farhi–Gutmann QAOA form. `X` is what moves probability mass between the `0` and
`1` value of each decision variable.

### Verification

The section prints the full Pauli decomposition and then checks the identity
that actually matters, numerically:

```
⟨x| H_C |x⟩  ==  E(x)     for every one of the 2**n basis states
```

A mismatch would mean the quantum energies are not comparable with the classical
table, so a non-zero residual aborts the run.


In [ ]:
# =============================================================================
# PRESERVED QUANTUM CORE - extracted verbatim from q_flare_quantum_core.py
# =============================================================================
#
# Lines below are copied byte-for-byte from the standalone Q-FLARE quantum
# engine. No line of the quantum mathematics was edited, reordered or
# rewritten. Only the upstream inputs changed (SECTION 14).
#
# The preserved core comment blocks (WHAT / WHY / HOW / Q-FLARE USE / INPUT /
# OUTPUT / MATHEMATICAL TRANSFORMATION) were kept with the code, because they
# are the justification for the mathematics.



def qubo_to_pauli_operator(qubo: QuboModel):
    """
    Convert a QUBO matrix into a `qiskit.quantum_info.SparsePauliOp`.

    Args:
        qubo: The QUBO model.

    Returns:
        A SparsePauliOp `H_C` with the correct qubit count and offset.
    """
    from qiskit.quantum_info import SparsePauliOp

    matrix = qubo.matrix
    n = qubo.n_variables

    linear = [float(matrix[i, i]) for i in range(n)]
    offset = float(qubo.offset)
    terms: list[tuple[str, list[int], float]] = []

    for i in range(n):
        if linear[i] != 0.0:
            terms.append(("Z", [i], -0.5 * linear[i]))
        offset += 0.5 * linear[i]

    for i in range(n):
        for j in range(i + 1, n):
            cross = float(matrix[i, j] + matrix[j, i])
            if cross == 0.0:
                continue
            terms.append(("Z", [i], -0.25 * cross))
            terms.append(("Z", [j], -0.25 * cross))
            terms.append(("ZZ", [i, j], 0.25 * cross))
            offset += 0.25 * cross

    if abs(offset) > 1e-12:
        terms.append(("", [0], float(offset)))

    # `.simplify()` merges the terms produced by the algebra above into the
    # canonical Pauli decomposition. It changes nothing mathematically, it only
    # makes the printed Hamiltonian and the circuit diagram readable.
    return SparsePauliOp.from_sparse_list(terms, num_qubits=n).simplify()


def build_mixer_operator(n_variables: int):
    """
    Build the QAOA mixer Hamiltonian `H_M = 1/2 * sum_i X_i`.

    The X operator creates superposition: it is what allows the algorithm to
    move probability mass between the "0" and "1" value of each decision
    variable. Using the 1/2 convention means the layer unitary is written as
    `exp(-i * 2 * beta * H_M) = exp(-i * beta * sum_i X_i)`, which is the
    standard Farhi-Gutmann QAOA form.

    Args:
        n_variables: Number of qubits.

    Returns:
        A SparsePauliOp mixer Hamiltonian.
    """
    from qiskit.quantum_info import SparsePauliOp

    return SparsePauliOp.from_sparse_list(
        [("X", [i], 0.5) for i in range(n_variables)], num_qubits=n_variables
    )





In [ ]:
# =============================================================================
# SECTION 17 - RUN: cost and mixer Hamiltonians
# =============================================================================

from qiskit.quantum_info import Statevector

COST_OPERATOR = qubo_to_pauli_operator(QUBO)
MIXER_OPERATOR = build_mixer_operator(QUBO.n_variables)

print("=" * 60)
print("COST HAMILTONIAN H_C")
print("=" * 60)
print(f"Qubits                : {COST_OPERATOR.num_qubits}")
print(f"Pauli terms           : {len(COST_OPERATOR)}")
print("")
print("Pauli decomposition (coefficient x Pauli word):")
for label, coefficient in sorted(
    zip(COST_OPERATOR.paulis.to_labels(), COST_OPERATOR.coeffs),
    key=lambda item: -abs(float(item[1].real)),
):
    print(f"  {float(coefficient).real:>16.4f} x {label}")
print("")
print("Every Z_i term comes from the diagonal Q[i][i] (objective + capacity);")
print("every Z_i Z_j term comes from the off-diagonal Q[i][j] (pairwise capacity).")
print("The scalar term is the Hamiltonian offset; it must be kept or every")
print("measured energy shifts by a constant.")

print("")
print("=" * 60)
print("MIXER HAMILTONIAN H_M")
print("=" * 60)
print(f"Qubits                : {MIXER_OPERATOR.num_qubits}")
print(f"Pauli terms           : {len(MIXER_OPERATOR)}")
for label, coefficient in zip(MIXER_OPERATOR.paulis.to_labels(), MIXER_OPERATOR.coeffs):
    print(f"  {float(coefficient).real:>16.4f} x {label}")
print("")
print("H_M = 1/2 sum_i X_i, so the mixer unitary is exp(-i * 2 * beta * H_M).")

# ---- Verify <x| H_C |x> == E(x) for every basis state -------------------------
worst_identity_error = 0.0
for index in range(1 << QUBO.n_variables):
    bitstring = format(index, f"0{QUBO.n_variables}b")
    vector = bitstring_to_vector(bitstring)
    operator_energy = float(
        np.real(
            Statevector.from_label(bitstring).expectation_value(COST_OPERATOR)
        )
    )
    worst_identity_error = max(
        worst_identity_error, abs(operator_energy - QUBO.energy(vector))
    )
print("")
print(f"Identity check <x|H_C|x> = E(x) over all {1 << QUBO.n_variables} basis "
      f"states: worst absolute error = {worst_identity_error:.3e}")
if worst_identity_error > 1e-8:
    raise QFlareQuboError(
        f"the cost Hamiltonian does not reproduce the QUBO energies "
        f"(worst error {worst_identity_error:.3e}); the quantum and classical "
        "energies would not be comparable"
    )

print("")
print("Hamiltonian status: BUILT AND VERIFIED AGAINST THE QUBO")
print("=" * 60)


## SECTION 18 — QAOA CIRCUIT CONSTRUCTION

The ansatz is written out **gate by gate**, with no black-box optimizer hiding
the logic.

```
|+⟩^n  = (|0⟩ + |1⟩)^n / √(2^n)          ← uniform superposition
   │
   ├─ layer 1 : exp(−i·γ₀·H_C)           ← penalise bad assignments
   │           exp(−i·2·β₀·H_M)          ← explore neighbours
   ├─ layer 2 : exp(−i·γ₁·H_C)
   │           exp(−i·2·β₁·H_M)
   └─ measure_all()
```

| Symbol | Meaning |
| --- | --- |
| `p` | QAOA depth (variational circuit depth) |
| `γₗ` | Cost angle of layer `l`. Large `|γ|` concentrates probability on low-energy assignments. |
| `βₗ` | Mixer angle of layer `l`. Controls the hopping rate between neighbouring assignments; too large destroys the cost bias. |
| total | `2p` variational parameters |

**Depth choice.** `p = 2` is the standard QAOA starting point. For this problem
size the classical optimum is reachable at `p = 1`, but `p = 2` is used so the
run exercises a genuinely layered variational circuit. The depth is
configurable, and SECTION 19 reports the *achieved* concentration rather than
assuming it.

**Simulator note.** Execution is on a **local classical simulator**. This is not
a physical quantum computer, and no speedup or advantage is claimed.


In [ ]:
# =============================================================================
# PRESERVED QUANTUM CORE - extracted verbatim from q_flare_quantum_core.py
# =============================================================================
#
# Lines below are copied byte-for-byte from the standalone Q-FLARE quantum
# engine. No line of the quantum mathematics was edited, reordered or
# rewritten. Only the upstream inputs changed (SECTION 14).
#
# The preserved core comment blocks (WHAT / WHY / HOW / Q-FLARE USE / INPUT /
# OUTPUT / MATHEMATICAL TRANSFORMATION) were kept with the code, because they
# are the justification for the mathematics.



def build_qaoa_circuit(
    n_variables: int,
    cost_operator,
    depth: int,
):
    """
    Build the QAOA circuit manually.

    Args:
        n_variables: Number of qubits == number of decision variables.
        cost_operator: The cost Hamiltonian `H_C`.
        depth: Number of QAOA layers `p`.

    Returns:
        Tuple `(circuit, gamma_parameters, beta_parameters)`.

    Raises:
        QFlareValidationError: If depth is smaller than 1.
        QFlareCircuitError: If the Qiskit circuit cannot be assembled. A build
            failure is a hard error: silently returning a partial circuit would
            invalidate every downstream statevector and measurement.
    """
    from qiskit import QuantumCircuit
    from qiskit.circuit import ParameterVector
    from qiskit.circuit.library import PauliEvolutionGate

    if depth < 1:
        raise QFlareValidationError(f"QAOA depth must be >= 1 (got {depth})")

    try:
        mixer_operator = build_mixer_operator(n_variables)

        gammas = ParameterVector("gamma", depth)
        betas = ParameterVector("beta", depth)

        circuit = QuantumCircuit(n_variables, name=f"qaoa_p{depth}")
        # Step 1 - prepare the uniform superposition |+>^n
        circuit.h(range(n_variables))
        circuit.barrier()

        for layer in range(depth):
            # Step 2a - cost layer: exp(-i * gamma_l * H_C)
            circuit.append(
                PauliEvolutionGate(cost_operator, time=gammas[layer]),
                range(n_variables),
            )
            circuit.barrier()
            # Step 2b - mixer layer: exp(-i * 2 * beta_l * H_M)
            circuit.append(
                PauliEvolutionGate(mixer_operator, time=2.0 * betas[layer]),
                range(n_variables),
            )
            circuit.barrier()

        # Step 3 - measurement in the computational basis
        circuit.measure_all()
    except QFlareError:
        raise
    except Exception as exc:
        raise QFlareCircuitError(
            f"could not assemble the QAOA circuit "
            f"({n_variables} qubits, depth {depth}): {exc}"
        ) from exc

    if circuit.num_qubits != n_variables:  # pragma: no cover - defensive guard
        raise QFlareCircuitError(
            f"the built circuit has {circuit.num_qubits} qubits, "
            f"expected {n_variables}"
        )
    return circuit, gammas, betas


# Box-drawing and block characters emitted by the Qiskit text drawer, mapped to
# pure ASCII. Needed because `style="ascii"` alone does not guarantee ASCII
# output, and a legacy Windows code page turns those characters into mojibake.
_ASCII_TRANSLATION = str.maketrans(
    {
        "─": "-", "━": "-", "═": "=", "┄": "-", "┅": "-", "┈": "-", "┉": "-",
        "╌": "-", "╍": "-", "‾": "-", "▔": "-", "▁": "_",
        "│": "|", "┃": "|", "║": "|", "▏": "|", "▕": "|",
        "┌": "+", "┍": "+", "┎": "+", "┏": "+", "╭": "+",
        "┐": "+", "┑": "+", "┒": "+", "┓": "+", "╮": "+",
        "└": "+", "┕": "+", "┖": "+", "┗": "+", "╰": "+", "╯": "+",
        "┘": "+", "┙": "+", "┚": "+", "┛": "+",
        "├": "+", "┝": "+", "┞": "+", "┠": "+", "╲": "\\", "╱": "/",
        "┤": "+", "┥": "+", "┦": "+", "┨": "+", "╳": "+",
        "┬": "+", "┯": "+", "┰": "+",
        "┴": "+", "┷": "+", "┸": "+",
        "┼": "+", "┿": "+", "╀": "+",
        "█": "#", "▓": "#", "▒": ":", "░": ".", "■": "*",
        "↑": "^", "↓": "v", "»": ">", "«": "<", "⌐": "+",
        "−": "-", "–": "-", "—": "-", "…": "...", "′": "'", "⏚": "_",
    }
)


def draw_circuit(circuit, strict_ascii: bool = False) -> str:
    """
    Render the circuit as text, degrading gracefully on restricted consoles.

    WHY:
        The text drawer is the most readable way to show the QAOA ansatz in a
        notebook or a terminal, but it draws with box-drawing characters. On a
        Windows console with a legacy code page that produces unreadable
        mojibake and, without `_configure_stdout()`, a hard UnicodeEncodeError.

    Args:
        circuit: The circuit to render.
        strict_ascii: Force pure-ASCII output by translating every non-ASCII
            drawing character. Useful on restricted consoles.

    Returns:
        The rendered circuit text.

    ERROR HANDLING
        If the styled drawer fails, the ASCII drawer is used and any remaining
        non-printable character is replaced. A rendering problem is therefore
        always visible but never fatal: the quantum result does not depend on it.
    """
    try:
        style = "ascii" if strict_ascii else None
        text = str(circuit.draw(output="text", fold=110, style=style))
    except Exception as exc:
        print(f"[circuit] the styled text drawer failed ({exc}); using plain ASCII.")
        text = str(circuit.draw(output="text", fold=110, style="ascii"))
        strict_ascii = True
    if strict_ascii:
        # Two passes: the table renders the known glyphs readably, the ord
        # check guarantees the promise of pure ASCII output for anything left.
        text = text.translate(_ASCII_TRANSLATION)
        text = "".join(char if ord(char) < 128 else "?" for char in text)
    return "".join(
        char if char.isprintable() or char in "\n\t" else "?" for char in text
    )





In [ ]:
# =============================================================================
# SECTION 18 - RUN: build the QAOA circuit
# =============================================================================

CIRCUIT, GAMMAS, BETAS = build_qaoa_circuit(
    QUBO.n_variables, COST_OPERATOR, CONFIG.quantum.depth
)

print("=" * 60)
print("QAOA CIRCUIT")
print("=" * 60)
print(f"Qubits                : {CIRCUIT.num_qubits} (one per location)")
print(f"Classical registers   : {CIRCUIT.num_clbits}")
print(f"Depth p               : {CONFIG.quantum.depth}")
print(f"Variational parameters: {len(GAMMAS) + len(BETAS)} "
      f"(= 2 x p: {len(GAMMAS)} gammas + {len(BETAS)} betas)")
print(f"Total gates           : {CIRCUIT.size()}")
print(f"Circuit depth         : {CIRCUIT.depth()}")
print("")
print("Gate composition:")
for name, count in sorted(CIRCUIT.count_ops().items()):
    print(f"  {name:<40}{count:>4}")
print("")
print("Circuit diagram:")
print(draw_circuit(CIRCUIT))
print("")
print("Structure check:")
print(f"  Hadamard layer on all qubits : "
      f"{'yes' if CIRCUIT.count_ops().get('h', 0) == QUBO.n_variables else 'NO'}")
print(f"  Cost evolution gates         : "
      f"{CIRCUIT.count_ops().get('PauliEvolution', 0)} "
      f"(expected {2 * CONFIG.quantum.depth} = 2 per layer)")
print(f"  Barriers                     : {CIRCUIT.count_ops().get('barrier', 0)}")
print(f"  Measurement instructions     : "
      f"{sum(1 for item in CIRCUIT.data if item.operation.name == 'measure')}")
print("")
print("Simulator note: this circuit is executed on a local classical simulator")
print("(AerSimulator). It is NOT a physical quantum computer and no quantum")
print("advantage, speedup or superiority is claimed anywhere in this notebook.")
print("")
print("Circuit status: BUILT")
print("=" * 60)


## SECTION 19 — QAOA PARAMETER OPTIMIZATION

A QAOA circuit with untuned angles samples almost uniformly and therefore solves
nothing. The `2p` angles carry all the problem-specific knowledge, and the
landscape is non-convex, so a single local run is unreliable.

### Objective

```
E(θ) = ⟨ψ(θ) | H_C | ψ(θ)⟩,      θ = (γ₀…γ_{p−1}, β₀…β_{p−1})
```

### Method

| Setting | Value | Why |
| --- | --- | --- |
| Optimizer | SciPy `COBYLA` | derivative-free; the objective has no usable gradient and the landscape is non-convex |
| Objective evaluation | **exact statevector** | no shot noise during tuning, so the optimization is deterministic and reproducible |
| Restarts | `QuantumConfig.optimizer_restarts` (seeded) | escapes local minima |
| Seed | `QuantumConfig.seed` | one master seed drives restarts, transpilation and sampling |

Sampling noise is introduced **only later**, at the measurement stage, where real
shot statistics are reported. Tuning on a noise-free objective and reporting a
noisy measurement is what makes the two stages separately interpretable.

### Honesty rules

* A non-finite parameter vector or objective value is never accepted.
* If no restart produces a finite result, the stage **raises** — it does not
  return a fabricated angle set.
* Every restart is printed with its start energy, achieved energy and objective
  evaluation count, so the improvement is auditable.


In [ ]:
# =============================================================================
# PRESERVED QUANTUM CORE - extracted verbatim from q_flare_quantum_core.py
# =============================================================================
#
# Lines below are copied byte-for-byte from the standalone Q-FLARE quantum
# engine. No line of the quantum mathematics was edited, reordered or
# rewritten. Only the upstream inputs changed (SECTION 14).
#
# The preserved core comment blocks (WHAT / WHY / HOW / Q-FLARE USE / INPUT /
# OUTPUT / MATHEMATICAL TRANSFORMATION) were kept with the code, because they
# are the justification for the mathematics.



@dataclass
class OptimizationResult:
    """
    Outcome of the QAOA parameter optimization.

    Attributes:
        initial_parameters: Angles of the first restart (the reported start).
        optimized_parameters: Best angles found across all restarts.
        initial_expectation: Exact expected energy at the start angles.
        final_expectation: Exact expected energy at the optimized angles.
        optimizer_status: SciPy status / message of the best restart.
        optimizer_success: Whether the best restart reported success.
        restarts: Per-restart records (start, best, status, success).
        function_evaluations: Total objective evaluations.
    """

    initial_parameters: list[float]
    optimized_parameters: list[float]
    initial_expectation: float
    final_expectation: float
    optimizer_status: str
    optimizer_success: bool
    restarts: list[dict[str, Any]] = field(default_factory=list)
    function_evaluations: int = 0

    @property
    def expectation_improvement(self) -> float:
        """Energy reduction achieved by the optimization."""
        return self.initial_expectation - self.final_expectation


def expected_energy_from_circuit(
    bound_circuit,
    cost_operator,
) -> float:
    """
    Compute the exact expected QUBO energy of a parameterized circuit.

    Args:
        bound_circuit: Circuit with all parameters already bound.
        cost_operator: The cost Hamiltonian `H_C`.

    Returns:
        The real expectation value <H_C>.
    """
    import numpy as np
    from qiskit.quantum_info import Statevector

    state = Statevector.from_instruction(bound_circuit)
    return float(np.real(state.expectation_value(cost_operator)))


def optimize_qaoa_parameters(
    circuit,
    gammas,
    betas,
    cost_operator,
    config: QFlareConfig,
) -> OptimizationResult:
    """
    Optimize the QAOA angles against the expected QUBO energy.

    Args:
        circuit: The parameterized QAOA circuit.
        gammas: Gamma parameters of the circuit.
        betas: Beta parameters of the circuit.
        cost_operator: The cost Hamiltonian `H_C`.
        config: Engine configuration (method, iterations, restarts, seed).

    Returns:
        The optimization result.

    Raises:
        QFlareOptimizerError: If no restart produced a finite, valid result.
    """
    import numpy as np
    from scipy import optimize as scipy_optimize

    ordered_parameters = list(gammas) + list(betas)
    parameter_count = len(ordered_parameters)
    invalid_objective_value = 1.0e9
    evaluations = 0
    first_error: str | None = None

    # The exact-statevector objective cannot run on a circuit that already
    # contains measurements, so the final measurements are stripped here. The
    # measurement stage (STAGE 12) uses the full circuit.
    analytic_circuit = circuit.remove_final_measurements(inplace=False)

    def objective(theta: Sequence[float]) -> float:
        """Exact expected energy, guarded against invalid parameter vectors."""
        nonlocal evaluations, first_error
        evaluations += 1
        values = np.asarray(theta, dtype=float)
        if values.shape != (parameter_count,) or not np.all(np.isfinite(values)):
            return invalid_objective_value
        try:
            bound = analytic_circuit.assign_parameters(
                dict(zip(ordered_parameters, values.tolist(), strict=True))
            )
            energy = expected_energy_from_circuit(bound, cost_operator)
        except Exception as exc:  # defensive guard - reported, never hidden
            if first_error is None:
                first_error = f"{type(exc).__name__}: {exc}"
            return invalid_objective_value
        if not np.isfinite(energy):
            return invalid_objective_value
        return energy

    rng = np.random.default_rng(config.seed)
    restarts: list[dict[str, Any]] = []
    best: dict[str, Any] | None = None
    first_initial: list[float] | None = None
    first_initial_expectation: float = float("nan")

    for restart in range(max(1, config.optimizer_restarts)):
        start = rng.uniform(-0.5 * np.pi, 0.5 * np.pi, parameter_count)
        start_energy = objective(start)
        if first_initial is None:
            first_initial = [float(v) for v in start]
            first_initial_expectation = float(start_energy)

        result = scipy_optimize.minimize(
            objective,
            start,
            method=config.optimizer_method,
            options={"maxiter": config.optimizer_maxiter, "rhobeg": 0.8},
        )

        candidate_angles = np.asarray(result.x, dtype=float)
        valid = bool(
            np.all(np.isfinite(candidate_angles))
            and np.isfinite(result.fun)
            and result.fun < invalid_objective_value
        )
        record = {
            "restart": restart,
            "start_parameters": [float(v) for v in start],
            "start_expectation": float(start_energy),
            "best_parameters": [float(v) for v in candidate_angles],
            "best_expectation": float(result.fun),
            "valid": valid,
            "status": str(getattr(result, "message", "unknown")),
            "status_code": int(getattr(result, "status", -1)),
            "success": bool(getattr(result, "success", False)),
            "iterations": int(getattr(result, "nit", 0) or 0),
            "evaluations": int(getattr(result, "nfev", 0) or 0),
        }
        restarts.append(record)
        print(
            f"  restart {restart}: expected energy {record['start_expectation']:+.6f} "
            f"-> {record['best_expectation']:+.6f} "
            f"(valid={record['valid']}, {record['evaluations']} objective evaluations)"
        )
        if valid and (
            best is None or record["best_expectation"] < best["best_expectation"]
        ):
            best = record

    if best is None or first_initial is None:
        raise QFlareOptimizerError(
            "QAOA optimization produced no finite result. This is a hard failure, "
            "not a scientific outcome."
            + (
                f"\nFirst objective evaluation error: {first_error}"
                if first_error
                else ""
            )
        )

    optimized = [float(v) for v in best["best_parameters"]]
    if not np.all(np.isfinite(optimized)):
        raise QFlareOptimizerError("optimized QAOA parameters contain NaN/Inf values")

    return OptimizationResult(
        initial_parameters=first_initial,
        optimized_parameters=optimized,
        initial_expectation=first_initial_expectation,
        final_expectation=float(best["best_expectation"]),
        optimizer_status=str(best["status"]),
        optimizer_success=bool(best["success"]),
        restarts=restarts,
        function_evaluations=evaluations,
    )





In [ ]:
# =============================================================================
# SECTION 19 - RUN: optimize the QAOA angles
# =============================================================================

print("=" * 60)
print("QAOA PARAMETER OPTIMIZATION")
print("=" * 60)
print(f"Method                : {CONFIG.quantum.optimizer_method}")
print(f"Max iterations        : {CONFIG.quantum.optimizer_maxiter} per restart")
print(f"Restarts              : {CONFIG.quantum.optimizer_restarts} (seeded)")
print(f"Master seed           : {CONFIG.quantum.seed}")
print(f"Parameters optimized  : {len(GAMMAS) + len(BETAS)}")
print("")

OPTIMIZATION = optimize_qaoa_parameters(
    CIRCUIT, GAMMAS, BETAS, COST_OPERATOR, CONFIG.quantum
)

print("")
print(f"Optimizer status      : {OPTIMIZATION.optimizer_status}")
print(f"Optimizer success flag: {OPTIMIZATION.optimizer_success}")
print(f"Objective evaluations : {OPTIMIZATION.function_evaluations}")
print("")
print(f"{'Parameter':<16}{'Initial':>14}{'Optimized':>14}")
print("-" * 44)
for index, (start, tuned) in enumerate(
    zip(OPTIMIZATION.initial_parameters, OPTIMIZATION.optimized_parameters, strict=True)
):
    label = f"gamma[{index}]" if index < len(GAMMAS) else f"beta[{index - len(GAMMAS)}]"
    print(f"{label:<16}{start:>14.6f}{tuned:>14.6f}")
print("-" * 44)
print(f"{'EXPECTED ENERGY':<16}{OPTIMIZATION.initial_expectation:>14.6f}"
      f"{OPTIMIZATION.final_expectation:>14.6f}")
print("")
print(f"Energy improvement   : {OPTIMIZATION.expectation_improvement:,.6f} "
      "(initial expected energy minus final expected energy)")
initial_gap = OPTIMIZATION.initial_expectation - CLASSICAL.best_energy
final_gap = OPTIMIZATION.final_expectation - CLASSICAL.best_energy
print(f"Gap to the optimum    : {initial_gap:,.6f} -> {final_gap:,.6f} "
      f"({100.0 * final_gap / initial_gap:.1f}% of the initial gap remains)" if initial_gap
      else "Gap to the optimum    : 0 (the start angles already matched the optimum)")
print("")
print("How to read this:")
print(f"  The optimum QUBO energy is {CLASSICAL.best_energy:,.4f}. QAOA minimizes the")
print("  EXPECTED energy over the whole distribution, so it cannot go below the")
print("  optimum. A final expectation near the optimum means the probability mass")
print("  has concentrated on the good assignments; a value far above it means the")
print("  angles did not converge well. Both outcomes are reported as they occur.")
print("")
print("Per-restart detail:")
print(f"  {'Restart':<9}{'Start energy':>18}{'Best energy':>18}{'Valid':>8}"
      f"{'Iters':>8}{'Evals':>8}")
for record in OPTIMIZATION.restarts:
    print(
        f"  {record['restart']:<9}{record['start_expectation']:>18.6f}"
        f"{record['best_expectation']:>18.6f}{str(record['valid']):>8}"
        f"{record['iterations']:>8}{record['evaluations']:>8}"
    )
print("")
print("Parameter optimization status: COMPLETED")
print("=" * 60)


## SECTION 20 — STATEVECTOR VALIDATION

Sampling hides errors: a mis-built circuit still produces a histogram. The
statevector is inspected **before** any sampling, so a defect is caught here
instead of being reported later as a "quantum answer".

### Checks

| Check | Requirement |
| --- | --- |
| Normalization | `Σ_k |⟨k|ψ⟩|² = 1` |
| Probability mass | every one of the `2**n` basis states accounted for |
| Energy consistency | `Σ_k p_k · E(k) == ⟨H_C⟩` |
| Optimum reachability | `p(optimum) > 0` — the exact optimum is *in* the support |

The last check matters operationally: if the exact optimum had **zero**
probability, the measured distribution could never return it, and reporting a
"quantum result" would be meaningless. A zero-probability optimum is stated as a
finding, not hidden.

### Bit ordering

The same little-endian convention is used everywhere — Qiskit `counts` keys,
`Statevector` basis indices, and classical enumeration — so the probability table
below can be read directly against the SECTION 16 enumeration table.


In [ ]:
# =============================================================================
# PRESERVED QUANTUM CORE - extracted verbatim from q_flare_quantum_core.py
# =============================================================================
#
# Lines below are copied byte-for-byte from the standalone Q-FLARE quantum
# engine. No line of the quantum mathematics was edited, reordered or
# rewritten. Only the upstream inputs changed (SECTION 14).
#
# The preserved core comment blocks (WHAT / WHY / HOW / Q-FLARE USE / INPUT /
# OUTPUT / MATHEMATICAL TRANSFORMATION) were kept with the code, because they
# are the justification for the mathematics.



@dataclass
class StatevectorReport:
    """
    Diagnostics of the optimized QAOA state.

    Attributes:
        probabilities: Probability of every basis state, index = little-endian
            integer value of the basis state.
        most_likely: List of `(bitstring, probability)` sorted descending.
        expected_energy: Exact expected energy recomputed from probabilities.
        operator_expectation: Exact <H_C> returned by Qiskit.
        norm: Squared norm of the statevector.
    """

    probabilities: Any
    most_likely: list[tuple[str, float]]
    expected_energy: float
    operator_expectation: float
    norm: float


def analyse_statevector(
    bound_circuit,
    cost_operator,
    qubo: QuboModel,
    top_k: int = 5,
) -> StatevectorReport:
    """
    Compute and validate the statevector of the bound QAOA circuit.

    Args:
        bound_circuit: Parameter-bound QAOA circuit (measurements removed).
        cost_operator: The cost Hamiltonian `H_C`.
        qubo: The QUBO model (used for the per-state energies).
        top_k: How many basis states to report.

    Returns:
        The statevector diagnostics.

    Raises:
        QFlareSimulationError: If the state is not normalized.
    """
    import numpy as np
    from qiskit.quantum_info import Statevector

    n = qubo.n_variables
    state = Statevector.from_instruction(bound_circuit)
    probabilities = np.asarray(np.abs(state.data) ** 2, dtype=float)
    norm = float(probabilities.sum())

    if not np.isclose(norm, 1.0, atol=1e-8):
        raise QFlareSimulationError(
            f"statevector is not normalized (norm = {norm})"
        )

    energies = [
        qubo.energy(bitstring_to_vector(format(index, f"0{n}b")))
        for index in range(1 << n)
    ]
    expected_from_probabilities = float(np.dot(probabilities, energies))
    operator_expectation = float(np.real(state.expectation_value(cost_operator)))

    ranked = sorted(
        (
            (format(index, f"0{n}b"), float(probabilities[index]))
            for index in range(1 << n)
        ),
        key=lambda item: item[1],
        reverse=True,
    )

    return StatevectorReport(
        probabilities=probabilities,
        most_likely=ranked[:top_k],
        expected_energy=expected_from_probabilities,
        operator_expectation=operator_expectation,
        norm=norm,
    )





In [ ]:
# =============================================================================
# SECTION 20 - RUN: validate the optimized quantum state
# =============================================================================

ORDERED_PARAMETERS = list(GAMMAS) + list(BETAS)
PARAMETER_BINDING = dict(
    zip(ORDERED_PARAMETERS, OPTIMIZATION.optimized_parameters, strict=True)
)

# The exact-statevector objective cannot run on a circuit that already contains
# measurements, so they are stripped for the analysis (the measurement stage
# uses the full circuit).
ANALYTIC_CIRCUIT = CIRCUIT.remove_final_measurements(inplace=False)
BOUND_CIRCUIT = ANALYTIC_CIRCUIT.assign_parameters(PARAMETER_BINDING)

STATE = analyse_statevector(BOUND_CIRCUIT, COST_OPERATOR, QUBO, top_k=PROBLEM.n_variables)

n = QUBO.n_variables
probabilities = STATE.probabilities
ranking = sorted(
    (
        (format(index, f"0{n}b"), float(probabilities[index]), index)
        for index in range(1 << n)
    ),
    key=lambda item: -item[1],
)

print("=" * 60)
print("STATEVECTOR VALIDATION")
print("=" * 60)
print(f"Qubits                : {n}")
print(f"Basis states          : {1 << n}")
print(f"Squared norm          : {STATE.norm:.15f}  (must be 1.0)")
print(f"Normalized            : {abs(STATE.norm - 1.0) < 1e-10}")
print("")
print(f"Energy from probabilities : {STATE.expected_energy:,.10f}")
print(f"Operator expectation <H_C> : {STATE.operator_expectation:,.10f}")
print(f"Difference                 : "
      f"{abs(STATE.expected_energy - STATE.operator_expectation):.3e}  (must be ~0)")
print("")
print("Probability of every basis state (descending), with its QUBO energy:")
print(f"  {'Bitstring':<11}{'Locations':<18}{'Probability':>13}{'p in %':>9}"
      f"{'QUBO energy':>16}{'Rank':>6}")
print("  " + "-" * 73)
for rank, (bitstring, probability, index) in enumerate(ranking, start=1):
    vector = bitstring_to_vector(bitstring)
    labels = ", ".join(label.split(" (")[0] for label in PROBLEM.labels_for(vector))
    print(
        f"  {bitstring:<11}{(labels or '(none)'):<18}{probability:>13.6f}"
        f"{100.0 * probability:>8.3f}%{QUBO.energy(vector):>16,.4f}{rank:>6}"
    )
print("  " + "-" * 73)
print(f"  {'TOTAL':<11}{'':<18}{probabilities.sum():>13.6f}"
      f"{100.0 * probabilities.sum():>8.3f}%")

# ---- Reachability of the exact optimum ---------------------------------------
optimum_index = int(CLASSICAL.best_bitstring[::-1], 2)
optimum_probability = float(probabilities[optimum_index])
print("")
print(f"Exact classical optimum {CLASSICAL.best_bitstring} "
      f"({', '.join(label.split(' (')[0] for label in PROBLEM.labels_for(CLASSICAL.best_vector))})")
print(f"  QUBO energy          : {CLASSICAL.best_energy:,.6f}")
print(f"  Probability in state : {optimum_probability:.6f} "
      f"({100.0 * optimum_probability:.3f}%)")
if optimum_probability <= 0.0:
    print("  REACHABILITY         : NO - the exact optimum has zero probability in this")
    print("                          state, so sampling could never return it.")
    print("                          This is reported, not hidden; SECTION 23 uses the")
    print("                          best feasible sample that was actually observed.")
else:
    print("  REACHABILITY         : YES - the optimum is inside the support of the")
    print("                          optimized state, so it can be measured.")

# ---- Probability mass of the feasible subspace -------------------------------
feasible_mass = sum(
    float(probabilities[index])
    for index in range(1 << n)
    if bin(index).count("1") <= PROBLEM.rescue_teams
)
print("")
print(f"Probability mass on FEASIBLE assignments : {feasible_mass:.6f} "
      f"({100.0 * feasible_mass:.3f}%)")
print(f"Probability mass on INFEASIBLE assignments: {1.0 - feasible_mass:.6f} "
      f"({100.0 * (1.0 - feasible_mass):.3f}%)")
print("Infeasible mass is expected: QAOA explores the whole space and the")
print("constraint is enforced by the penalty, then filtered again in SECTION 22.")
if abs(STATE.norm - 1.0) > 1e-10:
    raise QFlareSimulationError(
        f"statevector is not normalized (norm = {STATE.norm}); every downstream "
        "probability would be wrong"
    )
if abs(STATE.expected_energy - STATE.operator_expectation) > 1e-8:
    raise QFlareSimulationError(
        "the probability-weighted energy disagrees with <H_C>; the circuit and the "
        "cost Hamiltonian are inconsistent"
    )

print("")
print("Statevector status: VALIDATED")
print("=" * 60)


## SECTION 21 — AER SIMULATOR EXECUTION

This is where the ideal distribution meets **finite sampling**. The optimizer
worked with an exact statevector; a measurement returns counts. Sampling is the
only honest way to see what a decision would actually look like, including shot
noise and infeasible outcomes.

### Run settings

| Setting | Value | Reason |
| --- | --- | --- |
| Backend | `qiskit_aer.AerSimulator` | local classical simulator |
| Shots | `QuantumConfig.shots` | finite-sample statistics |
| Transpiler seed | `QuantumConfig.seed` | reproducible circuit layout |
| Simulator seed | `QuantumConfig.seed` | reproducible counts |

Both seeds are fixed, so the counts below are **bit-for-bit reproducible** for a
given runtime, and SECTION 27 tests exactly that.

### What is reported

* **shot-based expected energy** — `Σ_k (count_k/shots) · E(k)`, which must be
  close to but not equal to the statevector expectation (finite samples)
* **distinct states** — how many different bitstrings were observed at all
* **top measured states** — the frequency table
* **transpiled depth and size** — the circuit cost the backend actually sees

> **Simulator note.** `AerSimulator` is a local classical simulator. This is
> **not** execution on a physical quantum computer, and no quantum advantage,
> speedup or superiority is claimed anywhere in this notebook.


In [ ]:
# =============================================================================
# PRESERVED QUANTUM CORE - extracted verbatim from q_flare_quantum_core.py
# =============================================================================
#
# Lines below are copied byte-for-byte from the standalone Q-FLARE quantum
# engine. No line of the quantum mathematics was edited, reordered or
# rewritten. Only the upstream inputs changed (SECTION 14).
#
# The preserved core comment blocks (WHAT / WHY / HOW / Q-FLARE USE / INPUT /
# OUTPUT / MATHEMATICAL TRANSFORMATION) were kept with the code, because they
# are the justification for the mathematics.



@dataclass
class SamplingResult:
    """
    Raw output of the simulated measurement.

    Attributes:
        counts: Mapping bitstring -> number of shots.
        shots: Total number of shots executed.
        distinct_states: Number of distinct observed bitstrings.
        transpiled_depth / transpiled_size: Circuit metrics after transpilation.
        expected_energy_shot_based: Energy estimated from the sampled counts.
    """

    counts: dict[str, int]
    shots: int
    distinct_states: int
    transpiled_depth: int
    transpiled_size: int
    expected_energy_shot_based: float


def execute_on_simulator(
    circuit,
    parameters: dict[str, float] | Any,
    config: QFlareConfig,
    qubo: QuboModel,
    energy_function: Callable[[Sequence[int]], float],
) -> SamplingResult:
    """
    Run the QAOA circuit on `qiskit_aer.AerSimulator`.

    Args:
        circuit: The parameterized QAOA circuit.
        parameters: Mapping of circuit parameters to values.
        config: Engine configuration (shots, seed).
        qubo: The QUBO model (for its variable count).
        energy_function: Callable returning the QUBO energy of a vector.

    Returns:
        The sampling result.

    Raises:
        QFlareSimulationError: If the simulator fails, returns no usable
            samples, or produces a bitstring of the wrong width.
    """
    from qiskit import transpile
    from qiskit_aer import AerSimulator

    backend = AerSimulator()
    bound = circuit.assign_parameters(parameters)
    compiled = transpile(bound, backend, seed_transpiler=config.seed)

    try:
        with warnings.catch_warnings():
            # scipy emits a matrix-free efficiency hint during the
            # transpilation of PauliEvolution gates. It is a performance note,
            # not an error, so it is silenced here and nowhere else.
            warnings.simplefilter("ignore")
            job = backend.run(compiled, shots=config.shots, seed_simulator=config.seed)
            raw_counts = job.result().get_counts()
    except Exception as exc:
        # A backend/transpiler failure is a hard error, not a scientific result.
        # It is re-raised as a domain error so the stage recorder can attribute
        # it to the simulation step instead of surfacing as an opaque stack.
        raise QFlareSimulationError(
            f"the Aer simulator failed after {config.shots} shots: {exc}"
        ) from exc

    counts = {str(key): int(value) for key, value in raw_counts.items()}
    total_shots = sum(counts.values())
    if total_shots == 0:  # pragma: no cover - defensive guard
        raise QFlareSimulationError("the simulator returned zero shots")

    n = qubo.n_variables
    expected_energy = 0.0
    for bitstring, count in counts.items():
        vector = bitstring_to_vector(bitstring)
        if len(vector) != n:
            raise QFlareSimulationError(
                f"measured bitstring '{bitstring}' has {len(vector)} bits, expected {n}"
            )
        expected_energy += (count / total_shots) * energy_function(vector)

    return SamplingResult(
        counts=counts,
        shots=total_shots,
        distinct_states=len(counts),
        transpiled_depth=int(compiled.depth()),
        transpiled_size=int(compiled.size()),
        expected_energy_shot_based=float(expected_energy),
    )





In [ ]:
# =============================================================================
# SECTION 21 - RUN: execute the QAOA circuit on the Aer simulator
# =============================================================================

SAMPLING = execute_on_simulator(
    CIRCUIT,
    PARAMETER_BINDING,
    CONFIG.quantum,
    QUBO,
    lambda vector: QUBO.energy(vector),
)

counts_table = sorted(
    SAMPLING.counts.items(), key=lambda item: -item[1]
)

print("=" * 60)
print("QUANTUM SIMULATION (AerSimulator)")
print("=" * 60)
print(f"Backend                : AerSimulator (local classical simulator)")
print(f"Shots executed         : {SAMPLING.shots:,}")
print(f"Distinct states seen   : {SAMPLING.distinct_states} of {1 << PROBLEM.n_variables}")
print(f"Transpiler seed        : {CONFIG.quantum.seed}")
print(f"Simulator seed         : {CONFIG.quantum.seed}")
print(f"Transpiled depth       : {SAMPLING.transpiled_depth}")
print(f"Transpiled gate count  : {SAMPLING.transpiled_size:,}")
print("")
print(f"Shot-based expected energy : {SAMPLING.expected_energy_shot_based:,.6f}")
print(f"Statevector expectation    : {STATE.operator_expectation:,.6f}")
print(f"Classical exact optimum    : {CLASSICAL.best_energy:,.6f}")
print(f"Sampling error vs statevector: "
      f"{abs(SAMPLING.expected_energy_shot_based - STATE.operator_expectation):.6f}")
print("")
print("Measured counts (descending), with the ideal probability for comparison:")
print(f"  {'Bitstring':<11}{'Locations':<18}{'Count':>9}{'Measured p':>12}"
      f"{'Ideal p':>11}{'Feasible':>10}")
print("  " + "-" * 71)
for bitstring, count in counts_table:
    vector = bitstring_to_vector(bitstring)
    labels = ", ".join(label.split(" (")[0] for label in PROBLEM.labels_for(vector))
    ideal = float(probabilities[int(bitstring[::-1], 2)])
    feasible = "yes" if sum(vector) <= PROBLEM.rescue_teams else "NO"
    print(
        f"  {bitstring:<11}{(labels or '(none)'):<18}{count:>9,}"
        f"{count / SAMPLING.shots:>12.6f}{ideal:>11.6f}{feasible:>10}"
    )
print("  " + "-" * 71)
print(f"  {'TOTAL':<11}{'':<18}{sum(SAMPLING.counts.values()):>9,}"
      f"{sum(SAMPLING.counts.values()) / SAMPLING.shots:>12.6f}")

infeasible_shots = sum(
    count
    for bitstring, count in SAMPLING.counts.items()
    if sum(bitstring_to_vector(bitstring)) > PROBLEM.rescue_teams
)
print("")
print(f"Infeasible shots       : {infeasible_shots:,} of {SAMPLING.shots:,} "
      f"({100.0 * infeasible_shots / SAMPLING.shots:.3f}%)")
print("These are real measurements, not discarded. SECTION 22 filters them out")
print("of the decision path and reports how many were removed.")
print("")
print("Both seeds are fixed, so these counts are reproducible: re-running this")
print("notebook with the same runtime input reproduces them exactly.")
print("SECTION 27 asserts that.")
print("")
print("Simulation status: COMPLETED")
print("=" * 60)


## SECTION 22 — BITSTRING DECODING, FEASIBILITY AND SELECTION

This is where a bit pattern becomes an operational decision — and therefore the
step where a wrong answer becomes harmful. Decoding must be exact, and an
infeasible assignment must never be returned as a recommendation.

### Bit ordering

Qiskit prints counts with the **highest classical bit on the left**, so the last
decision variable appears first. Decoding is the mirror of encoding:

```
measured "0011"
      │
      ▼  reverse the characters
x = [1, 1, 0, 0]
      │
      ▼  map indices to locations
L1 (Upstream Riverside Settlement) + L2 (Downstream Delta Cluster)
```

The identical convention is used for `Statevector` basis indices, so the
SECTION 20 probability table, the SECTION 16 enumeration table and these counts
are all directly comparable.

### Selection rule

```
feasible  ⇔  sum(x_i) ≤ K            (K = deployment slots)
selected  =  argmin  E(x)   over the feasible observed samples
```

If **every** measured sample were infeasible, the result would be `None` and the
engine would say so. A fabricated fallback solution is never substituted.


In [ ]:
# =============================================================================
# PRESERVED QUANTUM CORE - extracted verbatim from q_flare_quantum_core.py
# =============================================================================
#
# Lines below are copied byte-for-byte from the standalone Q-FLARE quantum
# engine. No line of the quantum mathematics was edited, reordered or
# rewritten. Only the upstream inputs changed (SECTION 14).
#
# The preserved core comment blocks (WHAT / WHY / HOW / Q-FLARE USE / INPUT /
# OUTPUT / MATHEMATICAL TRANSFORMATION) were kept with the code, because they
# are the justification for the mathematics.



@dataclass
class DecodedSample:
    """
    One measured assignment after decoding and validation.

    Attributes:
        bitstring: Raw measurement bitstring.
        x: Decoded decision vector.
        selected_labels: Labels of selected locations.
        selected_count: Number of selected locations.
        count: How many times this bitstring was measured.
        probability: count / total shots.
        response_score: Response benefit B(x).
        energy: QUBO energy E(x).
        constraint_penalty: E(x) + B(x) (the penalty part only).
        feasible: Whether the team constraint is satisfied.
    """

    bitstring: str
    x: list[int]
    selected_labels: list[str]
    selected_count: int
    count: int
    probability: float
    response_score: float
    energy: float
    constraint_penalty: float
    feasible: bool


@dataclass
class QuantumOutcome:
    """
    Final decision extracted from the quantum measurement.

    Attributes:
        best: Best FEASIBLE observed sample (None when the run produced no
            feasible sample - reported honestly, never faked).
        samples: All observed samples sorted by energy ascending.
        infeasible_shots: Number of shots whose assignment violated the
            constraint.
        optimal_match: True when the best feasible sample is the exact
            classical optimum bitstring.
    """

    best: DecodedSample | None
    samples: list[DecodedSample]
    infeasible_shots: int
    optimal_match: bool


def decode_samples(
    counts: dict[str, int],
    total_shots: int,
    problem: ProblemDefinition,
    qubo: QuboModel,
) -> list[DecodedSample]:
    """
    Decode every observed bitstring and evaluate its feasibility.

    Args:
        counts: Mapping bitstring -> shot count.
        total_shots: Total number of shots (for probabilities).
        problem: The problem definition.
        qubo: The QUBO model.

    Returns:
        List of decoded samples sorted by QUBO energy ascending.

    Raises:
        QFlareDecodeError: If a measured bitstring has the wrong width.
    """
    samples: list[DecodedSample] = []
    for bitstring, count in counts.items():
        x = bitstring_to_vector(bitstring)
        if len(x) != problem.n_variables:
            raise QFlareDecodeError(
                f"cannot decode '{bitstring}': expected {problem.n_variables} bits"
            )
        selected = sum(x)
        score = compute_response_score(problem, x)
        energy = qubo.energy(x)
        samples.append(
            DecodedSample(
                bitstring=bitstring,
                x=x,
                selected_labels=problem.labels_for(x),
                selected_count=selected,
                count=int(count),
                probability=int(count) / total_shots if total_shots else 0.0,
                response_score=float(score),
                energy=float(energy),
                constraint_penalty=float(energy + score),
                feasible=bool(selected <= problem.rescue_teams),
            )
        )
    samples.sort(key=lambda sample: sample.energy)
    return samples


def is_feasible(x: Sequence[int], rescue_teams: int) -> bool:
    """
    Check the hard rescue-team constraint.

    Args:
        x: Binary decision vector.
        rescue_teams: Available rescue teams.

    Returns:
        True when `sum(x) <= rescue_teams`.
    """
    return sum(x) <= rescue_teams


def select_best_feasible(samples: Sequence[DecodedSample]) -> DecodedSample | None:
    """
    Select the best FEASIBLE observed sample (lowest QUBO energy).

    Args:
        samples: Decoded samples.

    Returns:
        The best feasible sample, or None when every sample was infeasible.
    """
    feasible_samples = [sample for sample in samples if sample.feasible]
    if not feasible_samples:
        return None
    return min(feasible_samples, key=lambda sample: sample.energy)





In [ ]:
# =============================================================================
# SECTION 22 - RUN: decode the measurement into a decision
# =============================================================================

SAMPLES = decode_samples(SAMPLING.counts, SAMPLING.shots, PROBLEM, QUBO)
SELECTED = select_best_feasible(SAMPLES)
INFEASIBLE_SHOT_COUNT = sum(sample.count for sample in SAMPLES if not sample.feasible)
OPTIMAL_MATCH = bool(
    SELECTED is not None and SELECTED.bitstring == CLASSICAL.best_bitstring
)

QUANTUM_OUTCOME = QuantumOutcome(
    best=SELECTED,
    samples=SAMPLES,
    infeasible_shots=INFEASIBLE_SHOT_COUNT,
    optimal_match=OPTIMAL_MATCH,
)

print("=" * 60)
print("DECODED SOLUTION")
print("=" * 60)
print("Every observed bitstring, sorted by QUBO energy (best first):")
print(f"  {'Bitstring':<11}{'x':<18}{'Count':>8}{'p':>10}{'Score':>11}"
      f"{'Energy':>13}{'Penalty':>11}{'Feas':>6}")
print("  " + "-" * 88)
for sample in SAMPLES:
    vector_text = "[" + ", ".join(str(bit) for bit in sample.x) + "]"
    print(
        f"  {sample.bitstring:<11}{vector_text:<18}{sample.count:>8,}"
        f"{sample.probability:>10.5f}{sample.response_score:>11.3f}"
        f"{sample.energy:>13,.3f}{sample.constraint_penalty:>11.2f}"
        f"{'yes' if sample.feasible else 'NO':>6}"
    )
print("  " + "-" * 88)

print("")
print(f"Total shots               : {SAMPLING.shots:,}")
print(f"Infeasible shots filtered : {INFEASIBLE_SHOT_COUNT:,} "
      f"({100.0 * INFEASIBLE_SHOT_COUNT / SAMPLING.shots:.3f}% of all shots)")
print(f"Feasible shots retained   : {SAMPLING.shots - INFEASIBLE_SHOT_COUNT:,}")

if SELECTED is None:
    print("")
    print("SELECTED SOLUTION         : NONE")
    print("")
    print("Every measured sample violated the deployment-slot constraint, so the")
    print("engine reports no quantum decision. The classical baseline below is the")
    print("only trustworthy allocation in this situation, and no fallback solution")
    print("is invented here.")
else:
    print("")
    print("SELECTED SOLUTION (lowest-energy feasible observed sample)")
    print(f"  Bitstring              : {SELECTED.bitstring}")
    print(f"  Decision vector x      : {SELECTED.x}")
    print(f"  Selected locations     : {', '.join(SELECTED.selected_labels) or '(none)'}")
    print(f"  Selected count         : {SELECTED.selected_count} of "
          f"{PROBLEM.rescue_teams} deployment slots")
    print(f"  Observed count         : {SELECTED.count:,} shots "
          f"({100.0 * SELECTED.probability:.3f}% of the run)")
    print(f"  Response score B(x)    : {SELECTED.response_score:,.4f}")
    print(f"  QUBO energy E(x)       : {SELECTED.energy:,.4f}")
    print(f"  Constraint penalty     : {SELECTED.constraint_penalty:,.4f} "
          "(0 when the constraint is satisfied exactly)")
    print(f"  Feasible               : {SELECTED.feasible}")

print("")
print(f"QAOA optimum match        : {'YES' if OPTIMAL_MATCH else 'NO'}")
print(f"  Classical optimum       : {CLASSICAL.best_bitstring} "
      f"(score {CLASSICAL.best_score:,.4f}, energy {CLASSICAL.best_energy:,.4f})")
if SELECTED is not None:
    score_gap = CLASSICAL.best_score - SELECTED.response_score
    energy_gap = SELECTED.energy - CLASSICAL.best_energy
    print(f"  QAOA selected           : {SELECTED.bitstring} "
          f"(score {SELECTED.response_score:,.4f}, energy {SELECTED.energy:,.4f})")
    print(f"  Response score gap      : {score_gap:,.4f} "
          f"({100.0 * score_gap / CLASSICAL.best_score:.3f}% of the optimum)" if CLASSICAL.best_score
          else "  Response score gap      : 0.0000")
    print(f"  Energy gap              : {energy_gap:,.4f} (must be >= 0)")
    if energy_gap < -1e-9:
        raise QFlareQuboError(
            f"the selected quantum sample has a LOWER energy "
            f"({SELECTED.energy:,.4f}) than the proven optimum "
            f"({CLASSICAL.best_energy:,.4f}); the classical enumeration is "
            "incomplete or the penalty is too weak"
        )
    if OPTIMAL_MATCH:
        print("  Interpretation          : QAOA found the exact optimum on this run.")
    else:
        print("  Interpretation          : QAOA did NOT find the exact optimum on this")
        print("                           run. QAOA is probabilistic and variational, so")
        print("                           this is an expected outcome and it is reported")
        print("                           as a fact rather than engineered away.")

print("")
print("Decoding status: COMPLETED")
print("=" * 60)


In [ ]:
# =============================================================================
# PRESERVED QUANTUM CORE - extracted verbatim from q_flare_quantum_core.py
# =============================================================================
#
# Lines below are copied byte-for-byte from the standalone Q-FLARE quantum
# engine. No line of the quantum mathematics was edited, reordered or
# rewritten. Only the upstream inputs changed (SECTION 14).
#
# The preserved core comment blocks (WHAT / WHY / HOW / Q-FLARE USE / INPUT /
# OUTPUT / MATHEMATICAL TRANSFORMATION) were kept with the code, because they
# are the justification for the mathematics.



def build_comparison_table(
    problem: ProblemDefinition,
    classical: ClassicalBaseline,
    quantum_best: DecodedSample | None,
):
    """
    Build the classical vs quantum comparison table.

    Args:
        problem: The problem definition.
        classical: Exact classical baseline.
        quantum_best: Best feasible quantum sample (or None).

    Returns:
        pandas.DataFrame with one row per method.
    """
    import pandas as pd

    rows = [
        {
            "Method": "Classical exact (brute force)",
            "Selected Locations": ", ".join(problem.labels_for(classical.best_vector))
            or "(none)",
            "Response Score": classical.best_score,
            "QUBO Energy": classical.best_energy,
            "Feasible": classical.feasible,
        }
    ]
    if quantum_best is not None:
        rows.append(
            {
                "Method": "QAOA measured (AerSimulator)",
                "Selected Locations": ", ".join(quantum_best.selected_labels)
                or "(none)",
                "Response Score": quantum_best.response_score,
                "QUBO Energy": quantum_best.energy,
                "Feasible": quantum_best.feasible,
            }
        )
    return pd.DataFrame(rows)



